In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

IMG_PATH = "/kaggle/input/datasets/tanjemahamed/odir5k-classification/datasets/diabetes/image1.png"

img_bgr = cv2.imread(IMG_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Could not read image at: {IMG_PATH}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


TARGET_SIZE = 1024 
h, w = img_rgb.shape[:2]
scale = TARGET_SIZE / max(h, w)
new_w, new_h = int(w * scale), int(h * scale)

img_resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
img_norm = img_resized.astype(np.float32) / 255.0  # [0,1]

print("Original:", (h, w), "| Resized:", img_resized.shape[:2])

plt.figure(figsize=(6, 6))
plt.imshow(img_resized)
plt.title("Retinal Image (Resized)")
plt.axis("off")
plt.show()

Original: (224, 224) | Resized: (1024, 1024)


<Figure size 600x600 with 1 Axes>

In [2]:
import os, sys
WORKDIR = "/kaggle/working"
os.chdir(WORKDIR)

if not os.path.exists("MedSAM"):
    !git clone https://github.com/bowang-lab/MedSAM.git
else:
    print("MedSAM repo already exists.")

sys.path.append(os.path.join(WORKDIR, "MedSAM"))
print("Ready. Current dir:", os.getcwd())

Cloning into 'MedSAM'...
remote: Enumerating objects: 967, done.
remote: Total 967 (delta 0), reused 0 (delta 0), pack-reused 967 (from 1)
Receiving objects: 100% (967/967), 62.91 MiB | 39.57 MiB/s, done.
Resolving deltas: 100% (475/475), done.
Ready. Current dir: /kaggle/working


In [3]:
!pip -q install opencv-python-headless==4.9.0.80
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

import torch
import numpy as np

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 37.4 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
Torch: 2.8.0+cu126
CUDA available: True


In [4]:
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

SAM_CKPT = os.path.join(CKPT_DIR, "sam_vit_b_01ec64.pth")
MEDSAM_CKPT = os.path.join(CKPT_DIR, "medsam_vit_b.pth")


if not os.path.exists(SAM_CKPT):
    !wget -q -O "$SAM_CKPT" https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
else:
    print("SAM checkpoint exists:", SAM_CKPT)

# Download MedSAM checkpoint (Zenodo)
if not os.path.exists(MEDSAM_CKPT):
    !wget -q -O "$MEDSAM_CKPT" https://zenodo.org/records/10689643/files/medsam_vit_b.pth
else:
    print("MedSAM checkpoint exists:", MEDSAM_CKPT)

print("Files in checkpoints:", os.listdir(CKPT_DIR))

Files in checkpoints: ['medsam_vit_b.pth', 'sam_vit_b_01ec64.pth']


In [5]:
import os

def file_info(path):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{path}  |  {size_mb:.2f} MB")
    with open(path, "rb") as f:
        head = f.read(16)
    print("Header bytes:", head)


file_info(MEDSAM_CKPT)


/kaggle/working/checkpoints/medsam_vit_b.pth  |  357.67 MB
Header bytes: b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00'


In [6]:
import torch
from segment_anything import sam_model_registry, SamPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


sam = sam_model_registry["vit_b"](checkpoint=SAM_CKPT)
sam.to(DEVICE)

state = torch.load(MEDSAM_CKPT, map_location=DEVICE, weights_only=True)

if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

missing, unexpected = sam.load_state_dict(state, strict=False)

sam.eval()
predictor = SamPredictor(sam)

print("MedSAM loaded on:", DEVICE)
print("Missing keys:", len(missing), "| Unexpected keys:", len(unexpected))


MedSAM loaded on: cuda
Missing keys: 0 | Unexpected keys: 0


In [7]:
img_rgb_resized = img_resized.copy()

print("Using resized image with shape:", img_rgb_resized.shape)


Using resized image with shape: (1024, 1024, 3)


In [8]:
import torch
import numpy as np
import torch.nn.functional as F


predictor.set_image(img_rgb_resized)

H, W = img_rgb_resized.shape[:2]
box = np.array([[0, 0, W, H]], dtype=np.float32)
box_torch = torch.as_tensor(box, device=DEVICE)

with torch.no_grad():
    sparse_embeddings, dense_embeddings = sam.prompt_encoder(
        points=None,
        boxes=box_torch,
        masks=None,
    )

    low_res_masks, iou_predictions = sam.mask_decoder(
        image_embeddings=predictor.get_image_embedding(),
        image_pe=sam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=True,
    )

    prob_masks = torch.sigmoid(low_res_masks)

    prob_masks = F.interpolate(
        prob_masks,
        size=(H, W),
        mode="bilinear",
        align_corners=False,
    )

prob_masks = prob_masks.cpu().numpy()
binary_masks = prob_masks > 0.5

print(f"Image Size: {W}x{H}")
print("Number of masks:", binary_masks.shape[1])
print("Model IoU predictions:", iou_predictions.cpu().numpy())


Image Size: 1024x1024
Number of masks: 3
Model IoU predictions: [[0.64710724 0.66507703 0.51663804]]


In [9]:
import numpy as np

prob_masks_np = prob_masks[0]                     
iou_np = iou_predictions.cpu().numpy()[0]        


best_idx = np.argmax(iou_np)

best_soft_mask = prob_masks_np[best_idx]         
best_mask = best_soft_mask > 0.5                 

print("Best mask index:", best_idx, "| IoU score:", iou_np[best_idx])


mask_area = int(best_mask.sum())

ys, xs = np.where(best_mask)
centroid_y = float(ys.mean()) if len(ys) > 0 else 0.0
centroid_x = float(xs.mean()) if len(xs) > 0 else 0.0


model_confidence = float(best_soft_mask[best_mask].mean()) if mask_area > 0 else 0.0

mask_info = {
    "mask": best_mask,
    "soft_mask": best_soft_mask,
    "area": mask_area,
    "centroid": (centroid_x, centroid_y),
    "model_confidence": model_confidence
}

print("Mask area:", mask_info["area"])
print("Mask centroid:", mask_info["centroid"])
print("Model confidence:", mask_info["model_confidence"])


Best mask index: 1 | IoU score: 0.66507703
Mask area: 681840
Mask centroid: (573.0354086002582, 509.4271925965036)
Model confidence: 0.814403772354126


In [10]:
plt.figure(figsize=(6,6))
plt.imshow(img_rgb_resized)
plt.scatter(mask_info["centroid"][0], mask_info["centroid"][1], c="red", s=40)
plt.imshow(best_mask, alpha=0.4)
plt.title("Selected Mask (Ready for Concept Assignment)")
plt.axis("off")
plt.show()


<Figure size 600x600 with 1 Axes>

In [11]:
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import random

img_u8 = img_rgb_resized.copy()
H, W = img_u8.shape[:2]

predictor.set_image(img_u8)
image_embedding = predictor.get_image_embedding()

candidate_masks = []
candidate_soft_masks = []
candidate_scores = []
candidate_types = []

def run_box_prompt(x0, y0, x1, y1):
    box = np.array([[x0, y0, x1, y1]], dtype=np.float32)
    box_torch = torch.as_tensor(box, device=DEVICE)

    with torch.no_grad():
        sparse_embeddings, dense_embeddings = sam.prompt_encoder(
            points=None,
            boxes=box_torch,
            masks=None,
        )

        low_res_masks, iou_predictions = sam.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
        )

        prob_masks = torch.sigmoid(low_res_masks)
        prob_masks = F.interpolate(
            prob_masks,
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        )

    prob_masks = prob_masks.cpu().numpy()[0]
    iou_np = iou_predictions.cpu().numpy()[0]
    best_idx = int(np.argmax(iou_np))

    return prob_masks[best_idx]


def run_point_prompt(x, y):
    point_coords = torch.tensor([[[x, y]]], dtype=torch.float, device=DEVICE)
    point_labels = torch.tensor([[1]], dtype=torch.int, device=DEVICE)

    with torch.no_grad():
        sparse_embeddings, dense_embeddings = sam.prompt_encoder(
            points=(point_coords, point_labels),
            boxes=None,
            masks=None,
        )

        low_res_masks, iou_predictions = sam.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
        )

        prob_masks = torch.sigmoid(low_res_masks)
        prob_masks = F.interpolate(
            prob_masks,
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        )

    prob_masks = prob_masks.cpu().numpy()[0]
    iou_np = iou_predictions.cpu().numpy()[0]
    best_idx = int(np.argmax(iou_np))

    return prob_masks[best_idx]


def add_object(soft_mask, obj_type):
    binary_m = soft_mask > 0.5
    if binary_m.sum() == 0:
        return
    model_conf = float(soft_mask[binary_m].mean())

    candidate_masks.append(binary_m)
    candidate_soft_masks.append(soft_mask)
    candidate_scores.append(model_conf)
    candidate_types.append(obj_type)




gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)
_, thr = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
thr = cv2.medianBlur(thr, 9)

cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if len(cnts) == 0:
    raise RuntimeError("Could not find fundus region.")

c = max(cnts, key=cv2.contourArea)
(xc, yc), radius = cv2.minEnclosingCircle(c)
xc, yc, radius = int(xc), int(yc), int(radius)

mask_circle = np.zeros((H, W), dtype=np.uint8)
cv2.circle(mask_circle, (xc, yc), int(radius * 0.98), 255, -1)



g = img_u8[:, :, 1]
g_blur = cv2.GaussianBlur(g, (21, 21), 0)
search = g_blur[:, : int(0.67 * W)].copy()

Y, X = np.ogrid[:H, :search.shape[1]]
circle = (X - xc) ** 2 + (Y - yc) ** 2 <= radius ** 2
search[~circle] = 0

_, _, _, maxLoc = cv2.minMaxLoc(search)
seed_x, seed_y = maxLoc

box_size = int(max(60, radius * 0.22))
soft = run_box_prompt(
    max(0, seed_x - box_size),
    max(0, seed_y - box_size),
    min(W - 1, seed_x + box_size),
    min(H - 1, seed_y + box_size),
)
add_object(soft, "optic_disc")




clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
g_eq = clahe.apply(g)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (19, 19))
blackhat = cv2.morphologyEx(g_eq, cv2.MORPH_BLACKHAT, kernel)
blackhat[mask_circle == 0] = 0

flat = blackhat.flatten()
if flat.max() > 0:
    thr_v = np.percentile(flat[flat > 0], 99.3)
    vessel_seeds = np.column_stack(np.where(blackhat >= thr_v)).tolist()

    random.seed(0)
    if len(vessel_seeds) > 20:
        vessel_seeds = random.sample(vessel_seeds, 20)

    for (yy, xx) in vessel_seeds:
        soft = run_point_prompt(xx, yy)
        add_object(soft, "vessel")




r = img_u8[:, :, 0]
r_blur = cv2.GaussianBlur(r, (9, 9), 0)
les = r_blur.copy()
les[mask_circle == 0] = 0

vals = les[les > 0]
if vals.size > 0:
    t = np.percentile(vals, 99.5)
    bin_les = (les >= t).astype(np.uint8) * 255
    bin_les = cv2.medianBlur(bin_les, 5)

    num, lbl, stats, _ = cv2.connectedComponentsWithStats(bin_les, connectivity=8)

    for i in range(1, num):
        x, y, w, h, area = stats[i]
        if 50 <= area <= 8000:
            pad = 10
            soft = run_box_prompt(
                max(0, x - pad),
                max(0, y - pad),
                min(W - 1, x + w + pad),
                min(H - 1, y + h + pad),
            )
            add_object(soft, "lesion")

print("Candidate objects found:", len(candidate_masks))
print("Types count:", {t: candidate_types.count(t) for t in set(candidate_types)})
print("Avg model confidence:", float(np.mean(candidate_scores)) if candidate_scores else None)


Candidate objects found: 21
Types count: {'lesion': 3, 'optic_disc': 1, 'vessel': 17}
Avg model confidence: 0.7276415597824824


In [12]:
import numpy as np

refined_masks = []
refined_soft_masks = []
refined_types = []
refined_scores = []

H, W = img_rgb_resized.shape[:2]
IMG_AREA = H * W

stats = []

for m, sm, t, conf in zip(candidate_masks,
                          candidate_soft_masks,
                          candidate_types,
                          candidate_scores):

    area = m.sum()
    if area == 0:
        continue

    area_ratio = area / IMG_AREA

    ys, xs = np.where(m)
    w_box = xs.max() - xs.min() + 1
    h_box = ys.max() - ys.min() + 1
    elongation = min(w_box, h_box) / max(w_box, h_box)

    stats.append((t, area_ratio, elongation, conf))


if len(stats) == 0:
    print("No valid masks to refine.")
else:
    types_arr = np.array([s[0] for s in stats])
    area_arr = np.array([s[1] for s in stats])
    elong_arr = np.array([s[2] for s in stats])
    conf_arr = np.array([s[3] for s in stats])

    # Data-driven thresholds
    conf_thr = np.percentile(conf_arr, 25)
    area_low = np.percentile(area_arr, 5)
    area_high = np.percentile(area_arr, 95)
    elong_low = np.percentile(elong_arr, 10)
    elong_high = np.percentile(elong_arr, 90)

    for m, sm, t, conf in zip(candidate_masks,
                              candidate_soft_masks,
                              candidate_types,
                              candidate_scores):

        area = m.sum()
        if area == 0:
            continue

        area_ratio = area / IMG_AREA

        ys, xs = np.where(m)
        w_box = xs.max() - xs.min() + 1
        h_box = ys.max() - ys.min() + 1
        elongation = min(w_box, h_box) / max(w_box, h_box)

        if conf < conf_thr:
            continue

        keep = (
            area_low <= area_ratio <= area_high and
            elong_low <= elongation <= elong_high
        )

        if keep:
            refined_masks.append(m)
            refined_soft_masks.append(sm)
            refined_types.append(t)
            refined_scores.append(float(conf))

    print("Before:", len(candidate_masks))
    print("After refinement:", len(refined_masks))
    print("Adaptive confidence threshold:", round(conf_thr, 3))
    print("Avg confidence kept:", np.mean(refined_scores) if refined_scores else None)


Before: 21
After refinement: 12
Adaptive confidence threshold: 0.664
Avg confidence kept: 0.7603061397870382


In [13]:
import numpy as np

rows = []

for i, (m, t, conf) in enumerate(
        zip(refined_masks, refined_types, refined_scores)):

    area = int(m.sum())
    name = f"{t}_{i:03d}"

    rows.append({
        "node_id": i,
        "name": name,
        "confidence": float(conf),   
        "area": area
    })


rows_sorted = sorted(rows, key=lambda x: x["confidence"], reverse=True)

print(f"Total Valid Nodes Extracted: {len(rows_sorted)}\n")

for r in rows_sorted:
    print(
        f"ID: {r['node_id']:02d} | "
        f"Name: {r['name']:15s} | "
        f"Conf: {r['confidence']:.4f} | "
        f"Area: {r['area']}"
    )


Total Valid Nodes Extracted: 12

ID: 09 | Name: lesion_009      | Conf: 0.9050 | Area: 4119
ID: 00 | Name: vessel_000      | Conf: 0.8931 | Area: 91257
ID: 04 | Name: vessel_004      | Conf: 0.8000 | Area: 102450
ID: 10 | Name: lesion_010      | Conf: 0.7722 | Area: 212
ID: 01 | Name: vessel_001      | Conf: 0.7663 | Area: 66715
ID: 06 | Name: vessel_006      | Conf: 0.7435 | Area: 125799
ID: 05 | Name: vessel_005      | Conf: 0.7423 | Area: 129160
ID: 08 | Name: vessel_008      | Conf: 0.7416 | Area: 125694
ID: 11 | Name: lesion_011      | Conf: 0.7406 | Area: 255
ID: 07 | Name: vessel_007      | Conf: 0.6743 | Area: 39396
ID: 02 | Name: vessel_002      | Conf: 0.6727 | Area: 18819
ID: 03 | Name: vessel_003      | Conf: 0.6722 | Area: 6216


In [14]:
import numpy as np
import math

H, W = img_u8.shape[:2]
IMG_DIAG = float(math.sqrt(H * H + W * W) + 1e-8)




def mask_centroid(mask):
    ys, xs = np.where(mask)
    if xs.size == 0:
        return None
    return float(xs.mean()), float(ys.mean())

def mask_bbox(mask):
    ys, xs = np.where(mask)
    if xs.size == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def bbox_iou(b1, b2):
    x0a, y0a, x1a, y1a = b1
    x0b, y0b, x1b, y1b = b2

    inter_x0 = max(x0a, x0b)
    inter_y0 = max(y0a, y0b)
    inter_x1 = min(x1a, x1b)
    inter_y1 = min(y1a, y1b)

    iw = max(0, inter_x1 - inter_x0 + 1)
    ih = max(0, inter_y1 - inter_y0 + 1)

    if iw <= 0 or ih <= 0:
        return 0.0

    inter = iw * ih
    area_a = (x1a - x0a + 1) * (y1a - y0a + 1)
    area_b = (x1b - x0b + 1) * (y1b - y0b + 1)

    return float(inter / (area_a + area_b - inter + 1e-8))

def centroid_dist(c1, c2):
    return float(math.hypot(c1[0] - c2[0], c1[1] - c2[1]))

def dist_conf(d, sigma=0.12):
    x = d / IMG_DIAG
    return float(math.exp(-(x * x) / (2 * sigma * sigma)))

def overlap_conf(iou):
    return float(max(0.0, min(1.0, iou * 3.0)))




node_names = [f"{t}_{i:03d}" for i, t in enumerate(refined_types)]

centroids = []
bboxes = []

for m in refined_masks:
    centroids.append(mask_centroid(m))
    bboxes.append(mask_bbox(m))




edges = []
NEAR_MAX_NORM = 0.18
IOU_MIN = 0.02

for i in range(len(node_names)):
    for j in range(i + 1, len(node_names)):

        if centroids[i] is None or centroids[j] is None:
            continue
        if bboxes[i] is None or bboxes[j] is None:
            continue

        di = centroid_dist(centroids[i], centroids[j])
        di_norm = di / IMG_DIAG
        iou = bbox_iou(bboxes[i], bboxes[j])

        rels = []

        if iou >= IOU_MIN:
            rels.append(("overlaps", overlap_conf(iou)))

        if di_norm <= NEAR_MAX_NORM:
            rels.append(("near", dist_conf(di)))

        ti, tj = refined_types[i], refined_types[j]

        if di_norm <= 0.22:
            if ("optic_disc" in (ti, tj)) and ("lesion" in (ti, tj)):
                rels.append(("lesion_near_disc", dist_conf(di)))

            if ("vessel" in (ti, tj)) and ("lesion" in (ti, tj)):
                rels.append(("lesion_near_vessel", dist_conf(di)))

        if not rels:
            continue

        rel, conf = max(rels, key=lambda x: x[1])

        edges.append({
            "source": node_names[i],
            "target": node_names[j],
            "relation": rel,
            "confidence": float(conf)
        })




edges_sorted = sorted(edges, key=lambda x: x["confidence"], reverse=True)

print("Edges:", len(edges_sorted), "\n")

for e in edges_sorted[:50]:
    print(
        f"{e['source']}  ->  {e['target']}  |  "
        f"{e['relation']}  |  conf={e['confidence']:.4f}"
    )


Edges: 45 

vessel_000  ->  vessel_005  |  overlaps  |  conf=1.0000
vessel_000  ->  vessel_006  |  overlaps  |  conf=1.0000
vessel_000  ->  vessel_008  |  overlaps  |  conf=1.0000
vessel_001  ->  vessel_002  |  overlaps  |  conf=1.0000
vessel_001  ->  vessel_004  |  overlaps  |  conf=1.0000
vessel_002  ->  vessel_004  |  overlaps  |  conf=1.0000
vessel_005  ->  vessel_006  |  overlaps  |  conf=1.0000
vessel_005  ->  vessel_008  |  overlaps  |  conf=1.0000
vessel_006  ->  vessel_008  |  overlaps  |  conf=1.0000
lesion_010  ->  lesion_011  |  near  |  conf=0.9797
vessel_002  ->  vessel_003  |  near  |  conf=0.9480
lesion_009  ->  lesion_011  |  near  |  conf=0.9024
vessel_003  ->  lesion_009  |  near  |  conf=0.8968
vessel_005  ->  vessel_007  |  overlaps  |  conf=0.8542
lesion_009  ->  lesion_010  |  near  |  conf=0.8516
vessel_006  ->  vessel_007  |  overlaps  |  conf=0.8433
vessel_007  ->  vessel_008  |  overlaps  |  conf=0.8235
vessel_002  ->  lesion_009  |  near  |  conf=0.8125
vess

In [15]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines


name_to_centroid = dict(zip(node_names, centroids))

color_map = {
    "optic_disc": "lime",
    "lesion": "red",
    "vessel": "cyan"
}

plt.figure(figsize=(12, 12))
plt.imshow(img_rgb_resized)


for edge in edges_sorted:
    n1 = edge["source"]
    n2 = edge["target"]
    rel = edge["relation"]

    c1 = name_to_centroid.get(n1)
    c2 = name_to_centroid.get(n2)

    if c1 is None or c2 is None:
        continue

    if rel == "overlaps":
        line_color = "magenta"
        line_style = "-"
    elif "near" in rel:
        line_color = "yellow"
        line_style = "--"
    else:
        line_color = "white"
        line_style = ":"

    plt.plot(
        [c1[0], c2[0]],
        [c1[1], c2[1]],
        color=line_color,
        linestyle=line_style,
        linewidth=2,
        alpha=0.7,
        zorder=2
    )


for name, centroid in name_to_centroid.items():
    if centroid is None:
        continue

    node_type = name.rsplit('_', 1)[0]
    color = color_map.get(node_type, "white")

    plt.scatter(
        centroid[0],
        centroid[1],
        c=color,
        s=100,
        edgecolors='black',
        zorder=5
    )


handles = [
    mlines.Line2D([], [], color='lime', marker='o', linestyle='None',
                  markersize=10, label='Optic Disc'),
    mlines.Line2D([], [], color='red', marker='o', linestyle='None',
                  markersize=10, label='Lesion'),
    mlines.Line2D([], [], color='cyan', marker='o', linestyle='None',
                  markersize=10, label='Vessel'),
    mlines.Line2D([], [], color='magenta', linestyle='-',
                  linewidth=2, label='Relation: Overlaps'),
    mlines.Line2D([], [], color='yellow', linestyle='--',
                  linewidth=2, label='Relation: Near')
]

plt.legend(handles=handles, loc="upper right",
           fontsize=12, facecolor='black', labelcolor='white')

plt.title("Retinal Scene Graph Visualization", fontsize=16)
plt.axis("off")
plt.tight_layout()
plt.show()


<Figure size 1200x1200 with 1 Axes>

In [16]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import os

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, "original_retinal_graph_with_confidence.png")


name_to_centroid = dict(zip(node_names, centroids))
name_to_conf = dict(zip(node_names, refined_scores))

color_map = {
    "optic_disc": "lime",
    "lesion": "red",
    "vessel": "cyan"
}

plt.figure(figsize=(12, 12))
plt.imshow(img_rgb_resized)


for edge in edges_sorted:
    n1 = edge["source"]
    n2 = edge["target"]
    rel = edge["relation"]

    c1 = name_to_centroid.get(n1)
    c2 = name_to_centroid.get(n2)

    if c1 is None or c2 is None:
        continue

    if rel == "overlaps":
        line_color = "magenta"
        line_style = "-"
    elif "near" in rel:
        line_color = "yellow"
        line_style = "--"
    else:
        line_color = "white"
        line_style = ":"

    plt.plot(
        [c1[0], c2[0]],
        [c1[1], c2[1]],
        color=line_color,
        linestyle=line_style,
        linewidth=2,
        alpha=0.8,
        zorder=2
    )


for name, centroid in name_to_centroid.items():

    if centroid is None:
        continue

    node_type = name.rsplit('_', 1)[0]
    color = color_map.get(node_type, "white")
    conf = name_to_conf.get(name, 0)

    plt.scatter(
        centroid[0],
        centroid[1],
        c=color,
        s=150,
        edgecolors='black',
        linewidths=1.5,
        zorder=5
    )

   
    plt.text(
        centroid[0] + 6,
        centroid[1] - 6,
        f"{conf:.2f}",
        color="white",
        fontsize=10,
        weight="bold",
        bbox=dict(facecolor='black', alpha=0.6, pad=2),
        zorder=6
    )


handles = [
    mlines.Line2D([], [], color='lime', marker='o', linestyle='None',
                  markersize=10, label='Optic Disc'),
    mlines.Line2D([], [], color='red', marker='o', linestyle='None',
                  markersize=10, label='Lesion'),
    mlines.Line2D([], [], color='cyan', marker='o', linestyle='None',
                  markersize=10, label='Vessel'),
    mlines.Line2D([], [], color='magenta', linestyle='-',
                  linewidth=2, label='Relation: Overlaps'),
    mlines.Line2D([], [], color='yellow', linestyle='--',
                  linewidth=2, label='Relation: Near')
]

plt.legend(
    handles=handles,
    loc="upper right",
    fontsize=12,
    facecolor='black',
    labelcolor='white'
)

plt.title("Original Retinal Scene Graph with Node Confidence", fontsize=16)
plt.axis("off")
plt.tight_layout()

plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print("Saved to:", save_path)


<Figure size 1200x1200 with 1 Axes>

Saved to: /kaggle/working/original_retinal_graph_with_confidence.png


In [17]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import slic
from skimage.util import img_as_float
import os
import random


image_float = img_as_float(img_rgb_resized)


segments = slic(
    image_float,
    n_segments=12,
    compactness=10,
    sigma=1,
    start_label=0
)

num_segments = len(np.unique(segments))
print("Number of superpixels:", num_segments)

# --- Prepare perturbation folder ---
PERT_DIR = "/kaggle/working/perturbed_samples"
os.makedirs(PERT_DIR, exist_ok=True)

perturbed_images = []
perturb_vectors = []

random.seed(42)


for i in range(50):

    mask_vector = [random.randint(0, 1) for _ in range(num_segments)]

    # avoid all-zero mask
    if sum(mask_vector) == 0:
        idx = random.randint(0, num_segments - 1)
        mask_vector[idx] = 1

    perturbed = image_float.copy()

    for seg_id in range(num_segments):
        if mask_vector[seg_id] == 0:
            perturbed[segments == seg_id] = 0

    perturbed_u8 = (perturbed * 255).astype(np.uint8)

    perturbed_images.append(perturbed_u8)
    perturb_vectors.append(mask_vector)

    save_path = os.path.join(PERT_DIR, f"pert_{i:03d}.png")
    plt.imsave(save_path, perturbed_u8)

print("Generated perturbed images:", len(perturbed_images))
print("Saved in:", PERT_DIR)


Number of superpixels: 6
Generated perturbed images: 50
Saved in: /kaggle/working/perturbed_samples


In [18]:
import torch
import numpy as np
import torch.nn.functional as F
import cv2
import random

perturbed_results = []

for pid, img_u8 in enumerate(perturbed_images):

    predictor.set_image(img_u8)
    image_embedding = predictor.get_image_embedding()
    H, W = img_u8.shape[:2]

    objects_info = []

    
    def run_box_prompt(x0, y0, x1, y1):
        box = np.array([[x0, y0, x1, y1]], dtype=np.float32)
        box_torch = torch.as_tensor(box, device=DEVICE)

        with torch.no_grad():
            sparse_embeddings, dense_embeddings = sam.prompt_encoder(
                points=None,
                boxes=box_torch,
                masks=None,
            )

            low_res_masks, iou_predictions = sam.mask_decoder(
                image_embeddings=image_embedding,
                image_pe=sam.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=True,
            )

            prob_masks = torch.sigmoid(low_res_masks)
            prob_masks = F.interpolate(
                prob_masks,
                size=(H, W),
                mode="bilinear",
                align_corners=False,
            )

        prob_masks = prob_masks.cpu().numpy()[0]
        iou_np = iou_predictions.cpu().numpy()[0]
        best_idx = int(np.argmax(iou_np))

        return prob_masks[best_idx]

    
    gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)
    _, thr = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    thr = cv2.medianBlur(thr, 9)

    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) == 0:
        perturbed_results.append({"perturb_id": pid, "objects": []})
        continue

    c = max(cnts, key=cv2.contourArea)
    (xc, yc), radius = cv2.minEnclosingCircle(c)
    xc, yc, radius = int(xc), int(yc), int(radius)

    mask_circle = np.zeros((H, W), dtype=np.uint8)
    cv2.circle(mask_circle, (xc, yc), int(radius * 0.98), 255, -1)

    
    g = img_u8[:, :, 1]
    g_blur = cv2.GaussianBlur(g, (21, 21), 0)
    search = g_blur[:, : int(0.67 * W)].copy()

    Y, X = np.ogrid[:H, :search.shape[1]]
    circle = (X - xc) ** 2 + (Y - yc) ** 2 <= radius ** 2
    search[~circle] = 0

    _, _, _, maxLoc = cv2.minMaxLoc(search)
    seed_x, seed_y = maxLoc
    box_size = int(max(60, radius * 0.22))

    soft = run_box_prompt(
        max(0, seed_x - box_size),
        max(0, seed_y - box_size),
        min(W - 1, seed_x + box_size),
        min(H - 1, seed_y + box_size),
    )

    binary = soft > 0.5
    if binary.sum() > 0:
        ys, xs = np.where(binary)
        objects_info.append({
            "type": "optic_disc",
            "centroid": (float(xs.mean()), float(ys.mean())),
            "confidence": float(soft[binary].mean()),
            "mask": binary.copy(),
            "bbox": (int(xs.min()), int(ys.min()),
                     int(xs.max()), int(ys.max()))
        })

    
    clahe = cv2.createCLAHE(2.0, (8, 8))
    g_eq = clahe.apply(g)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (19, 19))
    blackhat = cv2.morphologyEx(g_eq, cv2.MORPH_BLACKHAT, kernel)
    blackhat[mask_circle == 0] = 0

    flat = blackhat.flatten()
    if flat.max() > 0:
        thr_v = np.percentile(flat[flat > 0], 99.3)
        vessel_seeds = np.column_stack(np.where(blackhat >= thr_v)).tolist()

        random.seed(0)
        if len(vessel_seeds) > 20:
            vessel_seeds = random.sample(vessel_seeds, 20)

        for (yy, xx) in vessel_seeds:
            soft = run_box_prompt(xx-10, yy-10, xx+10, yy+10)
            binary = soft > 0.5
            if binary.sum() > 0:
                ys, xs = np.where(binary)
                objects_info.append({
                    "type": "vessel",
                    "centroid": (float(xs.mean()), float(ys.mean())),
                    "confidence": float(soft[binary].mean()),
                    "mask": binary.copy(),
                    "bbox": (int(xs.min()), int(ys.min()),
                             int(xs.max()), int(ys.max()))
                })

    
    r = img_u8[:, :, 0]
    r_blur = cv2.GaussianBlur(r, (9, 9), 0)
    les = r_blur.copy()
    les[mask_circle == 0] = 0

    vals = les[les > 0]
    if vals.size > 0:
        t = np.percentile(vals, 99.5)
        bin_les = (les >= t).astype(np.uint8) * 255
        bin_les = cv2.medianBlur(bin_les, 5)

        num, lbl, stats, _ = cv2.connectedComponentsWithStats(bin_les, connectivity=8)

        for i in range(1, num):
            x, y, w, h, area = stats[i]
            if 50 <= area <= 8000:
                pad = 10
                soft = run_box_prompt(
                    max(0, x - pad),
                    max(0, y - pad),
                    min(W - 1, x + w + pad),
                    min(H - 1, y + h + pad),
                )

                binary = soft > 0.5
                if binary.sum() > 0:
                    ys, xs = np.where(binary)
                    objects_info.append({
                        "type": "lesion",
                        "centroid": (float(xs.mean()), float(ys.mean())),
                        "confidence": float(soft[binary].mean()),
                        "mask": binary.copy(),
                        "bbox": (int(xs.min()), int(ys.min()),
                                 int(xs.max()), int(ys.max()))
                    })

    
    perturbed_results.append({
        "perturb_id": pid,
        "objects": objects_info
    })

print("Processed perturbed images:", len(perturbed_results))


Processed perturbed images: 50


In [19]:
import torch
import torch.nn.functional as F
import numpy as np

def get_confidence_from_bbox(image_u8, bbox):
    
    predictor.set_image(image_u8)
    image_embedding = predictor.get_image_embedding()
    H, W = image_u8.shape[:2]

    box = np.array([bbox], dtype=np.float32)
    box_torch = torch.as_tensor(box, device=DEVICE)

    with torch.no_grad():
        sparse_embeddings, dense_embeddings = sam.prompt_encoder(
            points=None,
            boxes=box_torch,
            masks=None,
        )

        low_res_masks, iou_predictions = sam.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
        )

        prob_masks = torch.sigmoid(low_res_masks)
        prob_masks = F.interpolate(
            prob_masks,
            size=(H, W),
            mode="bilinear",
            align_corners=False,
        )

    prob_masks = prob_masks.cpu().numpy()[0]
    iou_np = iou_predictions.cpu().numpy()[0]
    best_idx = int(np.argmax(iou_np))

    best_soft = prob_masks[best_idx]
    binary = best_soft > 0.5

    if binary.sum() == 0:
        return 0.0

    return float(best_soft[binary].mean())


In [20]:
original_masks = []

for m in refined_masks:
    original_masks.append(m.copy())

print("Original masks stored:", len(original_masks))


Original masks stored: 12


In [21]:
original_bboxes = []

for mask in original_masks:
    ys, xs = np.where(mask)
    x0, y0 = int(xs.min()), int(ys.min())
    x1, y1 = int(xs.max()), int(ys.max())
    original_bboxes.append((x0, y0, x1, y1))


In [22]:
original_nodes = node_names.copy()
num_original_nodes = len(original_nodes)

print("Rebuilt original nodes:", num_original_nodes)


Rebuilt original nodes: 12


In [23]:
original_nodes = node_names.copy()


original_conf = refined_scores.copy()


original_masks = refined_masks.copy()


original_bboxes = []

for m in original_masks:
    ys, xs = np.where(m)
    if len(xs) == 0:
        original_bboxes.append(None)
    else:
        original_bboxes.append((
            int(xs.min()),
            int(ys.min()),
            int(xs.max()),
            int(ys.max())
        ))


num_original_nodes = len(original_nodes)

print("Original nodes rebuilt:", num_original_nodes)


Original nodes rebuilt: 12


In [24]:
import pandas as pd

node_shift_rows = []

for node_id in range(num_original_nodes):

    orig_name = original_nodes[node_id]
    orig_conf = original_conf[node_id]
    orig_bbox = original_bboxes[node_id]

    for pid in range(len(perturbed_images)):

        pert_img = perturbed_images[pid]

        new_conf = get_confidence_from_bbox(pert_img, orig_bbox)

        shift = orig_conf - new_conf

        node_shift_rows.append({
            "node_id": node_id,
            "node_name": orig_name,
            "perturb_id": pid,
            "orig_conf": orig_conf,
            "pert_conf": new_conf,
            "shift": shift
        })

node_shift_df = pd.DataFrame(node_shift_rows)

print("New Shift Table Created (Region-Fixed Method)")
display(node_shift_df.head())


New Shift Table Created (Region-Fixed Method)


   node_id   node_name  perturb_id  orig_conf  pert_conf     shift
0        0  vessel_000           0   0.893065   0.891799  0.001266
1        0  vessel_000           1   0.893065   0.891799  0.001266
2        0  vessel_000           2   0.893065   0.894483 -0.001418
3        0  vessel_000           3   0.893065   0.764860  0.128205
4        0  vessel_000           4   0.893065   0.923916 -0.030851

In [25]:
node_shift_matrix = node_shift_df.pivot(
    index="perturb_id",
    columns="node_name",
    values="shift"
).sort_index()

display(node_shift_matrix.head())


node_name   lesion_009  lesion_010  lesion_011  vessel_000  vessel_001  \
perturb_id                                                               
0             0.009430    0.027726    0.056277    0.001266   -0.064492   
1             0.009430    0.027726    0.056277    0.001266   -0.064492   
2             0.009632    0.028923    0.056233   -0.001418   -0.059473   
3             0.142181    0.161395    0.740578    0.128205   -0.063244   
4             0.010738    0.040936    0.060910   -0.030851    0.057193   

node_name   vessel_002  vessel_003  vessel_004  vessel_005  vessel_006  \
perturb_id                                                               
0            -0.205656   -0.142530   -0.035148   -0.156367   -0.155243   
1            -0.205656   -0.142530   -0.035148   -0.156367   -0.155243   
2            -0.204493   -0.140526   -0.034386   -0.136411   -0.136192   
3            -0.223364   -0.106398   -0.007307   -0.164660   -0.164218   
4            -0.202270   -0.088582   

In [26]:
# ============================================================
# NEW CELL: Build cosine-distance weights from MedSAM embeddings
# Uses existing predictor, img_rgb_resized, and perturbed_images
# ============================================================

import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Safety checks
# -----------------------------
required_vars = ["predictor", "img_rgb_resized", "perturbed_images"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Missing required variables: {missing_vars}")

if len(perturbed_images) == 0:
    raise ValueError("perturbed_images is empty.")

# -----------------------------
# Helper: extract pooled image embedding
# predictor.get_image_embedding() usually returns [1, C, H, W]
# We global-average-pool it to [C]
# -----------------------------
def extract_pooled_embedding(image_u8):
    predictor.set_image(image_u8)
    with torch.no_grad():
        emb = predictor.get_image_embedding()

    if torch.is_tensor(emb):
        emb = emb.detach().cpu().numpy()

    emb = np.asarray(emb, dtype=np.float32)

    if emb.ndim == 4:
        # [1, C, H, W] -> [C]
        emb = emb.mean(axis=(0, 2, 3))
    elif emb.ndim == 3:
        # [C, H, W] -> [C]
        emb = emb.mean(axis=(1, 2))
    elif emb.ndim == 2:
        emb = emb.mean(axis=0)
    elif emb.ndim == 1:
        pass
    else:
        raise ValueError(f"Unexpected embedding shape: {emb.shape}")

    return emb.astype(np.float32)

# -----------------------------
# Extract original embedding
# -----------------------------
original_embedding = extract_pooled_embedding(img_rgb_resized)

# -----------------------------
# Extract perturbed embeddings
# -----------------------------
perturbed_embeddings = []
for i, img_u8 in enumerate(perturbed_images):
    emb_i = extract_pooled_embedding(img_u8)
    perturbed_embeddings.append(emb_i)

perturbed_embeddings = np.stack(perturbed_embeddings, axis=0)

# -----------------------------
# Check dimensions
# -----------------------------
if perturbed_embeddings.shape[1] != original_embedding.shape[0]:
    raise ValueError(
        f"Embedding dimension mismatch: "
        f"original={original_embedding.shape[0]}, "
        f"perturbed={perturbed_embeddings.shape[1]}"
    )

if "perturb_vectors" in globals():
    if len(perturbed_embeddings) != len(perturb_vectors):
        raise ValueError(
            f"Mismatch: {len(perturbed_embeddings)} perturbed embeddings vs "
            f"{len(perturb_vectors)} perturb_vectors."
        )

# -----------------------------
# Cosine similarity / distance
# -----------------------------
cos_sims = cosine_similarity(
    perturbed_embeddings,
    original_embedding.reshape(1, -1)
).flatten()

cos_dists = 1.0 - cos_sims
cos_dists = np.clip(cos_dists, 0.0, 2.0)

# -----------------------------
# Kernel weights from cosine distance
# -----------------------------
positive_dists = cos_dists[cos_dists > 0]

if len(positive_dists) > 0:
    sigma_cos = float(np.median(positive_dists))
else:
    sigma_cos = 0.25

if sigma_cos <= 0:
    sigma_cos = 0.25

kernel_weight_from_cosdist = np.exp(-(cos_dists ** 2) / (sigma_cos ** 2))

# -----------------------------
# Build distance_df
# -----------------------------
distance_df = pd.DataFrame({
    "perturb_id": np.arange(len(cos_dists)),
    "cosine_similarity_to_original": cos_sims,
    "cosine_distance_to_original": cos_dists,
    "kernel_weight_from_cosdist": kernel_weight_from_cosdist
})

print("distance_df created successfully from MedSAM image embeddings.")
print("Original embedding shape:", original_embedding.shape)
print("Perturbed embeddings shape:", perturbed_embeddings.shape)
print("Sigma used:", round(sigma_cos, 6))

display(distance_df.head())

print("\nCosine distance stats:")
display(distance_df["cosine_distance_to_original"].describe().to_frame().T)

print("\nKernel weight stats:")
display(distance_df["kernel_weight_from_cosdist"].describe().to_frame().T)

distance_df.to_csv("/kaggle/working/distance_df_from_medsam_cosine.csv", index=False)
print("Saved: /kaggle/working/distance_df_from_medsam_cosine.csv")

distance_df created successfully from MedSAM image embeddings.
Original embedding shape: (256,)
Perturbed embeddings shape: (50, 256)
Sigma used: 0.0871


   perturb_id  cosine_similarity_to_original  cosine_distance_to_original  \
0           0                       0.803547                     0.196453   
1           1                       0.803547                     0.196453   
2           2                       0.826111                     0.173889   
3           3                       0.979453                     0.020547   
4           4                       0.903171                     0.096829   

   kernel_weight_from_cosdist  
0                    0.006175  
1                    0.006175  
2                    0.018580  
3                    0.945873  
4                    0.290587  


Cosine distance stats:


                             count      mean       std       min       25%  \
cosine_distance_to_original   50.0  0.091848  0.061188  0.001431  0.042844   

                                50%       75%       max  
cosine_distance_to_original  0.0871  0.146233  0.196453  


Kernel weight stats:


                            count      mean       std       min       25%  \
kernel_weight_from_cosdist   50.0  0.450386  0.363603  0.006175  0.060987   

                                 50%       75%      max  
kernel_weight_from_cosdist  0.367913  0.785091  0.99973  

Saved: /kaggle/working/distance_df_from_medsam_cosine.csv


In [27]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

fidelity_results = []

X = np.array(perturb_vectors)

for node in node_shift_matrix.columns:

    y = node_shift_matrix[node].values

    if np.std(y) < 1e-8:
        r2 = 0.0
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )

        model = XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)

    fidelity_results.append({
        "node_name": node,
        "fidelity_R2": r2
    })

fidelity_df = pd.DataFrame(fidelity_results).sort_values(
    "fidelity_R2", ascending=False
)

print("\nReal Node Fidelity Results:")
display(fidelity_df)



Real Node Fidelity Results:


     node_name  fidelity_R2
0   lesion_009     0.989857
3   vessel_000     0.963779
4   vessel_001     0.950225
8   vessel_005     0.931491
5   vessel_002     0.929663
9   vessel_006     0.919357
1   lesion_010     0.915521
2   lesion_011     0.907794
11  vessel_008     0.906999
7   vessel_004     0.877455
10  vessel_007     0.820182
6   vessel_003     0.617157

In [28]:
# ============================================================
# NEW CELL AFTER YOUR CURRENT FIDELITY CELL
# Complete node-level fidelity table + variance
# ============================================================

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# -----------------------------
# Safety checks
# -----------------------------
if "perturb_vectors" not in globals():
    raise ValueError("perturb_vectors is missing.")

if "node_shift_matrix" not in globals():
    raise ValueError("node_shift_matrix is missing.")

X_all = np.array(perturb_vectors, dtype=float)

if X_all.ndim != 2:
    raise ValueError("perturb_vectors must be a 2D array.")

if len(node_shift_matrix) == 0:
    raise ValueError("node_shift_matrix is empty.")


# -----------------------------
# Weight utilities
# -----------------------------
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    w_sum = np.sum(w)
    if len(x) == 0:
        return np.nan
    if w_sum <= 0:
        return np.mean(x)
    return np.sum(w * x) / w_sum

def weighted_mse(y_true, y_pred, w):
    return weighted_mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2, w)

def weighted_mae(y_true, y_pred, w):
    return weighted_mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), w)

def weighted_r2(y_true, y_pred, w):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(w, dtype=float)

    if len(y_true) < 2:
        return np.nan

    y_bar = weighted_mean(y_true, w)
    ss_res = np.sum(w * (y_true - y_pred) ** 2)
    ss_tot = np.sum(w * (y_true - y_bar) ** 2)

    if ss_tot <= 1e-12:
        return np.nan

    return 1.0 - (ss_res / ss_tot)

def kish_effective_n(w):
    w = np.asarray(w, dtype=float)
    denom = np.sum(w ** 2)
    if denom <= 0:
        return len(w)
    return (np.sum(w) ** 2) / denom

def weighted_adjusted_r2(y_true, y_pred, w, p):
    r2w = weighted_r2(y_true, y_pred, w)
    if pd.isna(r2w):
        return np.nan

    n_eff = kish_effective_n(w)

    if n_eff <= (p + 1):
        return np.nan

    return 1.0 - (1.0 - r2w) * ((n_eff - 1.0) / (n_eff - p - 1.0))


# -----------------------------
# Build sample weights
# -----------------------------
def get_weight_map(n_rows):
    # default: equal weights
    weight_map = pd.Series(np.ones(n_rows, dtype=float), index=np.arange(n_rows))

    if "distance_df" not in globals():
        print("distance_df not found -> using equal weights.")
        return weight_map, "equal_weights"

    if not isinstance(distance_df, pd.DataFrame) or distance_df.empty:
        print("distance_df invalid/empty -> using equal weights.")
        return weight_map, "equal_weights"

    if "perturb_id" not in distance_df.columns:
        print("distance_df has no perturb_id -> using equal weights.")
        return weight_map, "equal_weights"

    if "kernel_weight_from_cosdist" in distance_df.columns:
        wcol = "kernel_weight_from_cosdist"
    elif "kernel_weight" in distance_df.columns:
        wcol = "kernel_weight"
    else:
        print("No usable weight column in distance_df -> using equal weights.")
        return weight_map, "equal_weights"

    tmp = distance_df[["perturb_id", wcol]].copy()
    tmp[wcol] = pd.to_numeric(tmp[wcol], errors="coerce").fillna(1.0)
    tmp = tmp.drop_duplicates(subset=["perturb_id"]).set_index("perturb_id")[wcol]

    return tmp, wcol


# -----------------------------
# Main repeated-run fidelity
# -----------------------------
N_REPEATS = 30
TEST_SIZE = 0.30

weight_map, weight_source = get_weight_map(len(X_all))

all_runs = []

for node in node_shift_matrix.columns:
    y = node_shift_matrix[node].values.astype(float)

    # remove NaN rows safely
    valid_mask = ~np.isnan(y)
    X = X_all[valid_mask]
    y = y[valid_mask]

    valid_ids = np.where(valid_mask)[0]
    w = weight_map.reindex(valid_ids).fillna(1.0).values.astype(float)

    if len(y) < 8:
        all_runs.append({
            "node_name": node,
            "repeat_id": np.nan,
            "status": "too_few_samples",
            "MSE": np.nan,
            "MAE": np.nan,
            "Lm": np.nan,
            "mean_L1": np.nan,
            "mean_L2": np.nan,
            "WMSE": np.nan,
            "WMAE": np.nan,
            "weighted_R2": np.nan,
            "weighted_adjusted_R2": np.nan,
            "R2": np.nan
        })
        continue

    if np.std(y) < 1e-8:
        for rep in range(N_REPEATS):
            all_runs.append({
                "node_name": node,
                "repeat_id": rep,
                "status": "no_variation",
                "MSE": 0.0,
                "MAE": 0.0,
                "Lm": 0.0,
                "mean_L1": 0.0,
                "mean_L2": 0.0,
                "WMSE": 0.0,
                "WMAE": 0.0,
                "weighted_R2": np.nan,
                "weighted_adjusted_R2": np.nan,
                "R2": np.nan
            })
        continue

    for rep in range(N_REPEATS):
        X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
            X, y, w,
            test_size=TEST_SIZE,
            random_state=42 + rep
        )

        model = XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42 + rep
        )

        model.fit(X_train, y_train, sample_weight=w_train)
        y_pred = model.predict(X_test)

        mse_val = mean_squared_error(y_test, y_pred)
        mae_val = mean_absolute_error(y_test, y_pred)
        mean_l1_val = np.mean(np.abs(y_test - y_pred))
        mean_l2_val = np.mean((y_test - y_pred) ** 2)

        # thesis-friendly combined loss
        lm_val = 0.5 * (mean_l1_val + mean_l2_val)

        wmse_val = weighted_mse(y_test, y_pred, w_test)
        wmae_val = weighted_mae(y_test, y_pred, w_test)
        wr2_val = weighted_r2(y_test, y_pred, w_test)
        wr2_adj_val = weighted_adjusted_r2(
            y_true=y_test,
            y_pred=y_pred,
            w=w_test,
            p=X_test.shape[1]
        )

        all_runs.append({
            "node_name": node,
            "repeat_id": rep,
            "status": "ok",
            "MSE": mse_val,
            "MAE": mae_val,
            "Lm": lm_val,
            "mean_L1": mean_l1_val,
            "mean_L2": mean_l2_val,
            "WMSE": wmse_val,
            "WMAE": wmae_val,
            "weighted_R2": wr2_val,
            "weighted_adjusted_R2": wr2_adj_val,
            "R2": r2_score(y_test, y_pred)
        })

node_fidelity_runs_df = pd.DataFrame(all_runs)

# -----------------------------
# Mean table per node
# -----------------------------
node_fidelity_complete_table = (
    node_fidelity_runs_df
    .groupby("node_name", as_index=False)
    .agg({
        "MSE": "mean",
        "MAE": "mean",
        "Lm": "mean",
        "mean_L1": "mean",
        "mean_L2": "mean",
        "WMSE": "mean",
        "WMAE": "mean",
        "R2": "mean",
        "weighted_R2": "mean",
        "weighted_adjusted_R2": "mean"
    })
)

# -----------------------------
# Variance table per node
# -----------------------------
node_fidelity_variance_table = (
    node_fidelity_runs_df
    .groupby("node_name", as_index=False)
    .agg({
        "R2": ["var", "std"],
        "weighted_R2": ["var", "std"]
    })
)

node_fidelity_variance_table.columns = [
    "node_name",
    "var_R2", "std_R2",
    "var_weighted_R2", "std_weighted_R2"
]

# -----------------------------
# Final merged table
# -----------------------------
node_fidelity_complete_table = node_fidelity_complete_table.merge(
    node_fidelity_variance_table,
    on="node_name",
    how="left"
).sort_values("weighted_R2", ascending=False)

print("Complete node-level fidelity table")
print("Weight source:", weight_source)
display(node_fidelity_complete_table.round(4))

# save
node_fidelity_runs_df.to_csv("/kaggle/working/node_fidelity_runs_df.csv", index=False)
node_fidelity_complete_table.to_csv("/kaggle/working/node_fidelity_complete_table.csv", index=False)

print("Saved:")
print("/kaggle/working/node_fidelity_runs_df.csv")
print("/kaggle/working/node_fidelity_complete_table.csv")

Complete node-level fidelity table
Weight source: kernel_weight_from_cosdist


     node_name     MSE     MAE      Lm  mean_L1  mean_L2    WMSE    WMAE  \
2   lesion_011  0.0025  0.0374  0.0199   0.0374   0.0025  0.0022  0.0326   
1   lesion_010  0.0001  0.0080  0.0041   0.0080   0.0001  0.0001  0.0075   
4   vessel_001  0.0004  0.0156  0.0080   0.0156   0.0004  0.0002  0.0111   
0   lesion_009  0.0004  0.0145  0.0075   0.0145   0.0004  0.0002  0.0088   
11  vessel_008  0.0000  0.0046  0.0023   0.0046   0.0000  0.0000  0.0033   
9   vessel_006  0.0000  0.0044  0.0022   0.0044   0.0000  0.0000  0.0033   
8   vessel_005  0.0000  0.0046  0.0023   0.0046   0.0000  0.0000  0.0034   
5   vessel_002  0.0003  0.0127  0.0065   0.0127   0.0003  0.0003  0.0117   
10  vessel_007  0.0011  0.0250  0.0130   0.0250   0.0011  0.0012  0.0249   
3   vessel_000  0.0009  0.0208  0.0109   0.0208   0.0009  0.0008  0.0181   
6   vessel_003  0.0006  0.0146  0.0076   0.0146   0.0006  0.0004  0.0118   
7   vessel_004  0.0002  0.0108  0.0055   0.0108   0.0002  0.0003  0.0119   

        R2 

Saved:
/kaggle/working/node_fidelity_runs_df.csv
/kaggle/working/node_fidelity_complete_table.csv


In [29]:
# ============================================================
# NEW CELL: Thesis-style concept-level fidelity table
# Groups node-level results into lesion / vessel / optic_disc
# ============================================================

import numpy as np
import pandas as pd

# -----------------------------
# Safety check
# -----------------------------
if "node_fidelity_complete_table" not in globals():
    raise ValueError(
        "node_fidelity_complete_table is missing. Run the complete node-level fidelity cell first."
    )

df = node_fidelity_complete_table.copy()

if "node_name" not in df.columns:
    raise ValueError("node_fidelity_complete_table must contain 'node_name' column.")

# -----------------------------
# Map each node to thesis concept
# -----------------------------
def map_concept(node_name):
    s = str(node_name).lower().strip().replace("-", "_").replace(" ", "_")
    if "lesion" in s:
        return "lesion"
    elif "vessel" in s or "blood_vessels" in s:
        return "vessel"
    elif "optic" in s and ("disc" in s or "disk" in s):
        return "optic_disc"
    else:
        return np.nan

df["concept"] = df["node_name"].apply(map_concept)
df = df.dropna(subset=["concept"]).copy()

if df.empty:
    raise ValueError("No lesion / vessel / optic_disc nodes were found in node_fidelity_complete_table.")

# -----------------------------
# Aggregate by concept
# -----------------------------
concept_level_fidelity_table = (
    df.groupby("concept", as_index=False)
      .agg({
          "MSE": "mean",
          "MAE": "mean",
          "Lm": "mean",
          "mean_L1": "mean",
          "mean_L2": "mean",
          "WMSE": "mean",
          "WMAE": "mean",
          "R2": "mean",
          "weighted_R2": "mean",
          "weighted_adjusted_R2": "mean",
          "var_R2": "mean",
          "std_R2": "mean",
          "var_weighted_R2": "mean",
          "std_weighted_R2": "mean"
      })
)

# -----------------------------
# Rename for thesis-friendly display
# -----------------------------
concept_level_fidelity_table = concept_level_fidelity_table.rename(columns={
    "concept": "Concept",
    "MSE": "MSE",
    "MAE": "MAE",
    "Lm": "Lm",
    "mean_L1": "mean-L1",
    "mean_L2": "mean-L2",
    "WMSE": "WMSE",
    "WMAE": "WMAE",
    "R2": "R2",
    "weighted_R2": "R2ω",
    "weighted_adjusted_R2": "R̂2ω",
    "var_R2": "Var(R2)",
    "std_R2": "Std(R2)",
    "var_weighted_R2": "Var(R2ω)",
    "std_weighted_R2": "Std(R2ω)"
})

# Optional ordering
order_map = {"lesion": 0, "vessel": 1, "optic_disc": 2}
concept_level_fidelity_table["__order"] = concept_level_fidelity_table["Concept"].map(order_map)
concept_level_fidelity_table = concept_level_fidelity_table.sort_values("__order").drop(columns="__order")

print("Thesis-style concept-level fidelity table:")
display(concept_level_fidelity_table.round(4))

# Save
concept_level_fidelity_table.to_csv("/kaggle/working/concept_level_fidelity_table.csv", index=False)
print("Saved: /kaggle/working/concept_level_fidelity_table.csv")

Thesis-style concept-level fidelity table:


  Concept     MSE     MAE      Lm  mean-L1  mean-L2    WMSE    WMAE      R2  \
0  lesion  0.0010  0.0199  0.0105   0.0199   0.0010  0.0008  0.0163  0.9307   
1  vessel  0.0004  0.0126  0.0065   0.0126   0.0004  0.0004  0.0110  0.6377   

      R2ω    R̂2ω  Var(R2)  Std(R2)  Var(R2ω)  Std(R2ω)  
0  0.9594  0.8206   0.0048   0.0509    0.0013    0.0335  
1  0.6600 -0.3508   0.3798   0.3573    0.4779    0.4368  

Saved: /kaggle/working/concept_level_fidelity_table.csv


In [30]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

plt.figure(figsize=(10, 8))

colors = plt.cm.tab20(np.linspace(0, 1, len(original_nodes)))

for i, node_name in enumerate(original_nodes):

    node_df = node_shift_df[node_shift_df["node_name"] == node_name]
    node_df = node_df.sort_values("perturb_id")

    y = node_df["shift"].values
    X = np.array(perturb_vectors)

    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    
    y_pred_test = model.predict(X_test)

    r2 = r2_score(y_test, y_pred_test)

    
    plt.scatter(
        y_test,
        y_pred_test,
        alpha=0.7,
        s=60,
        color=colors[i],
        label=f"{node_name} (R²={r2:.2f})"
    )


all_shifts = node_shift_df["shift"]
min_val = all_shifts.min()
max_val = all_shifts.max()

plt.plot([min_val, max_val],
         [min_val, max_val],
         'k--', linewidth=2)

plt.xlabel("Actual Confidence Shift (Test)")
plt.ylabel("Predicted Confidence Shift (Test)")
plt.title("Real Node Fidelity (Test R²) - XGBoost")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


<Figure size 1000x800 with 1 Axes>

In [31]:
# ============================================================
# REBUILD + FILL node_att_reference_df from scratch
# ============================================================

import numpy as np
import pandas as pd

required_vars = ["node_shift_df", "original_nodes", "original_masks", "segments", "perturb_vectors"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Missing required variables: {missing_vars}")

required_cols = {"perturb_id", "node_name", "shift"}
missing_cols = required_cols - set(node_shift_df.columns)
if missing_cols:
    raise ValueError(f"node_shift_df is missing required columns: {missing_cols}")

# 1) rebuild reference table from node_shift_df
node_att_reference_df = (
    node_shift_df[["perturb_id", "node_name"]]
    .drop_duplicates()
    .sort_values(["node_name", "perturb_id"])
    .reset_index(drop=True)
)

segments_np = np.array(segments)
X_perturb = np.array(perturb_vectors, dtype=int)

if X_perturb.ndim != 2:
    raise ValueError("perturb_vectors must be 2D.")

# IMPORTANT: map actual segment labels -> perturb vector column indices
segment_labels = np.sort(np.unique(segments_np))
if X_perturb.shape[1] != len(segment_labels):
    raise ValueError(
        f"Mismatch: perturb_vectors has {X_perturb.shape[1]} columns but segments has {len(segment_labels)} unique labels."
    )

seglabel_to_col = {int(lbl): i for i, lbl in enumerate(segment_labels)}

if len(original_nodes) != len(original_masks):
    raise ValueError("original_nodes and original_masks must have same length.")

node_to_mask = {}
for n, m in zip(original_nodes, original_masks):
    node_to_mask[str(n)] = np.array(m).astype(bool)

missing_nodes = [n for n in node_att_reference_df["node_name"].unique() if str(n) not in node_to_mask]
if missing_nodes:
    raise ValueError(f"These node names are missing from original_nodes: {missing_nodes[:10]}")

AFFECTED_THRESHOLD = 0.10

y_true_list = []
affected_fraction_list = []

for _, row in node_att_reference_df.iterrows():
    pid = int(row["perturb_id"])
    node_name = str(row["node_name"])

    if pid < 0 or pid >= len(X_perturb):
        y_true_list.append(np.nan)
        affected_fraction_list.append(np.nan)
        continue

    keep_vector = X_perturb[pid]
    node_mask = node_to_mask[node_name]

    if node_mask.sum() == 0:
        y_true_list.append(0)
        affected_fraction_list.append(0.0)
        continue

    overlapping_seg_labels = np.unique(segments_np[node_mask])

    total_node_pixels = float(node_mask.sum())
    affected_fraction = 0.0

    for seg_label in overlapping_seg_labels:
        seg_col = seglabel_to_col[int(seg_label)]
        overlap_pixels = np.logical_and(node_mask, segments_np == seg_label).sum()
        frac = overlap_pixels / total_node_pixels

        # 1 = kept, 0 = removed
        if keep_vector[seg_col] == 0:
            affected_fraction += frac

    y_true = int(affected_fraction >= AFFECTED_THRESHOLD)

    y_true_list.append(y_true)
    affected_fraction_list.append(float(affected_fraction))

node_att_reference_df["affected_fraction"] = affected_fraction_list
node_att_reference_df["y_true"] = y_true_list

print("Rebuilt node_att_reference_df successfully.")
print("Rows:", len(node_att_reference_df))
print("NaN y_true count:", int(node_att_reference_df["y_true"].isna().sum()))

display(node_att_reference_df.head(20))

print("\nClass balance:")
display(
    node_att_reference_df["y_true"]
    .value_counts(dropna=False)
    .rename_axis("y_true")
    .reset_index(name="count")
)

print("\nPer-node class balance:")
display(
    node_att_reference_df.groupby("node_name")["y_true"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={"sum": "positives", "mean": "positive_rate"})
    .sort_values("positive_rate", ascending=False)
)

Rebuilt node_att_reference_df successfully.
Rows: 600
NaN y_true count: 0


    perturb_id   node_name  affected_fraction  y_true
0            0  lesion_009                1.0       1
1            1  lesion_009                1.0       1
2            2  lesion_009                1.0       1
3            3  lesion_009                0.0       0
4            4  lesion_009                1.0       1
5            5  lesion_009                1.0       1
6            6  lesion_009                1.0       1
7            7  lesion_009                1.0       1
8            8  lesion_009                1.0       1
9            9  lesion_009                0.0       0
10          10  lesion_009                1.0       1
11          11  lesion_009                0.0       0
12          12  lesion_009                0.0       0
13          13  lesion_009                0.0       0
14          14  lesion_009                0.0       0
15          15  lesion_009                0.0       0
16          16  lesion_009                0.0       0
17          17  lesion_009  


Class balance:


   y_true  count
0       1    431
1       0    169


Per-node class balance:


     node_name  count  positives  positive_rate
10  vessel_007     50         45           0.90
5   vessel_002     50         40           0.80
4   vessel_001     50         40           0.80
7   vessel_004     50         40           0.80
9   vessel_006     50         39           0.78
8   vessel_005     50         39           0.78
11  vessel_008     50         39           0.78
3   vessel_000     50         39           0.78
6   vessel_003     50         29           0.58
2   lesion_011     50         27           0.54
0   lesion_009     50         27           0.54
1   lesion_010     50         27           0.54

In [32]:
# ============================================================
# ATT metrics without overall one-row table
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score

if "node_shift_df" not in globals():
    raise ValueError("node_shift_df is missing.")

if "node_att_reference_df" not in globals():
    raise ValueError("node_att_reference_df is missing.")

if node_att_reference_df["y_true"].isna().any():
    raise ValueError("node_att_reference_df['y_true'] still contains NaN values.")

att_score_df = node_shift_df[["perturb_id", "node_name", "shift"]].copy()
att_score_df["att_score"] = att_score_df["shift"].abs()

att_eval_df = att_score_df.merge(
    node_att_reference_df[["perturb_id", "node_name", "y_true"]],
    on=["perturb_id", "node_name"],
    how="inner"
)

if att_eval_df.empty:
    raise ValueError("ATT evaluation dataframe is empty after merge.")

att_eval_df["y_true"] = att_eval_df["y_true"].astype(int)

def map_concept(node_name):
    s = str(node_name).lower().strip().replace("-", "_").replace(" ", "_")
    if "lesion" in s:
        return "lesion"
    elif "vessel" in s or "blood_vessels" in s:
        return "vessel"
    elif "optic" in s and ("disc" in s or "disk" in s):
        return "optic_disc"
    else:
        return np.nan

node_att_rows = []

for node_name, g in att_eval_df.groupby("node_name"):
    g = g.dropna(subset=["att_score", "y_true"]).copy()

    if len(g) < 2 or g["y_true"].nunique() < 2:
        node_att_rows.append({
            "node_name": node_name,
            "concept": map_concept(node_name),
            "ATT_ACC": np.nan,
            "ATT_F1": np.nan,
            "ATT_AUROC": np.nan,
            "best_threshold": np.nan,
            "n_samples": len(g),
            "status": "single_class_or_too_few_samples"
        })
        continue

    y_true = g["y_true"].values.astype(int)
    att_score = g["att_score"].values.astype(float)

    att_auc = roc_auc_score(y_true, att_score)

    fpr, tpr, thresholds = roc_curve(y_true, att_score)
    j_scores = tpr - fpr
    best_idx = int(np.argmax(j_scores))
    thr_best = thresholds[best_idx]

    y_pred = (att_score >= thr_best).astype(int)

    att_acc = accuracy_score(y_true, y_pred)
    att_f1 = f1_score(y_true, y_pred, zero_division=0)

    node_att_rows.append({
        "node_name": node_name,
        "concept": map_concept(node_name),
        "ATT_ACC": att_acc,
        "ATT_F1": att_f1,
        "ATT_AUROC": att_auc,
        "best_threshold": thr_best,
        "n_samples": len(g),
        "status": "ok"
    })

node_level_att_metrics = pd.DataFrame(node_att_rows).sort_values(
    "ATT_AUROC", ascending=False
)

print("Node-level ATT metrics:")
display(node_level_att_metrics.round(4))

concept_level_att_metrics = (
    node_level_att_metrics
    .dropna(subset=["concept"])
    .groupby("concept", as_index=False)
    .agg({
        "ATT_ACC": "mean",
        "ATT_F1": "mean",
        "ATT_AUROC": "mean"
    })
)

concept_level_att_metrics = concept_level_att_metrics.rename(columns={
    "concept": "Concept",
    "ATT_ACC": "ATT ACC",
    "ATT_F1": "ATT F1",
    "ATT_AUROC": "ATT AUROC"
})

order_map = {"lesion": 0, "vessel": 1, "optic_disc": 2}
concept_level_att_metrics["__order"] = concept_level_att_metrics["Concept"].map(order_map)
concept_level_att_metrics = concept_level_att_metrics.sort_values("__order").drop(columns="__order")

print("\nConcept-level ATT table:")
display(concept_level_att_metrics.round(4))

node_level_att_metrics.to_csv("/kaggle/working/node_level_att_metrics.csv", index=False)
concept_level_att_metrics.to_csv("/kaggle/working/concept_level_att_metrics.csv", index=False)

print("Saved:")
print("/kaggle/working/node_level_att_metrics.csv")
print("/kaggle/working/concept_level_att_metrics.csv")

Node-level ATT metrics:


     node_name concept  ATT_ACC  ATT_F1  ATT_AUROC  best_threshold  n_samples  \
5   vessel_002  vessel     1.00  1.0000     1.0000          0.1773         50   
6   vessel_003  vessel     0.84  0.8788     0.7947          0.1064         50   
3   vessel_000  vessel     0.52  0.5556     0.3846          0.0314         50   
10  vessel_007  vessel     0.36  0.4483     0.3689          0.1972         50   
7   vessel_004  vessel     0.82  0.8989     0.1000          0.0073         50   
9   vessel_006  vessel     0.22  0.0000     0.0839             inf         50   
8   vessel_005  vessel     0.22  0.0000     0.0839             inf         50   
11  vessel_008  vessel     0.22  0.0000     0.0396             inf         50   
2   lesion_011  lesion     0.46  0.0000     0.0000             inf         50   
1   lesion_010  lesion     0.46  0.0000     0.0000             inf         50   
0   lesion_009  lesion     0.46  0.0000     0.0000             inf         50   
4   vessel_001  vessel     0


Concept-level ATT table:


  Concept  ATT ACC  ATT F1  ATT AUROC
0  lesion   0.4600  0.0000     0.0000
1  vessel   0.4889  0.4202     0.3173

Saved:
/kaggle/working/node_level_att_metrics.csv
/kaggle/working/concept_level_att_metrics.csv


In [33]:
# =========================================
# Node-level Faithfulness Table from saved fidelity file
# Faithfulness = R2
# =========================================

import pandas as pd
import numpy as np

fidelity_path = "/kaggle/working/node_fidelity_complete_table.csv"

df = pd.read_csv(fidelity_path)

required_cols = ["node_name", "R2"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in node_fidelity_complete_table.csv: {missing}")

df["R2"] = pd.to_numeric(df["R2"], errors="coerce")
df = df.dropna(subset=["node_name", "R2"]).copy()

faithfulness_table_df = df[["node_name", "R2"]].copy()
faithfulness_table_df = faithfulness_table_df.rename(columns={
    "node_name": "Concept",
    "R2": "Faithfulness"
})
faithfulness_table_df["Faithfulness"] = faithfulness_table_df["Faithfulness"].round(4)

overall_row = pd.DataFrame([{
    "Concept": "Overall_MedSAM",
    "Faithfulness": round(faithfulness_table_df["Faithfulness"].mean(), 4)
}])

faithfulness_table_df = pd.concat([overall_row, faithfulness_table_df], ignore_index=True)

display(faithfulness_table_df)

           Concept  Faithfulness
0   Overall_MedSAM        0.7110
1       lesion_011        0.9777
2       lesion_010        0.9740
3       vessel_001        0.8757
4       lesion_009        0.8405
5       vessel_008        0.8792
6       vessel_006        0.8944
7       vessel_005        0.8879
8       vessel_002        0.7726
9       vessel_007        0.7525
10      vessel_000        0.3717
11      vessel_003        0.2614
12      vessel_004        0.0439

In [34]:
# =========================================
# Concept-level Faithfulness only
# Groups:
#   lesion, vessel, optic_disc
# No overall row
# =========================================

import pandas as pd
import numpy as np

path = "/kaggle/working/node_fidelity_complete_table.csv"
df = pd.read_csv(path)

required_cols = ["node_name", "R2"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["R2"] = pd.to_numeric(df["R2"], errors="coerce")
df = df.dropna(subset=["node_name", "R2"]).copy()

# Better concept extraction:
# lesion_011 -> lesion
# vessel_004 -> vessel
# optic_disc_000 -> optic_disc
def extract_concept(node_name):
    if str(node_name).startswith("optic_disc"):
        return "optic_disc"
    elif str(node_name).startswith("lesion"):
        return "lesion"
    elif str(node_name).startswith("vessel"):
        return "vessel"
    else:
        return None

df["Concept"] = df["node_name"].apply(extract_concept)
df = df[df["Concept"].notna()].copy()

concept_faithfulness_table = (
    df.groupby("Concept", as_index=False)
      .agg(
          Faithfulness_mean=("R2", "mean"),
          Faithfulness_std=("R2", "std"),
          N=("R2", "count")
      )
)

concept_faithfulness_table["Faithfulness_mean"] = concept_faithfulness_table["Faithfulness_mean"].round(4)
concept_faithfulness_table["Faithfulness_std"] = concept_faithfulness_table["Faithfulness_std"].fillna(0).round(4)

display(concept_faithfulness_table)

  Concept  Faithfulness_mean  Faithfulness_std  N
0  lesion             0.9307            0.0782  3
1  vessel             0.6377            0.3240  9

In [35]:
# =========================================================
# NEW CELL AT THE END
# Wasserstein-based distance + kernel weights
# This does NOT replace cosine results.
# It creates a NEW dataframe and NEW csv file.
#
# Expected existing variables from your pipeline:
#   - original_embedding
#   - perturbed_embeddings
#
# Outputs:
#   - distance_df_from_medsam_wasserstein
#   - /kaggle/working/distance_df_from_medsam_wasserstein.csv
# =========================================================

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

# ---------------------------------------------------------
# 1) Check required variables
# ---------------------------------------------------------
if "original_embedding" not in globals():
    raise ValueError("original_embedding not found. Run the original image embedding cell first.")

if "perturbed_embeddings" not in globals():
    raise ValueError("perturbed_embeddings not found. Run the perturbed embedding extraction cell first.")

# ---------------------------------------------------------
# 2) Prepare embeddings
# ---------------------------------------------------------
orig = np.asarray(original_embedding, dtype=float).reshape(-1)

pert_list = []
for i, emb in enumerate(perturbed_embeddings):
    arr = np.asarray(emb, dtype=float).reshape(-1)
    if arr.shape != orig.shape:
        raise ValueError(
            f"Shape mismatch at perturb_id={i}: original={orig.shape}, perturbed={arr.shape}"
        )
    pert_list.append(arr)

if len(pert_list) == 0:
    raise ValueError("perturbed_embeddings is empty.")

# ---------------------------------------------------------
# 3) Compute Wasserstein distance for each perturbation
# ---------------------------------------------------------
rows = []
for perturb_id, emb in enumerate(pert_list):
    wdist = wasserstein_distance(orig, emb)

    rows.append({
        "perturb_id": perturb_id,
        "wasserstein_distance_to_original": float(wdist)
    })

distance_df_from_medsam_wasserstein = pd.DataFrame(rows)

# ---------------------------------------------------------
# 4) Convert distance to similarity
#    Larger similarity = closer to original
# ---------------------------------------------------------
distance_df_from_medsam_wasserstein["wasserstein_similarity_to_original"] = (
    1.0 / (1.0 + distance_df_from_medsam_wasserstein["wasserstein_distance_to_original"])
)

# ---------------------------------------------------------
# 5) Convert distance to kernel weights
#    Same idea as cosine version, but now with Wasserstein
# ---------------------------------------------------------
wd = distance_df_from_medsam_wasserstein["wasserstein_distance_to_original"].values.astype(float)

if wd.max() > wd.min():
    wd_norm = (wd - wd.min()) / (wd.max() - wd.min())
else:
    wd_norm = np.zeros_like(wd)

sigma = 0.75
distance_df_from_medsam_wasserstein["kernel_weight_from_wassdist"] = np.exp(
    -(wd_norm ** 2) / (2 * sigma ** 2)
)

# ---------------------------------------------------------
# 6) Save as NEW file
# ---------------------------------------------------------
save_path = "/kaggle/working/distance_df_from_medsam_wasserstein.csv"
distance_df_from_medsam_wasserstein.to_csv(save_path, index=False)

print("✅ New Wasserstein distance table created")
print("Saved to:", save_path)
display(distance_df_from_medsam_wasserstein.head())

✅ New Wasserstein distance table created
Saved to: /kaggle/working/distance_df_from_medsam_wasserstein.csv


   perturb_id  wasserstein_distance_to_original  \
0           0                          0.010591   
1           1                          0.010591   
2           2                          0.008956   
3           3                          0.006509   
4           4                          0.007138   

   wasserstein_similarity_to_original  kernel_weight_from_wassdist  
0                            0.989520                     0.411112  
1                            0.989520                     0.411112  
2                            0.991123                     0.565009  
3                            0.993533                     0.797761  
4                            0.992913                     0.741157  

In [36]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

fidelity_results = []

X = np.array(perturb_vectors)

for node in node_shift_matrix.columns:

    y = node_shift_matrix[node].values

    if np.std(y) < 1e-8:
        r2 = 0.0
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )

        model = XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)

    fidelity_results.append({
        "node_name": node,
        "fidelity_R2": r2
    })

fidelity_df = pd.DataFrame(fidelity_results).sort_values(
    "fidelity_R2", ascending=False
)

print("\nReal Node Fidelity Results:")
display(fidelity_df)



Real Node Fidelity Results:


     node_name  fidelity_R2
0   lesion_009     0.989857
3   vessel_000     0.963779
4   vessel_001     0.950225
8   vessel_005     0.931491
5   vessel_002     0.929663
9   vessel_006     0.919357
1   lesion_010     0.915521
2   lesion_011     0.907794
11  vessel_008     0.906999
7   vessel_004     0.877455
10  vessel_007     0.820182
6   vessel_003     0.617157

In [37]:
# ============================================================
# NEW CELL AFTER YOUR CURRENT FIDELITY CELL
# Complete node-level fidelity table + variance
# Wasserstein-weighted version
# ============================================================

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# -----------------------------
# Safety checks
# -----------------------------
if "perturb_vectors" not in globals():
    raise ValueError("perturb_vectors is missing.")

if "node_shift_matrix" not in globals():
    raise ValueError("node_shift_matrix is missing.")

X_all = np.array(perturb_vectors, dtype=float)

if X_all.ndim != 2:
    raise ValueError("perturb_vectors must be a 2D array.")

if len(node_shift_matrix) == 0:
    raise ValueError("node_shift_matrix is empty.")


# -----------------------------
# Weight utilities
# -----------------------------
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    w_sum = np.sum(w)
    if len(x) == 0:
        return np.nan
    if w_sum <= 0:
        return np.mean(x)
    return np.sum(w * x) / w_sum

def weighted_mse(y_true, y_pred, w):
    return weighted_mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2, w)

def weighted_mae(y_true, y_pred, w):
    return weighted_mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), w)

def weighted_r2(y_true, y_pred, w):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(w, dtype=float)

    if len(y_true) < 2:
        return np.nan

    y_bar = weighted_mean(y_true, w)
    ss_res = np.sum(w * (y_true - y_pred) ** 2)
    ss_tot = np.sum(w * (y_true - y_bar) ** 2)

    if ss_tot <= 1e-12:
        return np.nan

    return 1.0 - (ss_res / ss_tot)

def kish_effective_n(w):
    w = np.asarray(w, dtype=float)
    denom = np.sum(w ** 2)
    if denom <= 0:
        return len(w)
    return (np.sum(w) ** 2) / denom

def weighted_adjusted_r2(y_true, y_pred, w, p):
    r2w = weighted_r2(y_true, y_pred, w)
    if pd.isna(r2w):
        return np.nan

    n_eff = kish_effective_n(w)

    if n_eff <= (p + 1):
        return np.nan

    return 1.0 - (1.0 - r2w) * ((n_eff - 1.0) / (n_eff - p - 1.0))


# -----------------------------
# Build sample weights
# -----------------------------
def get_weight_map(n_rows):
    # default: equal weights
    weight_map = pd.Series(np.ones(n_rows, dtype=float), index=np.arange(n_rows))

    # first try in-memory dataframe
    if "distance_df_from_medsam_wasserstein" in globals():
        df_w = distance_df_from_medsam_wasserstein.copy()
    elif "distance_df" in globals():
        df_w = distance_df.copy()
    else:
        # then try saved csv
        path = "/kaggle/working/distance_df_from_medsam_wasserstein.csv"
        if not os.path.exists(path):
            print("Wasserstein distance file not found -> using equal weights.")
            return weight_map, "equal_weights"
        df_w = pd.read_csv(path)

    if not isinstance(df_w, pd.DataFrame) or df_w.empty:
        print("Wasserstein distance dataframe invalid/empty -> using equal weights.")
        return weight_map, "equal_weights"

    if "perturb_id" not in df_w.columns:
        print("Wasserstein distance dataframe has no perturb_id -> using equal weights.")
        return weight_map, "equal_weights"

    if "kernel_weight_from_wassdist" in df_w.columns:
        wcol = "kernel_weight_from_wassdist"
    elif "kernel_weight" in df_w.columns:
        wcol = "kernel_weight"
    else:
        print("No usable Wasserstein weight column found -> using equal weights.")
        return weight_map, "equal_weights"

    tmp = df_w[["perturb_id", wcol]].copy()
    tmp[wcol] = pd.to_numeric(tmp[wcol], errors="coerce").fillna(1.0)
    tmp = tmp.drop_duplicates(subset=["perturb_id"]).set_index("perturb_id")[wcol]

    return tmp, wcol


# -----------------------------
# Main repeated-run fidelity
# -----------------------------
import os

N_REPEATS = 30
TEST_SIZE = 0.30

weight_map, weight_source = get_weight_map(len(X_all))

all_runs = []

for node in node_shift_matrix.columns:
    y = node_shift_matrix[node].values.astype(float)

    # remove NaN rows safely
    valid_mask = ~np.isnan(y)
    X = X_all[valid_mask]
    y = y[valid_mask]

    valid_ids = np.where(valid_mask)[0]
    w = weight_map.reindex(valid_ids).fillna(1.0).values.astype(float)

    if len(y) < 8:
        all_runs.append({
            "node_name": node,
            "repeat_id": np.nan,
            "status": "too_few_samples",
            "MSE": np.nan,
            "MAE": np.nan,
            "Lm": np.nan,
            "mean_L1": np.nan,
            "mean_L2": np.nan,
            "WMSE": np.nan,
            "WMAE": np.nan,
            "weighted_R2_wass": np.nan,
            "weighted_adjusted_R2_wass": np.nan,
            "R2": np.nan
        })
        continue

    if np.std(y) < 1e-8:
        for rep in range(N_REPEATS):
            all_runs.append({
                "node_name": node,
                "repeat_id": rep,
                "status": "no_variation",
                "MSE": 0.0,
                "MAE": 0.0,
                "Lm": 0.0,
                "mean_L1": 0.0,
                "mean_L2": 0.0,
                "WMSE": 0.0,
                "WMAE": 0.0,
                "weighted_R2_wass": np.nan,
                "weighted_adjusted_R2_wass": np.nan,
                "R2": np.nan
            })
        continue

    for rep in range(N_REPEATS):
        X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
            X, y, w,
            test_size=TEST_SIZE,
            random_state=42 + rep
        )

        model = XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42 + rep
        )

        model.fit(X_train, y_train, sample_weight=w_train)
        y_pred = model.predict(X_test)

        mse_val = mean_squared_error(y_test, y_pred)
        mae_val = mean_absolute_error(y_test, y_pred)
        mean_l1_val = np.mean(np.abs(y_test - y_pred))
        mean_l2_val = np.mean((y_test - y_pred) ** 2)

        # thesis-friendly combined loss
        lm_val = 0.5 * (mean_l1_val + mean_l2_val)

        wmse_val = weighted_mse(y_test, y_pred, w_test)
        wmae_val = weighted_mae(y_test, y_pred, w_test)
        wr2_val = weighted_r2(y_test, y_pred, w_test)
        wr2_adj_val = weighted_adjusted_r2(
            y_true=y_test,
            y_pred=y_pred,
            w=w_test,
            p=X_test.shape[1]
        )

        all_runs.append({
            "node_name": node,
            "repeat_id": rep,
            "status": "ok",
            "MSE": mse_val,
            "MAE": mae_val,
            "Lm": lm_val,
            "mean_L1": mean_l1_val,
            "mean_L2": mean_l2_val,
            "WMSE": wmse_val,
            "WMAE": wmae_val,
            "weighted_R2_wass": wr2_val,
            "weighted_adjusted_R2_wass": wr2_adj_val,
            "R2": r2_score(y_test, y_pred)
        })

node_fidelity_runs_df_wass = pd.DataFrame(all_runs)

# -----------------------------
# Mean table per node
# -----------------------------
node_fidelity_complete_table_wass = (
    node_fidelity_runs_df_wass
    .groupby("node_name", as_index=False)
    .agg({
        "MSE": "mean",
        "MAE": "mean",
        "Lm": "mean",
        "mean_L1": "mean",
        "mean_L2": "mean",
        "WMSE": "mean",
        "WMAE": "mean",
        "R2": "mean",
        "weighted_R2_wass": "mean",
        "weighted_adjusted_R2_wass": "mean"
    })
)

# -----------------------------
# Variance table per node
# -----------------------------
node_fidelity_variance_table_wass = (
    node_fidelity_runs_df_wass
    .groupby("node_name", as_index=False)
    .agg({
        "R2": ["var", "std"],
        "weighted_R2_wass": ["var", "std"]
    })
)

node_fidelity_variance_table_wass.columns = [
    "node_name",
    "var_R2", "std_R2",
    "var_weighted_R2_wass", "std_weighted_R2_wass"
]

# -----------------------------
# Final merged table
# -----------------------------
node_fidelity_complete_table_wass = node_fidelity_complete_table_wass.merge(
    node_fidelity_variance_table_wass,
    on="node_name",
    how="left"
).sort_values("weighted_R2_wass", ascending=False)

print("Complete node-level fidelity table (Wasserstein-weighted)")
print("Weight source:", weight_source)
display(node_fidelity_complete_table_wass.round(4))

# save as new files
node_fidelity_runs_df_wass.to_csv("/kaggle/working/node_fidelity_runs_df_wass.csv", index=False)
node_fidelity_complete_table_wass.to_csv("/kaggle/working/node_fidelity_complete_table_wass.csv", index=False)

print("Saved:")
print("/kaggle/working/node_fidelity_runs_df_wass.csv")
print("/kaggle/working/node_fidelity_complete_table_wass.csv")

Complete node-level fidelity table (Wasserstein-weighted)
Weight source: kernel_weight_from_wassdist


     node_name     MSE     MAE      Lm  mean_L1  mean_L2    WMSE    WMAE  \
2   lesion_011  0.0019  0.0294  0.0156   0.0294   0.0019  0.0021  0.0311   
1   lesion_010  0.0001  0.0059  0.0030   0.0059   0.0001  0.0001  0.0063   
0   lesion_009  0.0001  0.0048  0.0024   0.0048   0.0001  0.0001  0.0051   
11  vessel_008  0.0000  0.0029  0.0014   0.0029   0.0000  0.0000  0.0030   
4   vessel_001  0.0002  0.0098  0.0050   0.0098   0.0002  0.0002  0.0100   
8   vessel_005  0.0000  0.0030  0.0015   0.0030   0.0000  0.0000  0.0031   
9   vessel_006  0.0000  0.0030  0.0015   0.0030   0.0000  0.0000  0.0031   
10  vessel_007  0.0006  0.0168  0.0087   0.0168   0.0006  0.0007  0.0179   
5   vessel_002  0.0003  0.0108  0.0055   0.0108   0.0003  0.0003  0.0112   
3   vessel_000  0.0007  0.0162  0.0084   0.0162   0.0007  0.0007  0.0171   
6   vessel_003  0.0003  0.0094  0.0048   0.0094   0.0003  0.0003  0.0096   
7   vessel_004  0.0002  0.0075  0.0039   0.0075   0.0002  0.0002  0.0086   

        R2 

Saved:
/kaggle/working/node_fidelity_runs_df_wass.csv
/kaggle/working/node_fidelity_complete_table_wass.csv


In [38]:
# ============================================================
# NEW CELL: Thesis-style concept-level fidelity table
# Wasserstein-weighted version
# Groups node-level Wasserstein results into lesion / vessel / optic_disc
# ============================================================

import numpy as np
import pandas as pd

# -----------------------------
# Safety check
# -----------------------------
if "node_fidelity_complete_table_wass" not in globals():
    raise ValueError(
        "node_fidelity_complete_table_wass is missing. Run the Wasserstein node-level fidelity cell first."
    )

df = node_fidelity_complete_table_wass.copy()

if "node_name" not in df.columns:
    raise ValueError("node_fidelity_complete_table_wass must contain 'node_name' column.")

# -----------------------------
# Map each node to thesis concept
# -----------------------------
def map_concept(node_name):
    s = str(node_name).lower().strip().replace("-", "_").replace(" ", "_")
    if "lesion" in s:
        return "lesion"
    elif "vessel" in s or "blood_vessels" in s:
        return "vessel"
    elif "optic" in s and ("disc" in s or "disk" in s):
        return "optic_disc"
    else:
        return np.nan

df["concept"] = df["node_name"].apply(map_concept)
df = df.dropna(subset=["concept"]).copy()

if df.empty:
    raise ValueError("No lesion / vessel / optic_disc nodes were found in node_fidelity_complete_table_wass.")

# -----------------------------
# Aggregate by concept
# -----------------------------
concept_level_fidelity_table_wass = (
    df.groupby("concept", as_index=False)
      .agg({
          "MSE": "mean",
          "MAE": "mean",
          "Lm": "mean",
          "mean_L1": "mean",
          "mean_L2": "mean",
          "WMSE": "mean",
          "WMAE": "mean",
          "R2": "mean",
          "weighted_R2_wass": "mean",
          "weighted_adjusted_R2_wass": "mean",
          "var_R2": "mean",
          "std_R2": "mean",
          "var_weighted_R2_wass": "mean",
          "std_weighted_R2_wass": "mean"
      })
)

# -----------------------------
# Rename for thesis-friendly display
# -----------------------------
concept_level_fidelity_table_wass = concept_level_fidelity_table_wass.rename(columns={
    "concept": "Concept",
    "MSE": "MSE",
    "MAE": "MAE",
    "Lm": "Lm",
    "mean_L1": "mean-L1",
    "mean_L2": "mean-L2",
    "WMSE": "WMSE",
    "WMAE": "WMAE",
    "R2": "R2",
    "weighted_R2_wass": "R2ω_wass",
    "weighted_adjusted_R2_wass": "R̂2ω_wass",
    "var_R2": "Var(R2)",
    "std_R2": "Std(R2)",
    "var_weighted_R2_wass": "Var(R2ω_wass)",
    "std_weighted_R2_wass": "Std(R2ω_wass)"
})

# Optional ordering
order_map = {"lesion": 0, "vessel": 1, "optic_disc": 2}
concept_level_fidelity_table_wass["__order"] = concept_level_fidelity_table_wass["Concept"].map(order_map)
concept_level_fidelity_table_wass = concept_level_fidelity_table_wass.sort_values("__order").drop(columns="__order")

print("Thesis-style concept-level fidelity table (Wasserstein-weighted):")
display(concept_level_fidelity_table_wass.round(4))

# Save as new file
concept_level_fidelity_table_wass.to_csv("/kaggle/working/concept_level_fidelity_table_wass.csv", index=False)
print("Saved: /kaggle/working/concept_level_fidelity_table_wass.csv")

Thesis-style concept-level fidelity table (Wasserstein-weighted):


  Concept     MSE     MAE      Lm  mean-L1  mean-L2    WMSE    WMAE      R2  \
0  lesion  0.0007  0.0134  0.0070   0.0134   0.0007  0.0007  0.0142  0.9811   
1  vessel  0.0003  0.0088  0.0045   0.0088   0.0003  0.0003  0.0093  0.7830   

   R2ω_wass  R̂2ω_wass  Var(R2)  Std(R2)  Var(R2ω_wass)  Std(R2ω_wass)  
0    0.9796     0.9629   0.0003   0.0163         0.0003         0.0175  
1    0.7702     0.5822   0.0857   0.2021         0.1064         0.2225  

Saved: /kaggle/working/concept_level_fidelity_table_wass.csv


In [39]:
# ============================================================
# REBUILD + FILL node_att_reference_df from scratch
# ============================================================

import numpy as np
import pandas as pd

required_vars = ["node_shift_df", "original_nodes", "original_masks", "segments", "perturb_vectors"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Missing required variables: {missing_vars}")

required_cols = {"perturb_id", "node_name", "shift"}
missing_cols = required_cols - set(node_shift_df.columns)
if missing_cols:
    raise ValueError(f"node_shift_df is missing required columns: {missing_cols}")

# 1) rebuild reference table from node_shift_df
node_att_reference_df = (
    node_shift_df[["perturb_id", "node_name"]]
    .drop_duplicates()
    .sort_values(["node_name", "perturb_id"])
    .reset_index(drop=True)
)

segments_np = np.array(segments)
X_perturb = np.array(perturb_vectors, dtype=int)

if X_perturb.ndim != 2:
    raise ValueError("perturb_vectors must be 2D.")

# IMPORTANT: map actual segment labels -> perturb vector column indices
segment_labels = np.sort(np.unique(segments_np))
if X_perturb.shape[1] != len(segment_labels):
    raise ValueError(
        f"Mismatch: perturb_vectors has {X_perturb.shape[1]} columns but segments has {len(segment_labels)} unique labels."
    )

seglabel_to_col = {int(lbl): i for i, lbl in enumerate(segment_labels)}

if len(original_nodes) != len(original_masks):
    raise ValueError("original_nodes and original_masks must have same length.")

node_to_mask = {}
for n, m in zip(original_nodes, original_masks):
    node_to_mask[str(n)] = np.array(m).astype(bool)

missing_nodes = [n for n in node_att_reference_df["node_name"].unique() if str(n) not in node_to_mask]
if missing_nodes:
    raise ValueError(f"These node names are missing from original_nodes: {missing_nodes[:10]}")

AFFECTED_THRESHOLD = 0.10

y_true_list = []
affected_fraction_list = []

for _, row in node_att_reference_df.iterrows():
    pid = int(row["perturb_id"])
    node_name = str(row["node_name"])

    if pid < 0 or pid >= len(X_perturb):
        y_true_list.append(np.nan)
        affected_fraction_list.append(np.nan)
        continue

    keep_vector = X_perturb[pid]
    node_mask = node_to_mask[node_name]

    if node_mask.sum() == 0:
        y_true_list.append(0)
        affected_fraction_list.append(0.0)
        continue

    overlapping_seg_labels = np.unique(segments_np[node_mask])

    total_node_pixels = float(node_mask.sum())
    affected_fraction = 0.0

    for seg_label in overlapping_seg_labels:
        seg_col = seglabel_to_col[int(seg_label)]
        overlap_pixels = np.logical_and(node_mask, segments_np == seg_label).sum()
        frac = overlap_pixels / total_node_pixels

        # 1 = kept, 0 = removed
        if keep_vector[seg_col] == 0:
            affected_fraction += frac

    y_true = int(affected_fraction >= AFFECTED_THRESHOLD)

    y_true_list.append(y_true)
    affected_fraction_list.append(float(affected_fraction))

node_att_reference_df["affected_fraction"] = affected_fraction_list
node_att_reference_df["y_true"] = y_true_list

print("Rebuilt node_att_reference_df successfully.")
print("Rows:", len(node_att_reference_df))
print("NaN y_true count:", int(node_att_reference_df["y_true"].isna().sum()))

display(node_att_reference_df.head(20))

print("\nClass balance:")
display(
    node_att_reference_df["y_true"]
    .value_counts(dropna=False)
    .rename_axis("y_true")
    .reset_index(name="count")
)

print("\nPer-node class balance:")
display(
    node_att_reference_df.groupby("node_name")["y_true"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={"sum": "positives", "mean": "positive_rate"})
    .sort_values("positive_rate", ascending=False)
)

Rebuilt node_att_reference_df successfully.
Rows: 600
NaN y_true count: 0


    perturb_id   node_name  affected_fraction  y_true
0            0  lesion_009                1.0       1
1            1  lesion_009                1.0       1
2            2  lesion_009                1.0       1
3            3  lesion_009                0.0       0
4            4  lesion_009                1.0       1
5            5  lesion_009                1.0       1
6            6  lesion_009                1.0       1
7            7  lesion_009                1.0       1
8            8  lesion_009                1.0       1
9            9  lesion_009                0.0       0
10          10  lesion_009                1.0       1
11          11  lesion_009                0.0       0
12          12  lesion_009                0.0       0
13          13  lesion_009                0.0       0
14          14  lesion_009                0.0       0
15          15  lesion_009                0.0       0
16          16  lesion_009                0.0       0
17          17  lesion_009  


Class balance:


   y_true  count
0       1    431
1       0    169


Per-node class balance:


     node_name  count  positives  positive_rate
10  vessel_007     50         45           0.90
5   vessel_002     50         40           0.80
4   vessel_001     50         40           0.80
7   vessel_004     50         40           0.80
9   vessel_006     50         39           0.78
8   vessel_005     50         39           0.78
11  vessel_008     50         39           0.78
3   vessel_000     50         39           0.78
6   vessel_003     50         29           0.58
2   lesion_011     50         27           0.54
0   lesion_009     50         27           0.54
1   lesion_010     50         27           0.54

In [40]:
# ============================================================
# ATT metrics without overall one-row table
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score

if "node_shift_df" not in globals():
    raise ValueError("node_shift_df is missing.")

if "node_att_reference_df" not in globals():
    raise ValueError("node_att_reference_df is missing.")

if node_att_reference_df["y_true"].isna().any():
    raise ValueError("node_att_reference_df['y_true'] still contains NaN values.")

att_score_df = node_shift_df[["perturb_id", "node_name", "shift"]].copy()
att_score_df["att_score"] = att_score_df["shift"].abs()

att_eval_df = att_score_df.merge(
    node_att_reference_df[["perturb_id", "node_name", "y_true"]],
    on=["perturb_id", "node_name"],
    how="inner"
)

if att_eval_df.empty:
    raise ValueError("ATT evaluation dataframe is empty after merge.")

att_eval_df["y_true"] = att_eval_df["y_true"].astype(int)

def map_concept(node_name):
    s = str(node_name).lower().strip().replace("-", "_").replace(" ", "_")
    if "lesion" in s:
        return "lesion"
    elif "vessel" in s or "blood_vessels" in s:
        return "vessel"
    elif "optic" in s and ("disc" in s or "disk" in s):
        return "optic_disc"
    else:
        return np.nan

node_att_rows = []

for node_name, g in att_eval_df.groupby("node_name"):
    g = g.dropna(subset=["att_score", "y_true"]).copy()

    if len(g) < 2 or g["y_true"].nunique() < 2:
        node_att_rows.append({
            "node_name": node_name,
            "concept": map_concept(node_name),
            "ATT_ACC": np.nan,
            "ATT_F1": np.nan,
            "ATT_AUROC": np.nan,
            "best_threshold": np.nan,
            "n_samples": len(g),
            "status": "single_class_or_too_few_samples"
        })
        continue

    y_true = g["y_true"].values.astype(int)
    att_score = g["att_score"].values.astype(float)

    att_auc = roc_auc_score(y_true, att_score)

    fpr, tpr, thresholds = roc_curve(y_true, att_score)
    j_scores = tpr - fpr
    best_idx = int(np.argmax(j_scores))
    thr_best = thresholds[best_idx]

    y_pred = (att_score >= thr_best).astype(int)

    att_acc = accuracy_score(y_true, y_pred)
    att_f1 = f1_score(y_true, y_pred, zero_division=0)

    node_att_rows.append({
        "node_name": node_name,
        "concept": map_concept(node_name),
        "ATT_ACC": att_acc,
        "ATT_F1": att_f1,
        "ATT_AUROC": att_auc,
        "best_threshold": thr_best,
        "n_samples": len(g),
        "status": "ok"
    })

node_level_att_metrics = pd.DataFrame(node_att_rows).sort_values(
    "ATT_AUROC", ascending=False
)

print("Node-level ATT metrics:")
display(node_level_att_metrics.round(4))

concept_level_att_metrics = (
    node_level_att_metrics
    .dropna(subset=["concept"])
    .groupby("concept", as_index=False)
    .agg({
        "ATT_ACC": "mean",
        "ATT_F1": "mean",
        "ATT_AUROC": "mean"
    })
)

concept_level_att_metrics = concept_level_att_metrics.rename(columns={
    "concept": "Concept",
    "ATT_ACC": "ATT ACC",
    "ATT_F1": "ATT F1",
    "ATT_AUROC": "ATT AUROC"
})

order_map = {"lesion": 0, "vessel": 1, "optic_disc": 2}
concept_level_att_metrics["__order"] = concept_level_att_metrics["Concept"].map(order_map)
concept_level_att_metrics = concept_level_att_metrics.sort_values("__order").drop(columns="__order")

print("\nConcept-level ATT table:")
display(concept_level_att_metrics.round(4))

node_level_att_metrics.to_csv("/kaggle/working/node_level_att_metrics.csv", index=False)
concept_level_att_metrics.to_csv("/kaggle/working/concept_level_att_metrics.csv", index=False)

print("Saved:")
print("/kaggle/working/node_level_att_metrics.csv")
print("/kaggle/working/concept_level_att_metrics.csv")

Node-level ATT metrics:


     node_name concept  ATT_ACC  ATT_F1  ATT_AUROC  best_threshold  n_samples  \
5   vessel_002  vessel     1.00  1.0000     1.0000          0.1773         50   
6   vessel_003  vessel     0.84  0.8788     0.7947          0.1064         50   
3   vessel_000  vessel     0.52  0.5556     0.3846          0.0314         50   
10  vessel_007  vessel     0.36  0.4483     0.3689          0.1972         50   
7   vessel_004  vessel     0.82  0.8989     0.1000          0.0073         50   
9   vessel_006  vessel     0.22  0.0000     0.0839             inf         50   
8   vessel_005  vessel     0.22  0.0000     0.0839             inf         50   
11  vessel_008  vessel     0.22  0.0000     0.0396             inf         50   
2   lesion_011  lesion     0.46  0.0000     0.0000             inf         50   
1   lesion_010  lesion     0.46  0.0000     0.0000             inf         50   
0   lesion_009  lesion     0.46  0.0000     0.0000             inf         50   
4   vessel_001  vessel     0


Concept-level ATT table:


  Concept  ATT ACC  ATT F1  ATT AUROC
0  lesion   0.4600  0.0000     0.0000
1  vessel   0.4889  0.4202     0.3173

Saved:
/kaggle/working/node_level_att_metrics.csv
/kaggle/working/concept_level_att_metrics.csv


In [41]:
# =========================================
# Node-level Faithfulness Table (Wasserstein)
# Faithfulness = R2
# =========================================

import pandas as pd
import numpy as np

fidelity_path = "/kaggle/working/node_fidelity_complete_table_wass.csv"

df = pd.read_csv(fidelity_path)

required_cols = ["node_name", "R2"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in node_fidelity_complete_table_wass.csv: {missing}")

df["R2"] = pd.to_numeric(df["R2"], errors="coerce")
df = df.dropna(subset=["node_name", "R2"]).copy()

# Clean table
node_faithfulness_table = df[["node_name", "R2"]].copy()
node_faithfulness_table = node_faithfulness_table.rename(columns={
    "node_name": "Node",
    "R2": "Faithfulness"
})

node_faithfulness_table["Faithfulness"] = node_faithfulness_table["Faithfulness"].round(4)

# Sort (important for readability)
node_faithfulness_table = node_faithfulness_table.sort_values(
    by="Faithfulness", ascending=False
).reset_index(drop=True)

display(node_faithfulness_table)

          Node  Faithfulness
0   lesion_011        0.9823
1   lesion_010        0.9817
2   lesion_009        0.9793
3   vessel_008        0.9451
4   vessel_005        0.9400
5   vessel_006        0.9400
6   vessel_001        0.9398
7   vessel_007        0.8502
8   vessel_002        0.8202
9   vessel_000        0.6851
10  vessel_003        0.5985
11  vessel_004        0.3285

In [42]:
# =========================================
# Concept-level Faithfulness (Wasserstein)
# =========================================

import pandas as pd
import numpy as np

path = "/kaggle/working/node_fidelity_complete_table_wass.csv"
df = pd.read_csv(path)

required_cols = ["node_name", "R2"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["R2"] = pd.to_numeric(df["R2"], errors="coerce")
df = df.dropna(subset=["node_name", "R2"]).copy()

def extract_concept(node_name):
    if str(node_name).startswith("optic_disc"):
        return "optic_disc"
    elif str(node_name).startswith("lesion"):
        return "lesion"
    elif str(node_name).startswith("vessel"):
        return "vessel"
    else:
        return None

df["Concept"] = df["node_name"].apply(extract_concept)
df = df[df["Concept"].notna()].copy()

concept_faithfulness_table = (
    df.groupby("Concept", as_index=False)
      .agg(
          Faithfulness_mean=("R2", "mean"),
          Faithfulness_std=("R2", "std"),
          N=("R2", "count")
      )
)

concept_faithfulness_table["Faithfulness_mean"] = concept_faithfulness_table["Faithfulness_mean"].round(4)
concept_faithfulness_table["Faithfulness_std"] = concept_faithfulness_table["Faithfulness_std"].fillna(0).round(4)

display(concept_faithfulness_table)

  Concept  Faithfulness_mean  Faithfulness_std  N
0  lesion             0.9811            0.0016  3
1  vessel             0.7830            0.2110  9

In [43]:
# =========================================
# MedSAM Faithfulness (Correlation-based: Pearson)
# Supervisor-style: correlation + p-value
# Uses:
#   explanation score = |shift|
#   perturbation strength = affected_fraction
# =========================================

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# -----------------------------
# Checks
# -----------------------------
if "node_shift_df" not in globals():
    raise ValueError("node_shift_df not found. Run Cell 23 first.")

if "node_att_reference_df" not in globals():
    raise ValueError("node_att_reference_df not found. Run Cell 30 first.")

required_shift_cols = {"perturb_id", "node_name", "shift"}
missing_shift = required_shift_cols - set(node_shift_df.columns)
if missing_shift:
    raise ValueError(f"node_shift_df missing columns: {missing_shift}")

required_ref_cols = {"perturb_id", "node_name", "affected_fraction"}
missing_ref = required_ref_cols - set(node_att_reference_df.columns)
if missing_ref:
    raise ValueError(f"node_att_reference_df missing columns: {missing_ref}")

# -----------------------------
# Build faithfulness dataframe
# -----------------------------
faith_df = node_shift_df[["perturb_id", "node_name", "shift"]].copy()
faith_df["att_score"] = faith_df["shift"].abs()

faith_df = faith_df.merge(
    node_att_reference_df[["perturb_id", "node_name", "affected_fraction"]],
    on=["perturb_id", "node_name"],
    how="inner"
)

faith_df = faith_df.dropna(subset=["att_score", "affected_fraction"]).copy()

if faith_df.empty:
    raise ValueError("faith_df is empty after merge.")

# -----------------------------
# Concept mapper
# -----------------------------
def map_concept(node_name):
    s = str(node_name).lower().strip().replace("-", "_").replace(" ", "_")
    if "lesion" in s:
        return "lesion"
    elif "vessel" in s or "blood_vessels" in s:
        return "vessel"
    elif "optic" in s and ("disc" in s or "disk" in s):
        return "optic_disc"
    else:
        return np.nan

faith_df["Concept"] = faith_df["node_name"].apply(map_concept)

# -----------------------------
# Node-level faithfulness
# -----------------------------
node_rows = []

for node_name, g in faith_df.groupby("node_name"):
    g = g.dropna(subset=["att_score", "affected_fraction"]).copy()

    if len(g) < 3:
        node_rows.append({
            "node_name": node_name,
            "Concept": map_concept(node_name),
            "Faithfulness_correlation": np.nan,
            "Faithfulness_pvalue": np.nan,
            "N": len(g),
            "status": "too_few_samples"
        })
        continue

    if g["att_score"].nunique() < 2 or g["affected_fraction"].nunique() < 2:
        node_rows.append({
            "node_name": node_name,
            "Concept": map_concept(node_name),
            "Faithfulness_correlation": np.nan,
            "Faithfulness_pvalue": np.nan,
            "N": len(g),
            "status": "no_variation"
        })
        continue

    rho, pval = pearsonr(g["affected_fraction"], g["att_score"])

    node_rows.append({
        "node_name": node_name,
        "Concept": map_concept(node_name),
        "Faithfulness_correlation": float(rho),
        "Faithfulness_pvalue": float(pval),
        "N": len(g),
        "status": "ok"
    })

node_faithfulness_table = pd.DataFrame(node_rows).sort_values(
    by="Faithfulness_correlation", ascending=False, na_position="last"
).reset_index(drop=True)

print("✅ Node-level MedSAM Faithfulness:")
display(node_faithfulness_table.round(4))

# -----------------------------
# Concept-level faithfulness
# -----------------------------
concept_rows = []

for concept, g in node_faithfulness_table.dropna(subset=["Concept"]).groupby("Concept"):
    valid = g.dropna(subset=["Faithfulness_correlation", "Faithfulness_pvalue"]).copy()

    if valid.empty:
        concept_rows.append({
            "Concept": concept,
            "Faithfulness_mean_correlation": np.nan,
            "Faithfulness_std_correlation": np.nan,
            "Faithfulness_mean_pvalue": np.nan,
            "N_nodes": 0
        })
        continue

    concept_rows.append({
        "Concept": concept,
        "Faithfulness_mean_correlation": valid["Faithfulness_correlation"].mean(),
        "Faithfulness_std_correlation": valid["Faithfulness_correlation"].std(ddof=1) if len(valid) > 1 else 0.0,
        "Faithfulness_mean_pvalue": valid["Faithfulness_pvalue"].mean(),
        "N_nodes": len(valid)
    })

concept_faithfulness_table = pd.DataFrame(concept_rows)

order_map = {"lesion": 0, "vessel": 1, "optic_disc": 2}
concept_faithfulness_table["__order"] = concept_faithfulness_table["Concept"].map(order_map)
concept_faithfulness_table = concept_faithfulness_table.sort_values("__order").drop(columns="__order")

print("\n✅ Concept-level MedSAM Faithfulness:")
display(concept_faithfulness_table.round(4))

# -----------------------------
# Optional overall faithfulness
# -----------------------------
if faith_df["att_score"].nunique() >= 2 and faith_df["affected_fraction"].nunique() >= 2:
    overall_rho, overall_p = pearsonr(faith_df["affected_fraction"], faith_df["att_score"])
    overall_faithfulness = pd.DataFrame([{
        "Overall_Faithfulness_correlation": float(overall_rho),
        "Overall_Faithfulness_pvalue": float(overall_p),
        "N_total": len(faith_df)
    }])

    print("\n✅ Overall MedSAM Faithfulness:")
    display(overall_faithfulness.round(4))
else:
    overall_faithfulness = pd.DataFrame()

# -----------------------------
# Save
# -----------------------------
node_faithfulness_table.to_csv("/kaggle/working/node_faithfulness_correlation_medsam.csv", index=False)
concept_faithfulness_table.to_csv("/kaggle/working/concept_faithfulness_correlation_medsam.csv", index=False)

if not overall_faithfulness.empty:
    overall_faithfulness.to_csv("/kaggle/working/overall_faithfulness_correlation_medsam.csv", index=False)

print("\nSaved:")
print("/kaggle/working/node_faithfulness_correlation_medsam.csv")
print("/kaggle/working/concept_faithfulness_correlation_medsam.csv")
if not overall_faithfulness.empty:
    print("/kaggle/working/overall_faithfulness_correlation_medsam.csv")

✅ Node-level MedSAM Faithfulness:


     node_name Concept  Faithfulness_correlation  Faithfulness_pvalue   N  \
0   vessel_002  vessel                    0.5415               0.0000  50   
1   vessel_003  vessel                    0.4327               0.0017  50   
2   vessel_007  vessel                    0.2804               0.0486  50   
3   vessel_000  vessel                    0.0177               0.9027  50   
4   vessel_004  vessel                   -0.3689               0.0084  50   
5   vessel_006  vessel                   -0.3863               0.0056  50   
6   vessel_005  vessel                   -0.4014               0.0039  50   
7   vessel_008  vessel                   -0.4281               0.0019  50   
8   vessel_001  vessel                   -0.7948               0.0000  50   
9   lesion_009  lesion                   -0.8708               0.0000  50   
10  lesion_010  lesion                   -0.9876               0.0000  50   
11  lesion_011  lesion                   -1.0000               0.0000  50   


✅ Concept-level MedSAM Faithfulness:


  Concept  Faithfulness_mean_correlation  Faithfulness_std_correlation  \
0  lesion                        -0.9528                        0.0713   
1  vessel                        -0.1230                        0.4589   

   Faithfulness_mean_pvalue  N_nodes  
0                    0.0000        3  
1                    0.1081        9  


✅ Overall MedSAM Faithfulness:


   Overall_Faithfulness_correlation  Overall_Faithfulness_pvalue  N_total
0                            -0.325                          0.0      600


Saved:
/kaggle/working/node_faithfulness_correlation_medsam.csv
/kaggle/working/concept_faithfulness_correlation_medsam.csv
/kaggle/working/overall_faithfulness_correlation_medsam.csv


In [44]:

# ============================================================
# COMPLETE SINGLE CELL: ConceptSMILE MedSAM robustness
# Paste at the END of the MedSAM notebook.
#
# Required existing variables:
# img_rgb_resized, predictor, sam, DEVICE,
# original_nodes, original_bboxes, segments, perturb_vectors
#
# Experiments:
# 1. Contrast variation
# 2. Simulated blink-related occlusion
#
# Outputs:
# - Preview image
# - Node-level repeated XGBoost results
# - Concept-level repeated results
# - Summary CSV
# - Publication-quality PNG, PDF and SVG
# ============================================================

import gc
import hashlib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ============================================================
# 1. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "img_rgb_resized",
    "predictor",
    "sam",
    "DEVICE",
    "original_nodes",
    "original_bboxes",
    "segments",
    "perturb_vectors"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the earlier MedSAM cells first. "
        "Missing objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. EXPERIMENT SETTINGS
# ============================================================

# True:
#   20 perturbations and 3 XGBoost repetitions
#   Suitable for checking whether the cell runs correctly.
#
# False:
#   50 perturbations and 30 repetitions
#   Recommended for the final paper experiment.
FAST_TEST = False


CONTRAST_LEVELS = [
    0.60,
    0.80,
    1.00,
    1.20,
    1.40
]


OCCLUSION_LEVELS = [
    0,
    1,
    2,
    3
]


# Horizontal location of the vertical occlusion:
# 0.0 = left
# 0.5 = centre
# 1.0 = right
OCCLUSION_X_RATIO = 0.50


# 1.0 means fully black.
OCCLUSION_DARKNESS = 1.0


if FAST_TEST:

    N_PERTURBATIONS = min(
        20,
        len(perturb_vectors)
    )

    N_XGB_REPEATS = 3

else:

    N_PERTURBATIONS = min(
        50,
        len(perturb_vectors)
    )

    N_XGB_REPEATS = 30


TEST_SIZE = 0.30

RANDOM_SEED = 42


OUTPUT_DIRECTORY = Path(
    "/kaggle/working/"
    "conceptsmile_medsam_robustness"
)


OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. FIXED ORIGINAL DATA
# ============================================================

BASE_IMAGE = np.asarray(
    img_rgb_resized,
    dtype=np.uint8
).copy()


FIXED_SEGMENTS = np.asarray(
    segments
)


SEGMENT_LABELS = np.sort(
    np.unique(
        FIXED_SEGMENTS
    )
)


FIXED_PERTURBATION_VECTORS = np.asarray(
    perturb_vectors[
        :N_PERTURBATIONS
    ],
    dtype=int
)


# Keep only valid nodes and bounding boxes.
valid_node_pairs = []


for node_name, bounding_box in zip(
    original_nodes,
    original_bboxes
):

    if bounding_box is None:
        continue

    if len(bounding_box) != 4:
        continue

    x0, y0, x1, y1 = map(
        int,
        bounding_box
    )

    if x1 <= x0 or y1 <= y0:
        continue

    valid_node_pairs.append(
        (
            str(node_name),
            (
                x0,
                y0,
                x1,
                y1
            )
        )
    )


if not valid_node_pairs:
    raise ValueError(
        "No valid original MedSAM nodes and "
        "bounding boxes were found."
    )


FIXED_NODE_NAMES = [
    pair[0]
    for pair in valid_node_pairs
]


FIXED_NODE_BBOXES = [
    pair[1]
    for pair in valid_node_pairs
]


if (
    BASE_IMAGE.ndim != 3
    or BASE_IMAGE.shape[2] != 3
):
    raise ValueError(
        "img_rgb_resized must have shape H x W x 3."
    )


if (
    FIXED_SEGMENTS.shape
    != BASE_IMAGE.shape[:2]
):
    raise ValueError(
        "segments and img_rgb_resized must have "
        "the same height and width."
    )


if FIXED_PERTURBATION_VECTORS.ndim != 2:
    raise ValueError(
        "perturb_vectors must be a two-dimensional array."
    )


if (
    FIXED_PERTURBATION_VECTORS.shape[1]
    != len(SEGMENT_LABELS)
):
    raise ValueError(
        "The number of columns in perturb_vectors "
        "does not match the number of superpixels."
    )


print(
    "MedSAM robustness setup"
)

print(
    "Image shape:",
    BASE_IMAGE.shape
)

print(
    "Valid original nodes:",
    len(FIXED_NODE_NAMES)
)

print(
    "Number of superpixels:",
    len(SEGMENT_LABELS)
)

print(
    "Perturbations per condition:",
    len(FIXED_PERTURBATION_VECTORS)
)

print(
    "XGBoost repeated splits:",
    N_XGB_REPEATS
)

print(
    "FAST_TEST:",
    FAST_TEST
)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

def set_all_random_seeds(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_all_random_seeds(
    RANDOM_SEED
)


# ============================================================
# 5. RETINAL FIELD-OF-VIEW MASK
# ============================================================

def create_retinal_fov_mask(
    image_rgb,
    threshold=10
):
    """
    Estimate the visible retinal field while excluding
    most of the surrounding black background.
    """

    grey_image = (
        image_rgb
        .astype(np.float32)
        .mean(axis=2)
    )


    retinal_mask = (
        grey_image > threshold
    )


    # Fallback for images without a black background.
    if retinal_mask.sum() < (
        0.05
        *
        retinal_mask.size
    ):

        retinal_mask = np.ones(
            grey_image.shape,
            dtype=bool
        )


    return retinal_mask


# ============================================================
# 6. CONTRAST MODIFICATION
# ============================================================

def modify_retinal_contrast(
    image_rgb,
    contrast_factor
):
    """
    Apply mean-preserving contrast modification inside
    the retinal field of view.

    1.0:
        Original image.

    Below 1.0:
        Reduced contrast.

    Above 1.0:
        Increased contrast.
    """

    if contrast_factor <= 0:

        raise ValueError(
            "contrast_factor must be greater than zero."
        )


    image_float = image_rgb.astype(
        np.float32
    )


    retinal_mask = create_retinal_fov_mask(
        image_rgb
    )


    mean_rgb = image_float[
        retinal_mask
    ].mean(
        axis=0
    )


    modified_image = image_float.copy()


    modified_image[
        retinal_mask
    ] = (
        mean_rgb
        +
        contrast_factor
        *
        (
            image_float[
                retinal_mask
            ]
            -
            mean_rgb
        )
    )


    return np.clip(
        modified_image,
        0,
        255
    ).astype(
        np.uint8
    )


# ============================================================
# 7. SIMULATED BLINK-RELATED OCCLUSION
# ============================================================

def add_simulated_occlusion(
    image_rgb,
    occlusion_percentage,
    x_ratio=0.50,
    darkness=1.0
):
    """
    Add a controlled vertical occlusion within the retinal
    field.

    This is described conservatively as a simulated
    blink-related acquisition occlusion rather than an exact
    reconstruction of eyelid geometry.

    occlusion_percentage:
        1 means 1% of retinal width.
        2 means 2%.
        3 means 3%.
    """

    if not 0 <= occlusion_percentage <= 100:

        raise ValueError(
            "occlusion_percentage must be between 0 and 100."
        )


    if not 0 <= x_ratio <= 1:

        raise ValueError(
            "x_ratio must be between 0 and 1."
        )


    if not 0 <= darkness <= 1:

        raise ValueError(
            "darkness must be between 0 and 1."
        )


    if occlusion_percentage == 0:

        return image_rgb.copy()


    retinal_mask = create_retinal_fov_mask(
        image_rgb
    )


    _, x_coordinates = np.where(
        retinal_mask
    )


    x_minimum = int(
        x_coordinates.min()
    )


    x_maximum = int(
        x_coordinates.max()
    )


    retinal_width = (
        x_maximum
        -
        x_minimum
        +
        1
    )


    line_width = max(
        2,
        int(
            round(
                retinal_width
                *
                occlusion_percentage
                /
                100.0
            )
        )
    )


    line_centre = int(
        round(
            x_minimum
            +
            x_ratio
            *
            (
                retinal_width - 1
            )
        )
    )


    line_start = max(
        0,
        line_centre
        -
        line_width // 2
    )


    line_end = min(
        image_rgb.shape[1],
        line_start
        +
        line_width
    )


    modified_image = (
        image_rgb
        .astype(np.float32)
        .copy()
    )


    local_mask = retinal_mask[
        :,
        line_start:line_end
    ]


    local_region = modified_image[
        :,
        line_start:line_end
    ]


    darkened_region = (
        local_region
        *
        (
            1.0
            -
            darkness
        )
    )


    modified_image[
        :,
        line_start:line_end
    ] = np.where(
        local_mask[..., None],
        darkened_region,
        local_region
    )


    return np.clip(
        modified_image,
        0,
        255
    ).astype(
        np.uint8
    )


# ============================================================
# 8. APPLY FIXED SUPERPIXEL PERTURBATION
# ============================================================

def apply_fixed_superpixel_perturbation(
    condition_image,
    keep_vector
):
    """
    Apply the same original superpixel layout and perturbation
    vector to every acquisition condition.
    """

    perturbed_image = (
        condition_image.copy()
    )


    for column_index, segment_label in enumerate(
        SEGMENT_LABELS
    ):

        if int(
            keep_vector[
                column_index
            ]
        ) == 0:

            perturbed_image[
                FIXED_SEGMENTS
                ==
                segment_label
            ] = 0


    return perturbed_image.astype(
        np.uint8
    )


# ============================================================
# 9. POOL MEDSAM IMAGE EMBEDDING
# ============================================================

def pool_medsam_embedding(
    embedding
):
    """
    Convert the MedSAM image embedding into one
    one-dimensional representation vector.
    """

    if torch.is_tensor(
        embedding
    ):

        embedding_array = (
            embedding
            .detach()
            .cpu()
            .numpy()
        )

    else:

        embedding_array = np.asarray(
            embedding
        )


    embedding_array = np.asarray(
        embedding_array,
        dtype=np.float32
    )


    if embedding_array.ndim == 4:

        pooled_embedding = (
            embedding_array.mean(
                axis=(
                    0,
                    2,
                    3
                )
            )
        )

    elif embedding_array.ndim == 3:

        pooled_embedding = (
            embedding_array.mean(
                axis=(
                    1,
                    2
                )
            )
        )

    elif embedding_array.ndim == 2:

        pooled_embedding = (
            embedding_array.mean(
                axis=0
            )
        )

    elif embedding_array.ndim == 1:

        pooled_embedding = (
            embedding_array
        )

    else:

        raise ValueError(
            "Unexpected MedSAM embedding shape: "
            f"{embedding_array.shape}"
        )


    return pooled_embedding.astype(
        np.float32
    )


# ============================================================
# 10. SCORE FIXED NODE BOXES WITH MEDSAM
# ============================================================

def score_fixed_boxes_and_embedding(
    image_rgb
):
    """
    Encode an image once and calculate each original node's
    confidence using its fixed bounding box.

    Confidence is calculated from:
        sigmoid(predicted masks)

    rather than from raw logits.
    """

    image_rgb = np.asarray(
        image_rgb,
        dtype=np.uint8
    )


    predictor.set_image(
        image_rgb
    )


    with torch.no_grad():

        image_embedding = (
            predictor.get_image_embedding()
        )


    pooled_embedding = (
        pool_medsam_embedding(
            image_embedding
        )
    )


    image_height, image_width = (
        image_rgb.shape[:2]
    )


    dense_positional_embedding = (
        sam.prompt_encoder
        .get_dense_pe()
    )


    confidence_values = []


    for bounding_box in FIXED_NODE_BBOXES:

        x0, y0, x1, y1 = bounding_box


        x0 = int(
            np.clip(
                x0,
                0,
                image_width - 1
            )
        )

        x1 = int(
            np.clip(
                x1,
                0,
                image_width - 1
            )
        )

        y0 = int(
            np.clip(
                y0,
                0,
                image_height - 1
            )
        )

        y1 = int(
            np.clip(
                y1,
                0,
                image_height - 1
            )
        )


        if x1 <= x0 or y1 <= y0:

            confidence_values.append(
                0.0
            )

            continue


        box_array = np.array(
            [
                [
                    x0,
                    y0,
                    x1,
                    y1
                ]
            ],
            dtype=np.float32
        )


        box_tensor = torch.as_tensor(
            box_array,
            device=DEVICE
        )


        with torch.no_grad():

            (
                sparse_embeddings,
                dense_embeddings
            ) = sam.prompt_encoder(
                points=None,
                boxes=box_tensor,
                masks=None
            )


            (
                low_resolution_masks,
                predicted_iou
            ) = sam.mask_decoder(
                image_embeddings=
                    image_embedding,

                image_pe=
                    dense_positional_embedding,

                sparse_prompt_embeddings=
                    sparse_embeddings,

                dense_prompt_embeddings=
                    dense_embeddings,

                multimask_output=True
            )


            probability_masks = torch.sigmoid(
                low_resolution_masks
            )


            probability_masks = F.interpolate(
                probability_masks,
                size=(
                    image_height,
                    image_width
                ),
                mode="bilinear",
                align_corners=False
            )


        probability_masks = (
            probability_masks
            .detach()
            .cpu()
            .numpy()[0]
        )


        predicted_iou_values = (
            predicted_iou
            .detach()
            .cpu()
            .numpy()[0]
        )


        best_mask_index = int(
            np.argmax(
                predicted_iou_values
            )
        )


        best_soft_mask = (
            probability_masks[
                best_mask_index
            ]
        )


        best_binary_mask = (
            best_soft_mask > 0.5
        )


        if best_binary_mask.sum() == 0:

            confidence = 0.0

        else:

            confidence = float(
                best_soft_mask[
                    best_binary_mask
                ].mean()
            )


        confidence_values.append(
            confidence
        )


    return (
        np.asarray(
            confidence_values,
            dtype=np.float32
        ),

        np.asarray(
            pooled_embedding,
            dtype=np.float32
        )
    )


# ============================================================
# 11. COSINE LOCALITY WEIGHTS
# ============================================================

def calculate_cosine_locality_weights(
    reference_embedding,
    perturbed_embeddings
):
    """
    Calculate cosine distance and the corresponding
    ConceptSMILE locality weights.
    """

    reference_embedding = np.asarray(
        reference_embedding,
        dtype=np.float32
    ).reshape(
        -1
    )


    perturbed_embeddings = np.asarray(
        perturbed_embeddings,
        dtype=np.float32
    )


    reference_norm = (
        np.linalg.norm(
            reference_embedding
        )
        +
        1e-12
    )


    perturbed_norms = (
        np.linalg.norm(
            perturbed_embeddings,
            axis=1
        )
        +
        1e-12
    )


    cosine_similarity = (
        perturbed_embeddings
        @
        reference_embedding
    ) / (
        perturbed_norms
        *
        reference_norm
    )


    cosine_similarity = np.clip(
        cosine_similarity,
        -1.0,
        1.0
    )


    cosine_distance = np.clip(
        1.0
        -
        cosine_similarity,
        0.0,
        2.0
    )


    positive_distances = (
        cosine_distance[
            cosine_distance > 0
        ]
    )


    if len(
        positive_distances
    ) > 0:

        sigma = float(
            np.median(
                positive_distances
            )
        )

    else:

        sigma = 0.25


    if (
        not np.isfinite(
            sigma
        )
        or sigma <= 1e-12
    ):

        sigma = 0.25


    locality_weights = np.exp(
        -(
            cosine_distance ** 2
        )
        /
        (
            sigma ** 2
        )
    )


    locality_weights = np.clip(
        locality_weights,
        1e-6,
        None
    )


    return (
        cosine_distance.astype(
            float
        ),

        locality_weights.astype(
            float
        ),

        sigma
    )


# ============================================================
# 12. MAP NODE NAMES TO RETINAL CONCEPTS
# ============================================================

def map_node_to_concept(
    node_name
):

    normalised_name = (
        str(node_name)
        .lower()
        .strip()
        .replace(
            "-",
            "_"
        )
        .replace(
            " ",
            "_"
        )
    )


    if "lesion" in normalised_name:

        return "lesion"


    if "vessel" in normalised_name:

        return "blood_vessels"


    if (
        "optic" in normalised_name
        and
        (
            "disc" in normalised_name
            or
            "disk" in normalised_name
        )
    ):

        return "optic_disc"


    return None


NODE_CONCEPTS = {
    node_name:
        map_node_to_concept(
            node_name
        )

    for node_name
    in FIXED_NODE_NAMES
}


unmatched_nodes = [
    node_name

    for node_name, concept
    in NODE_CONCEPTS.items()

    if concept is None
]


if unmatched_nodes:

    print(
        "Warning: these nodes will be excluded "
        "from concept-level plots:"
    )

    print(
        unmatched_nodes
    )


# ============================================================
# 13. EVALUATE ONE ACQUISITION CONDITION
# ============================================================

def evaluate_one_condition(
    condition_image,
    condition_name
):
    """
    For one contrast or occlusion condition:

    1. Score the unmasked image with MedSAM.
    2. Apply fixed superpixel perturbations.
    3. Recalculate MedSAM node confidence.
    4. Calculate confidence shifts.
    5. Calculate cosine locality weights.
    6. Train repeated XGBoost surrogates.
    7. Return node-level Test R² values.
    """

    print(
        "\n"
        +
        "=" * 70
    )

    print(
        "Running condition:",
        condition_name
    )

    print(
        "=" * 70
    )


    set_all_random_seeds(
        RANDOM_SEED
    )


    (
        condition_scores,
        condition_embedding
    ) = score_fixed_boxes_and_embedding(
        condition_image
    )


    perturbed_score_list = []

    perturbed_embedding_list = []


    for perturbation_index, keep_vector in enumerate(
        FIXED_PERTURBATION_VECTORS
    ):

        perturbed_image = (
            apply_fixed_superpixel_perturbation(
                condition_image,
                keep_vector
            )
        )


        (
            perturbed_scores,
            perturbed_embedding
        ) = score_fixed_boxes_and_embedding(
            perturbed_image
        )


        perturbed_score_list.append(
            perturbed_scores
        )


        perturbed_embedding_list.append(
            perturbed_embedding
        )


        if (
            perturbation_index == 0
            or
            (
                perturbation_index + 1
            ) % 5 == 0
        ):

            print(
                "Processed perturbations:",
                f"{perturbation_index + 1}/"
                f"{len(FIXED_PERTURBATION_VECTORS)}"
            )


        if (
            torch.cuda.is_available()
            and
            (
                perturbation_index + 1
            ) % 10 == 0
        ):

            torch.cuda.empty_cache()


    perturbed_score_matrix = np.vstack(
        perturbed_score_list
    ).astype(
        np.float32
    )


    perturbed_embedding_matrix = np.vstack(
        perturbed_embedding_list
    ).astype(
        np.float32
    )


    confidence_shift_matrix = (
        condition_scores[
            None,
            :
        ]
        -
        perturbed_score_matrix
    )


    (
        cosine_distances,
        locality_weights,
        sigma
    ) = calculate_cosine_locality_weights(
        condition_embedding,
        perturbed_embedding_matrix
    )


    X_all = (
        FIXED_PERTURBATION_VECTORS
        .astype(float)
    )


    node_result_rows = []


    for node_index, node_name in enumerate(
        FIXED_NODE_NAMES
    ):

        y_all = (
            confidence_shift_matrix[
                :,
                node_index
            ]
            .astype(float)
        )


        valid_mask = (
            np.isfinite(
                y_all
            )
            &
            np.isfinite(
                locality_weights
            )
        )


        X = X_all[
            valid_mask
        ]


        y = y_all[
            valid_mask
        ]


        weights = locality_weights[
            valid_mask
        ]


        concept_name = (
            NODE_CONCEPTS[
                node_name
            ]
        )


        common_results = {
            "condition":
                condition_name,

            "node_name":
                node_name,

            "concept":
                concept_name,

            "original_confidence":
                float(
                    condition_scores[
                        node_index
                    ]
                ),

            "mean_absolute_shift":
                float(
                    np.nanmean(
                        np.abs(
                            y
                        )
                    )
                )
                if len(y) > 0
                else np.nan,

            "mean_cosine_distance":
                float(
                    np.nanmean(
                        cosine_distances
                    )
                ),

            "sigma":
                float(
                    sigma
                ),

            "number_of_samples":
                int(
                    len(y)
                )
        }


        if (
            len(y) < 8
            or
            np.std(y) < 1e-8
        ):

            node_result_rows.append({
                **common_results,

                "repeat_id":
                    np.nan,

                "test_r2":
                    np.nan,

                "test_mae":
                    0.0
                    if len(y) > 0
                    else np.nan,

                "status":
                    "constant_or_too_few"
            })

            continue


        for repeat_id in range(
            N_XGB_REPEATS
        ):

            split_seed = (
                RANDOM_SEED
                +
                repeat_id
            )


            (
                X_train,
                X_test,
                y_train,
                y_test,
                weight_train,
                weight_test
            ) = train_test_split(
                X,
                y,
                weights,
                test_size=TEST_SIZE,
                random_state=split_seed
            )


            xgb_model = XGBRegressor(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                objective=
                    "reg:squarederror",
                random_state=
                    split_seed,
                n_jobs=-1
            )


            xgb_model.fit(
                X_train,
                y_train,
                sample_weight=
                    weight_train
            )


            predicted_shift = (
                xgb_model.predict(
                    X_test
                )
            )


            node_result_rows.append({
                **common_results,

                "repeat_id":
                    repeat_id,

                "test_r2":
                    float(
                        r2_score(
                            y_test,
                            predicted_shift
                        )
                    ),

                "test_mae":
                    float(
                        mean_absolute_error(
                            y_test,
                            predicted_shift
                        )
                    ),

                "status":
                    "ok"
            })


    del perturbed_score_matrix
    del perturbed_embedding_matrix
    del confidence_shift_matrix


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return pd.DataFrame(
        node_result_rows
    )


# ============================================================
# 14. CACHE IDENTICAL ORIGINAL CONDITION
# ============================================================

condition_cache = {}


def evaluate_condition_with_cache(
    condition_image,
    condition_name
):
    """
    Contrast 1.0 and occlusion 0% are identical images.
    The cache prevents running MedSAM twice on the same image.
    """

    image_hash = hashlib.sha1(
        condition_image.tobytes()
    ).hexdigest()


    if image_hash not in condition_cache:

        condition_cache[
            image_hash
        ] = evaluate_one_condition(
            condition_image,
            condition_name
        )


    result_copy = (
        condition_cache[
            image_hash
        ].copy()
    )


    result_copy[
        "condition"
    ] = condition_name


    return result_copy


# ============================================================
# 15. PREVIEW IMAGE CONDITIONS
# ============================================================

preview_conditions = [
    (
        "Original",
        BASE_IMAGE
    ),

    (
        "Contrast 0.60",
        modify_retinal_contrast(
            BASE_IMAGE,
            0.60
        )
    ),

    (
        "Contrast 1.40",
        modify_retinal_contrast(
            BASE_IMAGE,
            1.40
        )
    ),

    (
        "Occlusion 1%",
        add_simulated_occlusion(
            BASE_IMAGE,
            1,
            OCCLUSION_X_RATIO,
            OCCLUSION_DARKNESS
        )
    ),

    (
        "Occlusion 3%",
        add_simulated_occlusion(
            BASE_IMAGE,
            3,
            OCCLUSION_X_RATIO,
            OCCLUSION_DARKNESS
        )
    )
]


preview_figure, preview_axes = plt.subplots(
    1,
    5,
    figsize=(
        15.5,
        3.3
    )
)


for axis, (
    preview_title,
    preview_image
) in zip(
    preview_axes,
    preview_conditions
):

    axis.imshow(
        preview_image
    )

    axis.set_title(
        preview_title
    )

    axis.axis(
        "off"
    )


preview_figure.tight_layout()


preview_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_acquisition_perturbation_preview.png"
)


preview_figure.savefig(
    preview_path,
    dpi=400,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()


# ============================================================
# 16. RUN CONTRAST EXPERIMENT
# ============================================================

all_condition_results = []


for contrast_factor in CONTRAST_LEVELS:

    contrast_image = (
        modify_retinal_contrast(
            BASE_IMAGE,
            contrast_factor
        )
    )


    condition_name = (
        f"contrast_"
        f"{contrast_factor:.2f}"
    )


    condition_result = (
        evaluate_condition_with_cache(
            contrast_image,
            condition_name
        )
    )


    condition_result[
        "experiment"
    ] = "Contrast"


    condition_result[
        "severity"
    ] = float(
        contrast_factor
    )


    all_condition_results.append(
        condition_result
    )


# ============================================================
# 17. RUN OCCLUSION EXPERIMENT
# ============================================================

for occlusion_percentage in OCCLUSION_LEVELS:

    occlusion_image = (
        add_simulated_occlusion(
            BASE_IMAGE,
            occlusion_percentage,
            OCCLUSION_X_RATIO,
            OCCLUSION_DARKNESS
        )
    )


    condition_name = (
        f"occlusion_"
        f"{occlusion_percentage}"
        f"percent"
    )


    condition_result = (
        evaluate_condition_with_cache(
            occlusion_image,
            condition_name
        )
    )


    condition_result[
        "experiment"
    ] = "VerticalLine"


    condition_result[
        "severity"
    ] = float(
        occlusion_percentage
    )


    all_condition_results.append(
        condition_result
    )


# ============================================================
# 18. COMBINE NODE-LEVEL RESULTS
# ============================================================

medsam_node_fidelity_runs_df = pd.concat(
    all_condition_results,
    ignore_index=True
)


node_level_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_robustness_node_level_runs.csv"
)


medsam_node_fidelity_runs_df.to_csv(
    node_level_csv_path,
    index=False
)


# ============================================================
# 19. CREATE CONCEPT-LEVEL REPEATED RESULTS
# ============================================================

valid_node_runs_df = (
    medsam_node_fidelity_runs_df[
        (
            medsam_node_fidelity_runs_df[
                "status"
            ]
            ==
            "ok"
        )
        &
        medsam_node_fidelity_runs_df[
            "concept"
        ].notna()
        &
        medsam_node_fidelity_runs_df[
            "test_r2"
        ].notna()
    ]
    .copy()
)


if valid_node_runs_df.empty:

    raise ValueError(
        "No valid MedSAM robustness results were produced."
    )


# Average nodes first within each concept and repeated split.
# This avoids treating every vessel or lesion node as an
# independent repeated experiment.
medsam_robustness_fidelity_runs_df = (
    valid_node_runs_df
    .groupby(
        [
            "experiment",
            "severity",
            "condition",
            "concept",
            "repeat_id"
        ],
        as_index=False
    )
    .agg(
        test_r2=(
            "test_r2",
            "mean"
        ),

        test_mae=(
            "test_mae",
            "mean"
        ),

        mean_absolute_shift=(
            "mean_absolute_shift",
            "mean"
        ),

        mean_cosine_distance=(
            "mean_cosine_distance",
            "mean"
        ),

        number_of_nodes=(
            "node_name",
            "nunique"
        )
    )
)


concept_level_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_robustness_concept_level_runs.csv"
)


medsam_robustness_fidelity_runs_df.to_csv(
    concept_level_csv_path,
    index=False
)


# ============================================================
# 20. CREATE SUMMARY TABLE
# ============================================================

medsam_robustness_summary_df = (
    medsam_robustness_fidelity_runs_df
    .groupby(
        [
            "experiment",
            "severity",
            "concept"
        ],
        as_index=False
    )
    .agg(
        mean_test_r2=(
            "test_r2",
            "mean"
        ),

        median_test_r2=(
            "test_r2",
            "median"
        ),

        std_test_r2=(
            "test_r2",
            "std"
        ),

        mean_test_mae=(
            "test_mae",
            "mean"
        ),

        mean_absolute_shift=(
            "mean_absolute_shift",
            "mean"
        ),

        mean_cosine_distance=(
            "mean_cosine_distance",
            "mean"
        ),

        number_of_runs=(
            "test_r2",
            "count"
        ),

        mean_number_of_nodes=(
            "number_of_nodes",
            "mean"
        )
    )
)


medsam_robustness_summary_df[
    "std_test_r2"
] = (
    medsam_robustness_summary_df[
        "std_test_r2"
    ]
    .fillna(
        0.0
    )
)


medsam_robustness_summary_df[
    "lower_sd"
] = (
    medsam_robustness_summary_df[
        "mean_test_r2"
    ]
    -
    medsam_robustness_summary_df[
        "std_test_r2"
    ]
)


medsam_robustness_summary_df[
    "upper_sd"
] = (
    medsam_robustness_summary_df[
        "mean_test_r2"
    ]
    +
    medsam_robustness_summary_df[
        "std_test_r2"
    ]
)


summary_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_robustness_summary.csv"
)


medsam_robustness_summary_df.to_csv(
    summary_csv_path,
    index=False
)


print(
    "\nMedSAM robustness summary:"
)


display(
    medsam_robustness_summary_df.round(
        4
    )
)


# ============================================================
# 21. PUBLICATION-QUALITY STYLE
# ============================================================

plt.rcParams.update({
    "figure.dpi":
        180,

    "savefig.dpi":
        800,

    "font.family":
        "DejaVu Serif",

    "mathtext.fontset":
        "stix",

    "font.size":
        11,

    "axes.titlesize":
        14,

    "axes.labelsize":
        12,

    "xtick.labelsize":
        10,

    "ytick.labelsize":
        10,

    "legend.fontsize":
        10,

    "axes.linewidth":
        1.0,

    "lines.linewidth":
        2.4,

    "lines.markersize":
        6.5,

    "pdf.fonttype":
        42,

    "ps.fonttype":
        42
})


concept_order = [
    "blood_vessels",
    "lesion",
    "optic_disc"
]


concept_labels = {
    "blood_vessels":
        "Blood vessels",

    "lesion":
        "Lesion",

    "optic_disc":
        "Optic disc"
}


concept_colours = {
    "blood_vessels":
        "#2F73B7",

    "lesion":
        "#F28E2B",

    "optic_disc":
        "#3A9D3A"
}


experiment_order = [
    "Contrast",
    "VerticalLine"
]


experiment_titles = {
    "Contrast":
        "Contrast Variation",

    "VerticalLine":
        "Simulated Blink-Related "
        "Occlusion Artefact"
}


experiment_xlabels = {
    "Contrast":
        "Contrast factor "
        "(1.0 = original)",

    "VerticalLine":
        "Simulated occlusion "
        "(% of retinal width)"
}


plot_random_generator = np.random.default_rng(
    RANDOM_SEED
)


# ============================================================
# 22. DRAW FINAL 3 x 2 ROBUSTNESS FIGURE
# ============================================================

figure, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(
        14.2,
        9.8
    ),
    sharey="row",
    constrained_layout=False
)


figure.subplots_adjust(
    left=0.12,
    right=0.985,
    top=0.90,
    bottom=0.13,
    hspace=0.28,
    wspace=0.12
)


for row_index, concept in enumerate(
    concept_order
):

    concept_values = (
        medsam_robustness_fidelity_runs_df[
            medsam_robustness_fidelity_runs_df[
                "concept"
            ]
            ==
            concept
        ][
            "test_r2"
        ]
        .dropna()
        .to_numpy()
    )


    if len(
        concept_values
    ) > 0:

        y_minimum = float(
            np.min(
                concept_values
            )
        )

        y_maximum = float(
            np.max(
                concept_values
            )
        )

        y_range = (
            y_maximum
            -
            y_minimum
        )


        if y_range < 0.20:

            y_padding = 0.10

        else:

            y_padding = max(
                0.10,
                0.10
                *
                y_range
            )


        row_y_limits = (
            y_minimum
            -
            y_padding,

            y_maximum
            +
            y_padding
        )

    else:

        row_y_limits = (
            -0.1,
            1.0
        )


    for column_index, experiment in enumerate(
        experiment_order
    ):

        axis = axes[
            row_index,
            column_index
        ]


        panel_runs = (
            medsam_robustness_fidelity_runs_df[
                (
                    medsam_robustness_fidelity_runs_df[
                        "concept"
                    ]
                    ==
                    concept
                )
                &
                (
                    medsam_robustness_fidelity_runs_df[
                        "experiment"
                    ]
                    ==
                    experiment
                )
            ]
            .copy()
        )


        panel_summary = (
            medsam_robustness_summary_df[
                (
                    medsam_robustness_summary_df[
                        "concept"
                    ]
                    ==
                    concept
                )
                &
                (
                    medsam_robustness_summary_df[
                        "experiment"
                    ]
                    ==
                    experiment
                )
            ]
            .sort_values(
                "severity"
            )
            .copy()
        )


        severity_levels = sorted(
            panel_runs[
                "severity"
            ].unique()
        )


        # Individual repeated XGBoost values
        for severity in severity_levels:

            severity_values = (
                panel_runs[
                    panel_runs[
                        "severity"
                    ]
                    ==
                    severity
                ][
                    "test_r2"
                ]
                .dropna()
                .to_numpy()
            )


            jitter_scale = (
                0.007
                if experiment
                ==
                "Contrast"
                else
                0.025
            )


            jitter = (
                plot_random_generator
                .normal(
                    loc=0.0,
                    scale=
                        jitter_scale,
                    size=
                        len(
                            severity_values
                        )
                )
            )


            axis.scatter(
                severity
                +
                jitter,

                severity_values,

                s=50,
                alpha=0.55,

                color=
                    concept_colours[
                        concept
                    ],

                edgecolors=
                    "white",

                linewidths=
                    0.7,

                zorder=4
            )


        # Mean ± standard deviation
        axis.fill_between(
            panel_summary[
                "severity"
            ].to_numpy(),

            panel_summary[
                "lower_sd"
            ].to_numpy(),

            panel_summary[
                "upper_sd"
            ].to_numpy(),

            color=
                concept_colours[
                    concept
                ],

            alpha=0.16,
            linewidth=0,
            zorder=1
        )


        # Mean
        axis.plot(
            panel_summary[
                "severity"
            ],

            panel_summary[
                "mean_test_r2"
            ],

            marker="o",
            markersize=6.5,

            markeredgewidth=
                0.8,

            markeredgecolor=
                "white",

            linewidth=2.5,

            color=
                concept_colours[
                    concept
                ],

            zorder=6
        )


        # Median
        axis.scatter(
            panel_summary[
                "severity"
            ],

            panel_summary[
                "median_test_r2"
            ],

            marker="D",
            s=45,

            facecolors=
                "white",

            edgecolors=
                concept_colours[
                    concept
                ],

            linewidths=
                1.5,

            zorder=7
        )


        # R² = 0 reference line
        axis.axhline(
            0.0,
            color="#777777",
            linestyle="--",
            linewidth=1.0,
            alpha=0.70
        )


        # Original input condition
        original_condition = (
            1.0
            if experiment
            ==
            "Contrast"
            else
            0.0
        )


        axis.axvline(
            original_condition,
            color="#888888",
            linestyle=":",
            linewidth=1.2,
            alpha=0.80
        )


        axis.set_ylim(
            row_y_limits
        )


        axis.set_xticks(
            severity_levels
        )


        axis.set_xticklabels(
            [
                f"{severity:g}"

                for severity
                in severity_levels
            ]
        )


        axis.grid(
            alpha=0.18
        )


        axis.set_axisbelow(
            True
        )


        if row_index == 0:

            axis.set_title(
                experiment_titles[
                    experiment
                ],

                fontweight=
                    "semibold",

                pad=12
            )


        if column_index == 0:

            axis.text(
                -0.11,
                0.50,

                concept_labels[
                    concept
                ],

                transform=
                    axis.transAxes,

                rotation=90,

                va="center",
                ha="center",

                fontsize=12,

                fontweight=
                    "semibold"
            )

        else:

            axis.tick_params(
                labelleft=False
            )


        if row_index == 2:

            axis.set_xlabel(
                experiment_xlabels[
                    experiment
                ],

                labelpad=9
            )


# ============================================================
# 23. SHARED LABEL AND LEGEND
# ============================================================

figure.supylabel(
    "ConceptSMILE surrogate fidelity "
    "(Test $R^2$)",
    fontsize=13,
    x=0.035
)


legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markersize=7,
        markerfacecolor="#666666",
        markeredgecolor="white",
        alpha=0.65,
        label="Individual XGBoost runs"
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="-",
        linewidth=2.5,
        markersize=6,
        color="#444444",
        label="Mean"
    ),

    Line2D(
        [0],
        [0],
        marker="D",
        linestyle="none",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="#444444",
        markeredgewidth=1.4,
        label="Median"
    ),

    Patch(
        facecolor="#888888",
        alpha=0.18,
        edgecolor="none",
        label="Mean ± SD"
    )
]


figure.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(
        0.54,
        0.025
    ),
    ncol=4,
    frameon=False,
    handlelength=2.0,
    columnspacing=1.8
)


figure.suptitle(
    "MedSAM Concept-Level Robustness to Retinal "
    "Image Acquisition Variations",
    fontsize=16,
    fontweight="semibold",
    y=0.973
)


# ============================================================
# 24. SAVE FINAL FIGURE
# ============================================================

png_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_acquisition_robustness.png"
)


pdf_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_acquisition_robustness.pdf"
)


svg_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_acquisition_robustness.svg"
)


figure.savefig(
    png_path,
    dpi=800,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none",
    pil_kwargs={
        "compress_level":
            1
    }
)


figure.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none"
)


figure.savefig(
    svg_path,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none"
)


plt.show()


# ============================================================
# 25. FINAL OUTPUT PATHS
# ============================================================

print(
    "\nSaved MedSAM robustness files:"
)


for saved_path in [
    preview_path,
    node_level_csv_path,
    concept_level_csv_path,
    summary_csv_path,
    png_path,
    pdf_path,
    svg_path
]:

    print(
        saved_path
    )

MedSAM robustness setup
Image shape: (1024, 1024, 3)
Valid original nodes: 12
Number of superpixels: 6
Perturbations per condition: 50
XGBoost repeated splits: 30
FAST_TEST: False


<Figure size 1550x330 with 5 Axes>


Running condition: contrast_0.60
Processed perturbations: 1/50
Processed perturbations: 5/50
Processed perturbations: 10/50
Processed perturbations: 15/50
Processed perturbations: 20/50
Processed perturbations: 25/50
Processed perturbations: 30/50
Processed perturbations: 35/50
Processed perturbations: 40/50
Processed perturbations: 45/50
Processed perturbations: 50/50

Running condition: contrast_0.80
Processed perturbations: 1/50
Processed perturbations: 5/50
Processed perturbations: 10/50
Processed perturbations: 15/50
Processed perturbations: 20/50
Processed perturbations: 25/50
Processed perturbations: 30/50
Processed perturbations: 35/50
Processed perturbations: 40/50
Processed perturbations: 45/50
Processed perturbations: 50/50

Running condition: contrast_1.00
Processed perturbations: 1/50
Processed perturbations: 5/50
Processed perturbations: 10/50
Processed perturbations: 15/50
Processed perturbations: 20/50
Processed perturbations: 25/50
Processed perturbations: 30/50
Proce

      experiment  severity        concept  mean_test_r2  median_test_r2  \
0       Contrast       0.6  blood_vessels        0.6216          0.6726   
1       Contrast       0.6         lesion        0.9729          0.9783   
2       Contrast       0.8  blood_vessels        0.6444          0.6985   
3       Contrast       0.8         lesion        0.9091          0.9323   
4       Contrast       1.0  blood_vessels        0.6349          0.6961   
5       Contrast       1.0         lesion        0.9308          0.9473   
6       Contrast       1.2  blood_vessels        0.6151          0.6921   
7       Contrast       1.2         lesion        0.9135          0.9360   
8       Contrast       1.4  blood_vessels        0.5353          0.7248   
9       Contrast       1.4         lesion        0.9537          0.9613   
10  VerticalLine       0.0  blood_vessels        0.6349          0.6961   
11  VerticalLine       0.0         lesion        0.9308          0.9473   
12  VerticalLine       1.

<Figure size 2556x1764 with 6 Axes>


Saved MedSAM robustness files:
/kaggle/working/conceptsmile_medsam_robustness/medsam_acquisition_perturbation_preview.png
/kaggle/working/conceptsmile_medsam_robustness/medsam_robustness_node_level_runs.csv
/kaggle/working/conceptsmile_medsam_robustness/medsam_robustness_concept_level_runs.csv
/kaggle/working/conceptsmile_medsam_robustness/medsam_robustness_summary.csv
/kaggle/working/conceptsmile_medsam_robustness/medsam_acquisition_robustness.png
/kaggle/working/conceptsmile_medsam_robustness/medsam_acquisition_robustness.pdf
/kaggle/working/conceptsmile_medsam_robustness/medsam_acquisition_robustness.svg


In [45]:
# ============================================================
# COMPLETE CORRECTED SINGLE CELL: ConceptSMILE MedSAM Robustness
# Paste at the END of the MedSAM notebook.
#
# Required existing variables:
# img_rgb_resized, predictor, sam, DEVICE,
# original_nodes, original_bboxes, segments, perturb_vectors
# ============================================================

import gc
import hashlib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ============================================================
# 1. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "img_rgb_resized",
    "predictor",
    "sam",
    "DEVICE",
    "original_nodes",
    "original_bboxes",
    "segments",
    "perturb_vectors"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the earlier MedSAM cells first. "
        "Missing objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. EXPERIMENT SETTINGS
# ============================================================

# True:
#   20 perturbations and 3 XGBoost repetitions.
#   Use only to check that the cell runs successfully.
#
# False:
#   50 perturbations and 30 XGBoost repetitions.
#   Use for the final paper experiment.
FAST_TEST = False


CONTRAST_LEVELS = [
    0.60,
    0.80,
    1.00,
    1.20,
    1.40
]


OCCLUSION_LEVELS = [
    0,
    1,
    2,
    3
]


# Position of the simulated vertical acquisition artefact.
#
# 0.0 = left side
# 0.5 = centre
# 1.0 = right side
OCCLUSION_X_RATIO = 0.50


# 1.0 means completely black.
OCCLUSION_DARKNESS = 1.0


if FAST_TEST:

    N_PERTURBATIONS = min(
        20,
        len(perturb_vectors)
    )

    N_XGB_REPEATS = 3

else:

    N_PERTURBATIONS = min(
        50,
        len(perturb_vectors)
    )

    N_XGB_REPEATS = 30


TEST_SIZE = 0.30
RANDOM_SEED = 42


# Minimum requirements for reporting Test R².
#
# A nearly constant confidence-shift response makes R²
# numerically unstable and potentially extremely negative.
MIN_VALID_SAMPLES = 15
MIN_TARGET_STD = 1e-3
MIN_TEST_STD = 5e-4


# Prevent an excessively narrow locality kernel.
MIN_KERNEL_SIGMA = 1e-3
FALLBACK_KERNEL_SIGMA = 0.25


OUTPUT_DIRECTORY = Path(
    "/kaggle/working/"
    "conceptsmile_medsam_robustness_corrected"
)


OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. FIXED ORIGINAL DATA
# ============================================================

BASE_IMAGE = np.asarray(
    img_rgb_resized,
    dtype=np.uint8
).copy()


FIXED_SEGMENTS = np.asarray(
    segments
)


SEGMENT_LABELS = np.sort(
    np.unique(
        FIXED_SEGMENTS
    )
)


FIXED_PERTURBATION_VECTORS = np.asarray(
    perturb_vectors[
        :N_PERTURBATIONS
    ],
    dtype=int
)


if (
    BASE_IMAGE.ndim != 3
    or BASE_IMAGE.shape[2] != 3
):
    raise ValueError(
        "img_rgb_resized must have shape H x W x 3."
    )


if (
    FIXED_SEGMENTS.shape
    != BASE_IMAGE.shape[:2]
):
    raise ValueError(
        "segments and img_rgb_resized must have "
        "identical height and width."
    )


if FIXED_PERTURBATION_VECTORS.ndim != 2:
    raise ValueError(
        "perturb_vectors must be a two-dimensional array."
    )


if (
    FIXED_PERTURBATION_VECTORS.shape[1]
    != len(SEGMENT_LABELS)
):
    raise ValueError(
        "perturb_vectors has "
        f"{FIXED_PERTURBATION_VECTORS.shape[1]} columns, "
        "but segments has "
        f"{len(SEGMENT_LABELS)} unique labels."
    )


# Keep only valid node and bounding-box pairs.
valid_node_pairs = []


for node_name, bounding_box in zip(
    original_nodes,
    original_bboxes
):

    if bounding_box is None:
        continue

    if len(bounding_box) != 4:
        continue

    x0, y0, x1, y1 = map(
        float,
        bounding_box
    )

    coordinates = np.asarray(
        [
            x0,
            y0,
            x1,
            y1
        ],
        dtype=float
    )

    if not np.all(
        np.isfinite(
            coordinates
        )
    ):
        continue

    if (
        x1 <= x0
        or y1 <= y0
    ):
        continue

    valid_node_pairs.append(
        (
            str(node_name),
            (
                x0,
                y0,
                x1,
                y1
            )
        )
    )


if not valid_node_pairs:
    raise ValueError(
        "No valid original MedSAM node and "
        "bounding-box pairs were found."
    )


FIXED_NODE_NAMES = [
    item[0]
    for item in valid_node_pairs
]


FIXED_NODE_BBOXES = [
    item[1]
    for item in valid_node_pairs
]


print(
    "MedSAM robustness setup"
)

print(
    "Image shape:",
    BASE_IMAGE.shape
)

print(
    "Valid original nodes:",
    len(FIXED_NODE_NAMES)
)

print(
    "Number of superpixels:",
    len(SEGMENT_LABELS)
)

print(
    "Perturbations per condition:",
    len(FIXED_PERTURBATION_VECTORS)
)

print(
    "Repeated XGBoost splits:",
    N_XGB_REPEATS
)

print(
    "FAST_TEST:",
    FAST_TEST
)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

def set_all_random_seeds(
    seed
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


set_all_random_seeds(
    RANDOM_SEED
)


# ============================================================
# 5. RETINAL FIELD-OF-VIEW MASK
# ============================================================

def create_retinal_fov_mask(
    image_rgb,
    threshold=10
):
    """
    Estimate the visible retinal field while excluding
    most of the surrounding black background.
    """

    grey_image = (
        image_rgb
        .astype(np.float32)
        .mean(axis=2)
    )


    retinal_mask = (
        grey_image > threshold
    )


    # Fallback for images without a black background.
    if retinal_mask.sum() < (
        0.05
        *
        retinal_mask.size
    ):

        retinal_mask = np.ones(
            grey_image.shape,
            dtype=bool
        )


    return retinal_mask


# ============================================================
# 6. CONTRAST MODIFICATION
# ============================================================

def modify_retinal_contrast(
    image_rgb,
    contrast_factor
):
    """
    Apply a mean-preserving contrast transformation inside
    the retinal field.

    contrast_factor = 1.0:
        Original image.

    contrast_factor < 1.0:
        Reduced contrast.

    contrast_factor > 1.0:
        Increased contrast.
    """

    if contrast_factor <= 0:
        raise ValueError(
            "contrast_factor must be greater than zero."
        )


    image_float = image_rgb.astype(
        np.float32
    )


    retinal_mask = create_retinal_fov_mask(
        image_rgb
    )


    mean_rgb = image_float[
        retinal_mask
    ].mean(
        axis=0
    )


    modified_image = image_float.copy()


    modified_image[
        retinal_mask
    ] = (
        mean_rgb
        +
        contrast_factor
        *
        (
            image_float[
                retinal_mask
            ]
            -
            mean_rgb
        )
    )


    return np.clip(
        modified_image,
        0,
        255
    ).astype(
        np.uint8
    )


# ============================================================
# 7. SIMULATED BLINK-RELATED OCCLUSION
# ============================================================

def add_simulated_occlusion(
    image_rgb,
    occlusion_percentage,
    x_ratio=0.50,
    darkness=1.0
):
    """
    Add a controlled vertical acquisition occlusion.

    The artefact is described conservatively as a simulated
    blink-related occlusion, not as an exact reconstruction
    of eyelid geometry.

    occlusion_percentage:
        1 means 1% of retinal width.
        2 means 2%.
        3 means 3%.
    """

    if not 0 <= occlusion_percentage <= 100:
        raise ValueError(
            "occlusion_percentage must be between 0 and 100."
        )


    if not 0 <= x_ratio <= 1:
        raise ValueError(
            "x_ratio must be between 0 and 1."
        )


    if not 0 <= darkness <= 1:
        raise ValueError(
            "darkness must be between 0 and 1."
        )


    if occlusion_percentage == 0:
        return image_rgb.copy()


    retinal_mask = create_retinal_fov_mask(
        image_rgb
    )


    _, x_coordinates = np.where(
        retinal_mask
    )


    x_minimum = int(
        x_coordinates.min()
    )


    x_maximum = int(
        x_coordinates.max()
    )


    retinal_width = (
        x_maximum
        -
        x_minimum
        +
        1
    )


    line_width = max(
        2,
        int(
            round(
                retinal_width
                *
                occlusion_percentage
                /
                100.0
            )
        )
    )


    line_centre = int(
        round(
            x_minimum
            +
            x_ratio
            *
            (
                retinal_width
                -
                1
            )
        )
    )


    line_start = max(
        0,
        line_centre
        -
        line_width // 2
    )


    line_end = min(
        image_rgb.shape[1],
        line_start
        +
        line_width
    )


    modified_image = (
        image_rgb
        .astype(np.float32)
        .copy()
    )


    local_mask = retinal_mask[
        :,
        line_start:line_end
    ]


    local_region = modified_image[
        :,
        line_start:line_end
    ]


    darkened_region = (
        local_region
        *
        (
            1.0
            -
            darkness
        )
    )


    modified_image[
        :,
        line_start:line_end
    ] = np.where(
        local_mask[..., None],
        darkened_region,
        local_region
    )


    return np.clip(
        modified_image,
        0,
        255
    ).astype(
        np.uint8
    )


# ============================================================
# 8. APPLY FIXED SUPERPIXEL PERTURBATION
# ============================================================

def apply_fixed_superpixel_perturbation(
    condition_image,
    keep_vector
):
    """
    Apply the same original superpixel layout and
    perturbation vector to every acquisition condition.
    """

    perturbed_image = condition_image.copy()


    for column_index, segment_label in enumerate(
        SEGMENT_LABELS
    ):

        if int(
            keep_vector[
                column_index
            ]
        ) == 0:

            perturbed_image[
                FIXED_SEGMENTS
                ==
                segment_label
            ] = 0


    return perturbed_image.astype(
        np.uint8
    )


# ============================================================
# 9. POOL MEDSAM IMAGE EMBEDDING
# ============================================================

def pool_medsam_embedding(
    embedding
):
    """
    Convert the MedSAM image embedding into a single
    one-dimensional representation vector.
    """

    if torch.is_tensor(
        embedding
    ):

        embedding_array = (
            embedding
            .detach()
            .float()
            .cpu()
            .numpy()
        )

    else:

        embedding_array = np.asarray(
            embedding,
            dtype=np.float32
        )


    if embedding_array.ndim == 4:

        # Batch, channels, height, width
        pooled_embedding = embedding_array.mean(
            axis=(
                0,
                2,
                3
            )
        )

    elif embedding_array.ndim == 3:

        # Channels, height, width
        pooled_embedding = embedding_array.mean(
            axis=(
                1,
                2
            )
        )

    elif embedding_array.ndim == 2:

        pooled_embedding = embedding_array.mean(
            axis=0
        )

    elif embedding_array.ndim == 1:

        pooled_embedding = embedding_array

    else:

        raise ValueError(
            "Unexpected MedSAM embedding shape: "
            f"{embedding_array.shape}"
        )


    return np.asarray(
        pooled_embedding,
        dtype=np.float32
    )


# ============================================================
# 10. CORRECT MEDSAM BOX-PROMPT SCORING
# ============================================================

def score_fixed_boxes_and_embedding(
    image_rgb
):
    """
    Encode an image once and score all original node boxes.

    Important corrections:

    1. Original-image bounding boxes are transformed using:
       predictor.transform.apply_boxes_torch()

    2. predictor.predict_torch() is used to ensure that
       coordinate transformation and mask post-processing
       follow the official SAM pipeline.

    3. Node confidence is calculated from:
       sigmoid(mask logits)

       rather than raw mask logits or predicted IoU.
    """

    image_rgb = np.asarray(
        image_rgb,
        dtype=np.uint8
    )


    image_height, image_width = (
        image_rgb.shape[:2]
    )


    predictor.set_image(
        image_rgb
    )


    with torch.no_grad():

        image_embedding = (
            predictor.get_image_embedding()
        )


    pooled_embedding = (
        pool_medsam_embedding(
            image_embedding
        )
    )


    clipped_boxes = []
    valid_indices = []


    for node_index, (
        x0,
        y0,
        x1,
        y1
    ) in enumerate(
        FIXED_NODE_BBOXES
    ):

        x0 = float(
            np.clip(
                x0,
                0,
                image_width - 1
            )
        )

        x1 = float(
            np.clip(
                x1,
                0,
                image_width - 1
            )
        )

        y0 = float(
            np.clip(
                y0,
                0,
                image_height - 1
            )
        )

        y1 = float(
            np.clip(
                y1,
                0,
                image_height - 1
            )
        )


        if (
            x1 <= x0
            or y1 <= y0
        ):
            continue


        clipped_boxes.append(
            [
                x0,
                y0,
                x1,
                y1
            ]
        )


        valid_indices.append(
            node_index
        )


    confidence_values = np.full(
        len(FIXED_NODE_BBOXES),
        np.nan,
        dtype=np.float32
    )


    if not clipped_boxes:

        return (
            confidence_values,
            pooled_embedding
        )


    box_tensor = torch.as_tensor(
        np.asarray(
            clipped_boxes,
            dtype=np.float32
        ),
        dtype=torch.float32,
        device=DEVICE
    )


    # Critical correction:
    # transform original-image boxes to the coordinate space
    # expected by SAM.
    transformed_boxes = (
        predictor
        .transform
        .apply_boxes_torch(
            box_tensor,
            image_rgb.shape[:2]
        )
    )


    with torch.no_grad():

        (
            mask_logits,
            iou_predictions,
            _
        ) = predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=transformed_boxes,
            mask_input=None,
            multimask_output=True,
            return_logits=True
        )


        # mask_logits are post-processed to the original
        # image resolution by SamPredictor.
        probability_masks = torch.sigmoid(
            mask_logits
        )


        # Select the best candidate mask independently
        # for every prompted node box.
        best_mask_indices = torch.argmax(
            iou_predictions,
            dim=1
        )


        batch_indices = torch.arange(
            probability_masks.shape[0],
            device=probability_masks.device
        )


        best_probability_masks = probability_masks[
            batch_indices,
            best_mask_indices
        ]


    best_probability_masks = (
        best_probability_masks
        .detach()
        .float()
        .cpu()
        .numpy()
    )


    for local_index, node_index in enumerate(
        valid_indices
    ):

        soft_mask = best_probability_masks[
            local_index
        ]


        binary_mask = (
            soft_mask > 0.5
        )


        if binary_mask.sum() == 0:

            confidence = 0.0

        else:

            confidence = float(
                soft_mask[
                    binary_mask
                ].mean()
            )


        confidence_values[
            node_index
        ] = confidence


    return (
        confidence_values,
        pooled_embedding
    )


# ============================================================
# 11. COSINE DISTANCE AND LOCALITY WEIGHTS
# ============================================================

def calculate_cosine_distances(
    reference_embedding,
    perturbed_embeddings
):
    reference_embedding = np.asarray(
        reference_embedding,
        dtype=np.float32
    ).reshape(
        -1
    )


    perturbed_embeddings = np.asarray(
        perturbed_embeddings,
        dtype=np.float32
    )


    reference_norm = (
        np.linalg.norm(
            reference_embedding
        )
        +
        1e-12
    )


    perturbed_norms = (
        np.linalg.norm(
            perturbed_embeddings,
            axis=1
        )
        +
        1e-12
    )


    cosine_similarities = (
        perturbed_embeddings
        @
        reference_embedding
    ) / (
        perturbed_norms
        *
        reference_norm
    )


    cosine_similarities = np.clip(
        cosine_similarities,
        -1.0,
        1.0
    )


    cosine_distances = np.clip(
        1.0
        -
        cosine_similarities,
        0.0,
        2.0
    )


    return cosine_distances.astype(
        float
    )


def locality_weights_from_distances(
    cosine_distances,
    kernel_sigma
):
    cosine_distances = np.asarray(
        cosine_distances,
        dtype=float
    )


    locality_weights = np.exp(
        -(
            cosine_distances ** 2
        )
        /
        (
            float(
                kernel_sigma
            ) ** 2
        )
    )


    return np.clip(
        locality_weights,
        1e-6,
        None
    )


# ============================================================
# 12. MAP NODE NAMES TO RETINAL CONCEPTS
# ============================================================

def map_node_to_concept(
    node_name
):
    normalised_name = (
        str(node_name)
        .lower()
        .strip()
        .replace(
            "-",
            "_"
        )
        .replace(
            " ",
            "_"
        )
    )


    if "lesion" in normalised_name:
        return "lesion"


    if "vessel" in normalised_name:
        return "blood_vessels"


    if (
        "optic" in normalised_name
        and
        (
            "disc" in normalised_name
            or
            "disk" in normalised_name
        )
    ):
        return "optic_disc"


    return None


NODE_CONCEPTS = {
    node_name:
        map_node_to_concept(
            node_name
        )

    for node_name
    in FIXED_NODE_NAMES
}


unmatched_nodes = [
    node_name

    for node_name, concept
    in NODE_CONCEPTS.items()

    if concept is None
]


if unmatched_nodes:

    print(
        "Warning: unmatched nodes will be excluded "
        "from concept-level plots:"
    )

    print(
        unmatched_nodes
    )


# ============================================================
# 13. CACHE MEDSAM RESPONSES FOR EACH CONDITION
# ============================================================

condition_data_cache = {}


def extract_condition_data(
    condition_image,
    condition_name
):
    """
    Run MedSAM on the unmasked condition and every fixed
    superpixel perturbation.

    Results are cached using the image byte hash.
    """

    image_hash = hashlib.sha1(
        condition_image.tobytes()
    ).hexdigest()


    if image_hash in condition_data_cache:

        return condition_data_cache[
            image_hash
        ]


    print(
        "\n"
        +
        "=" * 70
    )

    print(
        "Extracting MedSAM responses for:",
        condition_name
    )

    print(
        "=" * 70
    )


    (
        condition_scores,
        condition_embedding
    ) = score_fixed_boxes_and_embedding(
        condition_image
    )


    perturbed_score_list = []
    perturbed_embedding_list = []


    for perturbation_index, keep_vector in enumerate(
        FIXED_PERTURBATION_VECTORS
    ):

        perturbed_image = (
            apply_fixed_superpixel_perturbation(
                condition_image,
                keep_vector
            )
        )


        (
            perturbed_scores,
            perturbed_embedding
        ) = score_fixed_boxes_and_embedding(
            perturbed_image
        )


        perturbed_score_list.append(
            perturbed_scores
        )


        perturbed_embedding_list.append(
            perturbed_embedding
        )


        if (
            perturbation_index == 0
            or
            (
                perturbation_index + 1
            ) % 5 == 0
        ):

            print(
                f"Processed "
                f"{perturbation_index + 1}/"
                f"{len(FIXED_PERTURBATION_VECTORS)} "
                f"perturbations"
            )


        if (
            torch.cuda.is_available()
            and
            (
                perturbation_index + 1
            ) % 10 == 0
        ):

            torch.cuda.empty_cache()


    perturbed_score_matrix = np.vstack(
        perturbed_score_list
    ).astype(
        np.float32
    )


    perturbed_embedding_matrix = np.vstack(
        perturbed_embedding_list
    ).astype(
        np.float32
    )


    confidence_shift_matrix = (
        condition_scores[
            None,
            :
        ]
        -
        perturbed_score_matrix
    )


    cosine_distances = calculate_cosine_distances(
        condition_embedding,
        perturbed_embedding_matrix
    )


    condition_data = {
        "condition_scores":
            condition_scores,

        "confidence_shift_matrix":
            confidence_shift_matrix,

        "cosine_distances":
            cosine_distances
    }


    condition_data_cache[
        image_hash
    ] = condition_data


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return condition_data


# ============================================================
# 14. CALIBRATE ONE FIXED KERNEL WIDTH
# ============================================================

print(
    "\nCalibrating locality kernel from the original image..."
)


BASELINE_DATA = extract_condition_data(
    BASE_IMAGE,
    "original image (kernel calibration)"
)


baseline_positive_distances = (
    BASELINE_DATA[
        "cosine_distances"
    ][
        BASELINE_DATA[
            "cosine_distances"
        ]
        >
        0
    ]
)


if len(
    baseline_positive_distances
) > 0:

    GLOBAL_KERNEL_SIGMA = float(
        np.median(
            baseline_positive_distances
        )
    )

else:

    GLOBAL_KERNEL_SIGMA = (
        FALLBACK_KERNEL_SIGMA
    )


if not np.isfinite(
    GLOBAL_KERNEL_SIGMA
):

    GLOBAL_KERNEL_SIGMA = (
        FALLBACK_KERNEL_SIGMA
    )


GLOBAL_KERNEL_SIGMA = max(
    GLOBAL_KERNEL_SIGMA,
    MIN_KERNEL_SIGMA
)


print(
    "Fixed kernel sigma:",
    GLOBAL_KERNEL_SIGMA
)


# ============================================================
# 15. EVALUATE ONE ACQUISITION CONDITION
# ============================================================

def evaluate_one_condition(
    condition_image,
    condition_name
):
    """
    Fit repeated XGBoost surrogates for one acquisition
    condition.

    Low-variance responses are retained in the diagnostic
    table but are not assigned an unstable Test R².
    """

    condition_data = extract_condition_data(
        condition_image,
        condition_name
    )


    condition_scores = condition_data[
        "condition_scores"
    ]


    confidence_shift_matrix = condition_data[
        "confidence_shift_matrix"
    ]


    cosine_distances = condition_data[
        "cosine_distances"
    ]


    locality_weights = (
        locality_weights_from_distances(
            cosine_distances,
            GLOBAL_KERNEL_SIGMA
        )
    )


    X_all = (
        FIXED_PERTURBATION_VECTORS
        .astype(float)
    )


    node_result_rows = []


    for node_index, node_name in enumerate(
        FIXED_NODE_NAMES
    ):

        y_all = (
            confidence_shift_matrix[
                :,
                node_index
            ]
            .astype(float)
        )


        valid_mask = (
            np.isfinite(
                y_all
            )
            &
            np.isfinite(
                locality_weights
            )
            &
            (
                locality_weights > 0
            )
        )


        X = X_all[
            valid_mask
        ]


        y = y_all[
            valid_mask
        ]


        weights = locality_weights[
            valid_mask
        ]


        if len(y) > 0:

            target_std = float(
                np.std(
                    y
                )
            )

            mean_absolute_shift = float(
                np.mean(
                    np.abs(
                        y
                    )
                )
            )

        else:

            target_std = np.nan
            mean_absolute_shift = np.nan


        common_results = {
            "condition":
                condition_name,

            "node_name":
                node_name,

            "concept":
                NODE_CONCEPTS[
                    node_name
                ],

            "original_confidence":
                float(
                    condition_scores[
                        node_index
                    ]
                ),

            "mean_absolute_shift":
                mean_absolute_shift,

            "target_std":
                target_std,

            "mean_cosine_distance":
                float(
                    np.mean(
                        cosine_distances
                    )
                ),

            "kernel_sigma":
                float(
                    GLOBAL_KERNEL_SIGMA
                ),

            "number_of_samples":
                int(
                    len(y)
                )
        }


        if len(y) < MIN_VALID_SAMPLES:

            node_result_rows.append({
                **common_results,

                "repeat_id":
                    np.nan,

                "test_r2":
                    np.nan,

                "test_mae":
                    np.nan,

                "test_target_std":
                    np.nan,

                "status":
                    "too_few_valid_samples"
            })

            continue


        if target_std < MIN_TARGET_STD:

            node_result_rows.append({
                **common_results,

                "repeat_id":
                    np.nan,

                "test_r2":
                    np.nan,

                "test_mae":
                    np.nan,

                "test_target_std":
                    np.nan,

                "status":
                    "low_variance_response"
            })

            continue


        for repeat_id in range(
            N_XGB_REPEATS
        ):

            split_seed = (
                RANDOM_SEED
                +
                repeat_id
            )


            (
                X_train,
                X_test,
                y_train,
                y_test,
                weight_train,
                weight_test
            ) = train_test_split(
                X,
                y,
                weights,
                test_size=TEST_SIZE,
                random_state=split_seed
            )


            test_target_std = float(
                np.std(
                    y_test
                )
            )


            # Avoid invalid or highly unstable R² when the
            # response in one test split is nearly constant.
            if test_target_std < MIN_TEST_STD:

                node_result_rows.append({
                    **common_results,

                    "repeat_id":
                        repeat_id,

                    "test_r2":
                        np.nan,

                    "test_mae":
                        np.nan,

                    "test_target_std":
                        test_target_std,

                    "status":
                        "low_variance_test_split"
                })

                continue


            xgb_model = XGBRegressor(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=split_seed,
                n_jobs=-1
            )


            xgb_model.fit(
                X_train,
                y_train,
                sample_weight=weight_train
            )


            predicted_shift = (
                xgb_model.predict(
                    X_test
                )
            )


            node_result_rows.append({
                **common_results,

                "repeat_id":
                    repeat_id,

                "test_r2":
                    float(
                        r2_score(
                            y_test,
                            predicted_shift
                        )
                    ),

                "test_mae":
                    float(
                        mean_absolute_error(
                            y_test,
                            predicted_shift
                        )
                    ),

                "test_target_std":
                    test_target_std,

                "status":
                    "ok"
            })


    return pd.DataFrame(
        node_result_rows
    )


# ============================================================
# 16. PREVIEW IMAGE CONDITIONS
# ============================================================

preview_conditions = [
    (
        "Original",
        BASE_IMAGE
    ),

    (
        "Contrast 0.60",
        modify_retinal_contrast(
            BASE_IMAGE,
            0.60
        )
    ),

    (
        "Contrast 1.40",
        modify_retinal_contrast(
            BASE_IMAGE,
            1.40
        )
    ),

    (
        "Occlusion 1%",
        add_simulated_occlusion(
            BASE_IMAGE,
            1,
            OCCLUSION_X_RATIO,
            OCCLUSION_DARKNESS
        )
    ),

    (
        "Occlusion 3%",
        add_simulated_occlusion(
            BASE_IMAGE,
            3,
            OCCLUSION_X_RATIO,
            OCCLUSION_DARKNESS
        )
    )
]


preview_figure, preview_axes = plt.subplots(
    1,
    5,
    figsize=(
        15.5,
        3.3
    )
)


for axis, (
    preview_title,
    preview_image
) in zip(
    preview_axes,
    preview_conditions
):

    axis.imshow(
        preview_image
    )

    axis.set_title(
        preview_title
    )

    axis.axis(
        "off"
    )


preview_figure.tight_layout()


preview_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_acquisition_preview.png"
)


preview_figure.savefig(
    preview_path,
    dpi=400,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()


# ============================================================
# 17. RUN CONTRAST EXPERIMENT
# ============================================================

all_condition_results = []


for contrast_factor in CONTRAST_LEVELS:

    contrast_image = modify_retinal_contrast(
        BASE_IMAGE,
        contrast_factor
    )


    condition_name = (
        f"contrast_"
        f"{contrast_factor:.2f}"
    )


    condition_result = evaluate_one_condition(
        contrast_image,
        condition_name
    )


    condition_result[
        "experiment"
    ] = "Contrast"


    condition_result[
        "severity"
    ] = float(
        contrast_factor
    )


    all_condition_results.append(
        condition_result
    )


    condition_result.to_csv(
        OUTPUT_DIRECTORY
        /
        f"{condition_name}_node_runs.csv",
        index=False
    )


# ============================================================
# 18. RUN OCCLUSION EXPERIMENT
# ============================================================

for occlusion_percentage in OCCLUSION_LEVELS:

    occlusion_image = add_simulated_occlusion(
        BASE_IMAGE,
        occlusion_percentage,
        OCCLUSION_X_RATIO,
        OCCLUSION_DARKNESS
    )


    condition_name = (
        f"occlusion_"
        f"{occlusion_percentage}"
        f"percent"
    )


    condition_result = evaluate_one_condition(
        occlusion_image,
        condition_name
    )


    condition_result[
        "experiment"
    ] = "VerticalLine"


    condition_result[
        "severity"
    ] = float(
        occlusion_percentage
    )


    all_condition_results.append(
        condition_result
    )


    condition_result.to_csv(
        OUTPUT_DIRECTORY
        /
        f"{condition_name}_node_runs.csv",
        index=False
    )


# ============================================================
# 19. COMBINE NODE-LEVEL RESULTS
# ============================================================

medsam_node_fidelity_runs_df = pd.concat(
    all_condition_results,
    ignore_index=True
)


node_level_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_node_level_runs.csv"
)


medsam_node_fidelity_runs_df.to_csv(
    node_level_csv_path,
    index=False
)


# Save diagnostic statuses.
status_counts_df = (
    medsam_node_fidelity_runs_df
    .groupby(
        [
            "experiment",
            "severity",
            "concept",
            "status"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="count"
    )
)


status_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_status_counts.csv"
)


status_counts_df.to_csv(
    status_csv_path,
    index=False
)


print(
    "\nNode-level status counts:"
)


display(
    status_counts_df
)


# ============================================================
# 20. CREATE CONCEPT-LEVEL REPEATED RESULTS
# ============================================================

valid_node_runs_df = (
    medsam_node_fidelity_runs_df[
        (
            medsam_node_fidelity_runs_df[
                "status"
            ]
            ==
            "ok"
        )
        &
        medsam_node_fidelity_runs_df[
            "concept"
        ].notna()
        &
        medsam_node_fidelity_runs_df[
            "test_r2"
        ].notna()
    ]
    .copy()
)


if valid_node_runs_df.empty:
    raise ValueError(
        "No valid MedSAM R² values were produced. "
        "Inspect medsam_corrected_status_counts.csv."
    )


# Average node-level R² values inside each concept and
# repeated split.
#
# This avoids treating every vessel or lesion node as an
# independent repetition.
medsam_robustness_fidelity_runs_df = (
    valid_node_runs_df
    .groupby(
        [
            "experiment",
            "severity",
            "condition",
            "concept",
            "repeat_id"
        ],
        as_index=False
    )
    .agg(
        test_r2=(
            "test_r2",
            "mean"
        ),

        test_mae=(
            "test_mae",
            "mean"
        ),

        mean_absolute_shift=(
            "mean_absolute_shift",
            "mean"
        ),

        mean_cosine_distance=(
            "mean_cosine_distance",
            "mean"
        ),

        mean_target_std=(
            "target_std",
            "mean"
        ),

        number_of_nodes=(
            "node_name",
            "nunique"
        )
    )
)


concept_level_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_concept_level_runs.csv"
)


medsam_robustness_fidelity_runs_df.to_csv(
    concept_level_csv_path,
    index=False
)


# ============================================================
# 21. CREATE SUMMARY TABLE
# ============================================================

medsam_robustness_summary_df = (
    medsam_robustness_fidelity_runs_df
    .groupby(
        [
            "experiment",
            "severity",
            "concept"
        ],
        as_index=False
    )
    .agg(
        mean_test_r2=(
            "test_r2",
            "mean"
        ),

        median_test_r2=(
            "test_r2",
            "median"
        ),

        std_test_r2=(
            "test_r2",
            "std"
        ),

        mean_test_mae=(
            "test_mae",
            "mean"
        ),

        mean_absolute_shift=(
            "mean_absolute_shift",
            "mean"
        ),

        mean_cosine_distance=(
            "mean_cosine_distance",
            "mean"
        ),

        mean_target_std=(
            "mean_target_std",
            "mean"
        ),

        number_of_runs=(
            "test_r2",
            "count"
        ),

        mean_number_of_nodes=(
            "number_of_nodes",
            "mean"
        )
    )
)


medsam_robustness_summary_df[
    "std_test_r2"
] = (
    medsam_robustness_summary_df[
        "std_test_r2"
    ]
    .fillna(
        0.0
    )
)


medsam_robustness_summary_df[
    "lower_sd"
] = (
    medsam_robustness_summary_df[
        "mean_test_r2"
    ]
    -
    medsam_robustness_summary_df[
        "std_test_r2"
    ]
)


medsam_robustness_summary_df[
    "upper_sd"
] = (
    medsam_robustness_summary_df[
        "mean_test_r2"
    ]
    +
    medsam_robustness_summary_df[
        "std_test_r2"
    ]
)


summary_csv_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_summary.csv"
)


medsam_robustness_summary_df.to_csv(
    summary_csv_path,
    index=False
)


print(
    "\nCorrected MedSAM robustness summary:"
)


display(
    medsam_robustness_summary_df.round(
        4
    )
)


# ============================================================
# 22. PUBLICATION-QUALITY FIGURE STYLE
# ============================================================

plt.rcParams.update({
    "figure.dpi":
        180,

    "savefig.dpi":
        800,

    "font.family":
        "DejaVu Serif",

    "mathtext.fontset":
        "stix",

    "font.size":
        11,

    "axes.titlesize":
        14,

    "axes.labelsize":
        12,

    "xtick.labelsize":
        10,

    "ytick.labelsize":
        10,

    "legend.fontsize":
        10,

    "axes.linewidth":
        1.0,

    "lines.linewidth":
        2.4,

    "lines.markersize":
        6.5,

    "pdf.fonttype":
        42,

    "ps.fonttype":
        42
})


concept_order = [
    "blood_vessels",
    "lesion",
    "optic_disc"
]


concept_labels = {
    "blood_vessels":
        "Blood vessels",

    "lesion":
        "Lesion",

    "optic_disc":
        "Optic disc"
}


concept_colours = {
    "blood_vessels":
        "#2F73B7",

    "lesion":
        "#F28E2B",

    "optic_disc":
        "#3A9D3A"
}


experiment_order = [
    "Contrast",
    "VerticalLine"
]


experiment_titles = {
    "Contrast":
        "Contrast Variation",

    "VerticalLine":
        "Simulated Blink-Related "
        "Occlusion Artefact"
}


experiment_xlabels = {
    "Contrast":
        "Contrast factor "
        "(1.0 = original)",

    "VerticalLine":
        "Simulated occlusion "
        "(% of retinal width)"
}


plot_random_generator = np.random.default_rng(
    RANDOM_SEED
)


# ============================================================
# 23. DRAW FINAL CORRECTED 3 x 2 FIGURE
# ============================================================

figure, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(
        14.2,
        9.8
    ),
    sharey="row",
    constrained_layout=False
)


figure.subplots_adjust(
    left=0.12,
    right=0.985,
    top=0.90,
    bottom=0.13,
    hspace=0.28,
    wspace=0.12
)


for row_index, concept in enumerate(
    concept_order
):

    concept_values = (
        medsam_robustness_fidelity_runs_df[
            medsam_robustness_fidelity_runs_df[
                "concept"
            ]
            ==
            concept
        ][
            "test_r2"
        ]
        .dropna()
        .to_numpy()
    )


    if len(
        concept_values
    ) > 0:

        y_minimum = float(
            np.min(
                concept_values
            )
        )


        y_maximum = float(
            np.max(
                concept_values
            )
        )


        y_range = (
            y_maximum
            -
            y_minimum
        )


        if y_range < 0.20:

            y_padding = 0.10

        else:

            y_padding = max(
                0.10,
                0.10
                *
                y_range
            )


        row_y_limits = (
            y_minimum
            -
            y_padding,

            y_maximum
            +
            y_padding
        )

    else:

        row_y_limits = (
            -0.1,
            1.0
        )


    for column_index, experiment in enumerate(
        experiment_order
    ):

        axis = axes[
            row_index,
            column_index
        ]


        panel_runs = (
            medsam_robustness_fidelity_runs_df[
                (
                    medsam_robustness_fidelity_runs_df[
                        "concept"
                    ]
                    ==
                    concept
                )
                &
                (
                    medsam_robustness_fidelity_runs_df[
                        "experiment"
                    ]
                    ==
                    experiment
                )
            ]
            .copy()
        )


        panel_summary = (
            medsam_robustness_summary_df[
                (
                    medsam_robustness_summary_df[
                        "concept"
                    ]
                    ==
                    concept
                )
                &
                (
                    medsam_robustness_summary_df[
                        "experiment"
                    ]
                    ==
                    experiment
                )
            ]
            .sort_values(
                "severity"
            )
            .copy()
        )


        if (
            panel_runs.empty
            or panel_summary.empty
        ):

            axis.text(
                0.5,
                0.5,

                "Fidelity not estimable\n"
                "(low response variance)",

                ha="center",
                va="center",

                transform=
                    axis.transAxes,

                fontsize=10
            )


            axis.set_ylim(
                row_y_limits
            )


            axis.grid(
                alpha=0.18
            )


            if row_index == 0:

                axis.set_title(
                    experiment_titles[
                        experiment
                    ],

                    fontweight=
                        "semibold",

                    pad=12
                )


            continue


        severity_levels = sorted(
            panel_runs[
                "severity"
            ].unique()
        )


        # Individual repeated XGBoost results.
        for severity in severity_levels:

            severity_values = (
                panel_runs[
                    panel_runs[
                        "severity"
                    ]
                    ==
                    severity
                ][
                    "test_r2"
                ]
                .dropna()
                .to_numpy()
            )


            jitter_scale = (
                0.007
                if experiment
                ==
                "Contrast"
                else
                0.025
            )


            jitter = (
                plot_random_generator
                .normal(
                    loc=0.0,
                    scale=
                        jitter_scale,
                    size=
                        len(
                            severity_values
                        )
                )
            )


            axis.scatter(
                severity
                +
                jitter,

                severity_values,

                s=28,
                alpha=0.38,

                color=
                    concept_colours[
                        concept
                    ],

                edgecolors=
                    "white",

                linewidths=
                    0.45,

                zorder=4
            )


        # Mean ± standard deviation.
        axis.fill_between(
            panel_summary[
                "severity"
            ].to_numpy(),

            panel_summary[
                "lower_sd"
            ].to_numpy(),

            panel_summary[
                "upper_sd"
            ].to_numpy(),

            color=
                concept_colours[
                    concept
                ],

            alpha=0.16,
            linewidth=0,
            zorder=1
        )


        # Mean trend.
        axis.plot(
            panel_summary[
                "severity"
            ],

            panel_summary[
                "mean_test_r2"
            ],

            marker="o",
            markersize=6.5,

            markeredgewidth=
                0.8,

            markeredgecolor=
                "white",

            linewidth=2.5,

            color=
                concept_colours[
                    concept
                ],

            zorder=6
        )


        # Median.
        axis.scatter(
            panel_summary[
                "severity"
            ],

            panel_summary[
                "median_test_r2"
            ],

            marker="D",
            s=45,

            facecolors=
                "white",

            edgecolors=
                concept_colours[
                    concept
                ],

            linewidths=
                1.5,

            zorder=7
        )


        # R² = 0 reference line.
        axis.axhline(
            0.0,

            color="#777777",

            linestyle="--",
            linewidth=1.0,
            alpha=0.70
        )


        # Original image condition.
        original_condition = (
            1.0
            if experiment
            ==
            "Contrast"
            else
            0.0
        )


        axis.axvline(
            original_condition,

            color="#888888",

            linestyle=":",
            linewidth=1.2,
            alpha=0.80
        )


        axis.set_ylim(
            row_y_limits
        )


        axis.set_xticks(
            severity_levels
        )


        axis.set_xticklabels(
            [
                f"{severity:g}"

                for severity
                in severity_levels
            ]
        )


        axis.grid(
            alpha=0.18
        )


        axis.set_axisbelow(
            True
        )


        if row_index == 0:

            axis.set_title(
                experiment_titles[
                    experiment
                ],

                fontweight=
                    "semibold",

                pad=12
            )


        if column_index == 0:

            axis.text(
                -0.11,
                0.50,

                concept_labels[
                    concept
                ],

                transform=
                    axis.transAxes,

                rotation=90,

                va="center",
                ha="center",

                fontsize=12,

                fontweight=
                    "semibold"
            )

        else:

            axis.tick_params(
                labelleft=False
            )


        if row_index == 2:

            axis.set_xlabel(
                experiment_xlabels[
                    experiment
                ],

                labelpad=9
            )


# ============================================================
# 24. SHARED LABEL, LEGEND AND TITLE
# ============================================================

figure.supylabel(
    "ConceptSMILE surrogate fidelity "
    "(Test $R^2$)",
    fontsize=13,
    x=0.035
)


legend_handles = [
    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",
        markersize=7,

        markerfacecolor=
            "#666666",

        markeredgecolor=
            "white",

        alpha=0.65,

        label=
            "Individual XGBoost runs"
    ),

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="-",

        linewidth=2.5,
        markersize=6,

        color="#444444",

        label="Mean"
    ),

    Line2D(
        [0],
        [0],

        marker="D",
        linestyle="none",

        markersize=6,

        markerfacecolor=
            "white",

        markeredgecolor=
            "#444444",

        markeredgewidth=
            1.4,

        label="Median"
    ),

    Patch(
        facecolor="#888888",
        alpha=0.18,
        edgecolor="none",
        label="Mean ± SD"
    )
]


figure.legend(
    handles=legend_handles,

    loc="lower center",

    bbox_to_anchor=(
        0.54,
        0.025
    ),

    ncol=4,
    frameon=False,

    handlelength=2.0,
    columnspacing=1.8
)


figure.suptitle(
    "Corrected MedSAM Concept-Level Robustness "
    "to Retinal Image Acquisition Variations",

    fontsize=16,
    fontweight="semibold",

    y=0.973
)


# ============================================================
# 25. SAVE FINAL FIGURE
# ============================================================

png_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_acquisition_robustness.png"
)


pdf_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_acquisition_robustness.pdf"
)


svg_path = (
    OUTPUT_DIRECTORY
    /
    "medsam_corrected_acquisition_robustness.svg"
)


figure.savefig(
    png_path,

    dpi=800,

    bbox_inches="tight",

    facecolor="white",
    edgecolor="none",

    pil_kwargs={
        "compress_level":
            1
    }
)


figure.savefig(
    pdf_path,

    bbox_inches="tight",

    facecolor="white",
    edgecolor="none"
)


figure.savefig(
    svg_path,

    bbox_inches="tight",

    facecolor="white",
    edgecolor="none"
)


plt.show()


# ============================================================
# 26. FINAL OUTPUT PATHS
# ============================================================

print(
    "\nSaved corrected MedSAM robustness files:"
)


for saved_path in [
    preview_path,
    node_level_csv_path,
    status_csv_path,
    concept_level_csv_path,
    summary_csv_path,
    png_path,
    pdf_path,
    svg_path
]:

    print(
        saved_path
    )

MedSAM robustness setup
Image shape: (1024, 1024, 3)
Valid original nodes: 12
Number of superpixels: 6
Perturbations per condition: 50
Repeated XGBoost splits: 30
FAST_TEST: False

Calibrating locality kernel from the original image...

Extracting MedSAM responses for: original image (kernel calibration)
Processed 1/50 perturbations
Processed 5/50 perturbations
Processed 10/50 perturbations
Processed 15/50 perturbations
Processed 20/50 perturbations
Processed 25/50 perturbations
Processed 30/50 perturbations
Processed 35/50 perturbations
Processed 40/50 perturbations
Processed 45/50 perturbations
Processed 50/50 perturbations
Fixed kernel sigma: 0.08680859208106995


<Figure size 2790x594 with 5 Axes>


Extracting MedSAM responses for: contrast_0.60
Processed 1/50 perturbations
Processed 5/50 perturbations
Processed 10/50 perturbations
Processed 15/50 perturbations
Processed 20/50 perturbations
Processed 25/50 perturbations
Processed 30/50 perturbations
Processed 35/50 perturbations
Processed 40/50 perturbations
Processed 45/50 perturbations
Processed 50/50 perturbations

Extracting MedSAM responses for: contrast_0.80
Processed 1/50 perturbations
Processed 5/50 perturbations
Processed 10/50 perturbations
Processed 15/50 perturbations
Processed 20/50 perturbations
Processed 25/50 perturbations
Processed 30/50 perturbations
Processed 35/50 perturbations
Processed 40/50 perturbations
Processed 45/50 perturbations
Processed 50/50 perturbations

Extracting MedSAM responses for: contrast_1.20
Processed 1/50 perturbations
Processed 5/50 perturbations
Processed 10/50 perturbations
Processed 15/50 perturbations
Processed 20/50 perturbations
Processed 25/50 perturbations
Processed 30/50 pertur

      experiment  severity        concept status  count
0       Contrast       0.6  blood_vessels     ok    270
1       Contrast       0.6         lesion     ok     90
2       Contrast       0.8  blood_vessels     ok    270
3       Contrast       0.8         lesion     ok     90
4       Contrast       1.0  blood_vessels     ok    270
5       Contrast       1.0         lesion     ok     90
6       Contrast       1.2  blood_vessels     ok    270
7       Contrast       1.2         lesion     ok     90
8       Contrast       1.4  blood_vessels     ok    270
9       Contrast       1.4         lesion     ok     90
10  VerticalLine       0.0  blood_vessels     ok    270
11  VerticalLine       0.0         lesion     ok     90
12  VerticalLine       1.0  blood_vessels     ok    270
13  VerticalLine       1.0         lesion     ok     90
14  VerticalLine       2.0  blood_vessels     ok    270
15  VerticalLine       2.0         lesion     ok     90
16  VerticalLine       3.0  blood_vessels     ok


Corrected MedSAM robustness summary:


      experiment  severity        concept  mean_test_r2  median_test_r2  \
0       Contrast       0.6  blood_vessels        0.3345          0.4582   
1       Contrast       0.6         lesion        0.9607          0.9664   
2       Contrast       0.8  blood_vessels        0.5299          0.6372   
3       Contrast       0.8         lesion        0.9043          0.9306   
4       Contrast       1.0  blood_vessels        0.6331          0.6945   
5       Contrast       1.0         lesion        0.9436          0.9501   
6       Contrast       1.2  blood_vessels        0.7221          0.7564   
7       Contrast       1.2         lesion        0.9669          0.9700   
8       Contrast       1.4  blood_vessels        0.7839          0.8063   
9       Contrast       1.4         lesion        0.9828          0.9845   
10  VerticalLine       0.0  blood_vessels        0.6331          0.6945   
11  VerticalLine       0.0         lesion        0.9436          0.9501   
12  VerticalLine       1.

<Figure size 2556x1764 with 6 Axes>


Saved corrected MedSAM robustness files:
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_acquisition_preview.png
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_node_level_runs.csv
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_status_counts.csv
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_concept_level_runs.csv
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_summary.csv
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_acquisition_robustness.png
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_acquisition_robustness.pdf
/kaggle/working/conceptsmile_medsam_robustness_corrected/medsam_corrected_acquisition_robustness.svg


In [49]:
# ============================================================
# NEW CELL: Surrogate and Locality Comparison Table
# Fills:
# ConceptLIME      = C  + WLR
# ConceptBayLIME   = C  + BRR
# ConceptSMILE-WLR = WD + WLR
# ConceptLIME-XGB  = C  + XGB
# ConceptSMILE     = WD + XGB
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------
if "perturb_vectors" not in globals():
    raise ValueError("perturb_vectors is missing. Run the perturbation cell first.")

if "node_shift_matrix" not in globals():
    raise ValueError("node_shift_matrix is missing. Run the node shift matrix cell first.")

X_all = np.asarray(perturb_vectors, dtype=float)

if X_all.ndim != 2:
    raise ValueError("perturb_vectors must be a 2D array.")

if node_shift_matrix.empty:
    raise ValueError("node_shift_matrix is empty.")

# ------------------------------------------------------------
# 2. Metric utilities
# ------------------------------------------------------------
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    w_sum = np.sum(w)

    if len(x) == 0:
        return np.nan

    if w_sum <= 0:
        return np.mean(x)

    return np.sum(w * x) / w_sum


def weighted_mse(y_true, y_pred, w):
    return weighted_mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2, w)


def weighted_mae(y_true, y_pred, w):
    return weighted_mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), w)


def weighted_r2(y_true, y_pred, w):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.asarray(w, dtype=float)

    if len(y_true) < 2:
        return np.nan

    y_bar = weighted_mean(y_true, w)
    ss_res = np.sum(w * (y_true - y_pred) ** 2)
    ss_tot = np.sum(w * (y_true - y_bar) ** 2)

    if ss_tot <= 1e-12:
        return np.nan

    return 1.0 - (ss_res / ss_tot)

# ------------------------------------------------------------
# 3. Weight loaders
# ------------------------------------------------------------
def get_cosine_weight_map(n_rows):
    """
    Returns cosine locality weights aligned by perturb_id.
    """
    if "distance_df" in globals():
        df = distance_df.copy()
    else:
        path = "/kaggle/working/distance_df_from_medsam_cosine.csv"
        if not os.path.exists(path):
            print("Cosine distance file not found. Using equal weights.")
            return pd.Series(np.ones(n_rows), index=np.arange(n_rows)), "equal"
        df = pd.read_csv(path)

    if "kernel_weight_from_cosdist" in df.columns:
        wcol = "kernel_weight_from_cosdist"
    elif "kernel_weight" in df.columns:
        wcol = "kernel_weight"
    else:
        print("No cosine kernel weight column found. Using equal weights.")
        return pd.Series(np.ones(n_rows), index=np.arange(n_rows)), "equal"

    weight_map = (
        df[["perturb_id", wcol]]
        .drop_duplicates("perturb_id")
        .set_index("perturb_id")[wcol]
        .astype(float)
    )

    return weight_map, wcol


def get_wasserstein_weight_map(n_rows):
    """
    Returns Wasserstein locality weights aligned by perturb_id.
    """
    if "distance_df_from_medsam_wasserstein" in globals():
        df = distance_df_from_medsam_wasserstein.copy()
    else:
        path = "/kaggle/working/distance_df_from_medsam_wasserstein.csv"
        if not os.path.exists(path):
            print("Wasserstein distance file not found. Using equal weights.")
            return pd.Series(np.ones(n_rows), index=np.arange(n_rows)), "equal"
        df = pd.read_csv(path)

    if "kernel_weight_from_wassdist" in df.columns:
        wcol = "kernel_weight_from_wassdist"
    elif "kernel_weight" in df.columns:
        wcol = "kernel_weight"
    else:
        print("No Wasserstein kernel weight column found. Using equal weights.")
        return pd.Series(np.ones(n_rows), index=np.arange(n_rows)), "equal"

    weight_map = (
        df[["perturb_id", wcol]]
        .drop_duplicates("perturb_id")
        .set_index("perturb_id")[wcol]
        .astype(float)
    )

    return weight_map, wcol


# ------------------------------------------------------------
# 4. Surrogate model factory
# ------------------------------------------------------------
def build_surrogate_model(surrogate_name, random_state):
    if surrogate_name == "WLR":
        return LinearRegression()

    if surrogate_name == "BRR":
        return BayesianRidge()

    if surrogate_name == "XGB":
        return XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state
        )

    raise ValueError(f"Unknown surrogate: {surrogate_name}")


# ------------------------------------------------------------
# 5. Variant definitions
# ------------------------------------------------------------
variants = [
    {
        "Variant": "ConceptLIME",
        "Dist.": "C",
        "Surr.": "WLR",
        "Purpose": "Concept-level LIME baseline"
    },
    {
        "Variant": "ConceptBayLIME",
        "Dist.": "C",
        "Surr.": "BRR",
        "Purpose": "Bayesian LIME-style baseline"
    },
    {
        "Variant": "ConceptSMILE-WLR",
        "Dist.": "WD",
        "Surr.": "WLR",
        "Purpose": "Tests Wasserstein locality with a linear surrogate"
    },
    {
        "Variant": "ConceptLIME-XGB",
        "Dist.": "C",
        "Surr.": "XGB",
        "Purpose": "Tests XGBoost under cosine locality"
    },
    {
        "Variant": "ConceptSMILE",
        "Dist.": "WD",
        "Surr.": "XGB",
        "Purpose": "Final selected ConceptSMILE configuration"
    }
]

# ------------------------------------------------------------
# 6. Run comparison
# ------------------------------------------------------------
N_REPEATS = 30
TEST_SIZE = 0.30

cos_weight_map, cos_weight_source = get_cosine_weight_map(len(X_all))
wd_weight_map, wd_weight_source = get_wasserstein_weight_map(len(X_all))

print("Cosine weight source:", cos_weight_source)
print("Wasserstein weight source:", wd_weight_source)

all_variant_runs = []

for variant in variants:

    variant_name = variant["Variant"]
    dist_name = variant["Dist."]
    surrogate_name = variant["Surr."]

    if dist_name == "C":
        weight_map = cos_weight_map
    elif dist_name == "WD":
        weight_map = wd_weight_map
    else:
        raise ValueError(f"Unknown distance: {dist_name}")

    for node in node_shift_matrix.columns:

        y_all = node_shift_matrix[node].values.astype(float)

        valid_mask = ~np.isnan(y_all)
        X = X_all[valid_mask]
        y = y_all[valid_mask]

        valid_ids = np.where(valid_mask)[0]
        w = weight_map.reindex(valid_ids).fillna(1.0).values.astype(float)

        if len(y) < 8 or np.std(y) < 1e-8:
            continue

        for rep in range(N_REPEATS):

            rs = 42 + rep

            X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
                X, y, w,
                test_size=TEST_SIZE,
                random_state=rs
            )

            model = build_surrogate_model(surrogate_name, random_state=rs)

            # All three models support sample_weight in modern sklearn/xgboost.
            model.fit(X_train, y_train, sample_weight=w_train)

            y_pred = model.predict(X_test)

            mse_val = mean_squared_error(y_test, y_pred)
            mae_val = mean_absolute_error(y_test, y_pred)
            wmse_val = weighted_mse(y_test, y_pred, w_test)
            wmae_val = weighted_mae(y_test, y_pred, w_test)

            try:
                r2_val = r2_score(y_test, y_pred)
            except Exception:
                r2_val = np.nan

            r2w_val = weighted_r2(y_test, y_pred, w_test)

            all_variant_runs.append({
                "Variant": variant_name,
                "Dist.": dist_name,
                "Surr.": surrogate_name,
                "Purpose": variant["Purpose"],
                "node_name": node,
                "repeat_id": rep,
                "MSE": mse_val,
                "MAE": mae_val,
                "WMSE": wmse_val,
                "WMAE": wmae_val,
                "R2": r2_val,
                "R2_w": r2w_val
            })

surrogate_comparison_runs_df = pd.DataFrame(all_variant_runs)

if surrogate_comparison_runs_df.empty:
    raise ValueError("No valid surrogate comparison results were produced.")

# ------------------------------------------------------------
# 7. Aggregate into the exact table format
# ------------------------------------------------------------
surrogate_model_selection_table_df = (
    surrogate_comparison_runs_df
    .groupby(["Variant", "Dist.", "Surr.", "Purpose"], as_index=False)
    .agg({
        "WMSE": "mean",
        "WMAE": "mean",
        "R2": "mean",
        "R2_w": "mean"
    })
)

order = {
    "ConceptLIME": 0,
    "ConceptBayLIME": 1,
    "ConceptSMILE-WLR": 2,
    "ConceptLIME-XGB": 3,
    "ConceptSMILE": 4
}

surrogate_model_selection_table_df["__order"] = surrogate_model_selection_table_df["Variant"].map(order)
surrogate_model_selection_table_df = (
    surrogate_model_selection_table_df
    .sort_values("__order")
    .drop(columns="__order")
)

print("Surrogate model and locality selection table:")
display(surrogate_model_selection_table_df.round(4))

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------
surrogate_comparison_runs_df.to_csv(
    "/kaggle/working/surrogate_comparison_runs_df.csv",
    index=False
)

surrogate_model_selection_table_df.to_csv(
    "/kaggle/working/surrogate_model_selection_table.csv",
    index=False
)

print("Saved:")
print("/kaggle/working/surrogate_comparison_runs_df.csv")
print("/kaggle/working/surrogate_model_selection_table.csv")

Cosine weight source: kernel_weight_from_cosdist
Wasserstein weight source: kernel_weight_from_wassdist
Surrogate model and locality selection table:


            Variant Dist. Surr.  \
1       ConceptLIME     C   WLR   
0    ConceptBayLIME     C   BRR   
4  ConceptSMILE-WLR    WD   WLR   
2   ConceptLIME-XGB     C   XGB   
3      ConceptSMILE    WD   XGB   

                                             Purpose    WMSE    WMAE      R2  \
1                        Concept-level LIME baseline  0.0008  0.0183  0.2529   
0                       Bayesian LIME-style baseline  0.0008  0.0181  0.3436   
4  Tests Wasserstein locality with a linear surro...  0.0009  0.0195  0.4310   
2                Tests XGBoost under cosine locality  0.0005  0.0124  0.7110   
3          Final selected ConceptSMILE configuration  0.0004  0.0105  0.8326   

     R2_w  
1  0.4067  
0  0.4783  
4  0.4400  
2  0.7349  
3  0.8225  

Saved:
/kaggle/working/surrogate_comparison_runs_df.csv
/kaggle/working/surrogate_model_selection_table.csv
